## VMR Scorecard Version - (last update: 2024-07-09)

### Developer: Juan Morice

##### This project is also based on Katherine Mertens, Arthur Li and Kim Kuhn previous work.

##### *UPDATES: This code version is allowed to:*

* Run "Reading_SharePoint_Lists.ipynb" script to import the parameters loaded by Sales Team through both Power Apps and Power Automate developed tool.
* Define the segments you want to use for reporting the metrics: 

        1. UPC hierarchy groups
        2.YB brand description (trademark_brand_desc)
        3. YB category description (cat_desc)
        4. Custom Brand's descriptions from mapping file provided. A mapping file with the Custom Brand description (first column) and cat_desc (second column) fields is needed.
        5. Retailer's descriptions from trade_item_owner_hierarchy_v table (except Kroger). A UPC list based on one specific retailer is needed.
        6. Kroger's own descriptions from nz_hierarchy table. A UPC list based only in Kroger is needed.
        
        
* Define automatically the static parameters based on the number of retailers involved.
* Handle cases when you have NEW UPCs, so YAGO metrics cannot be shown.
* Deliver both top 10 or all segment combinations, regarding what the analyst or Director wants to report.
* Create two final ouputs for delivering: formatted Excel data and populated Power Point slides.

In [1]:
# import modules
import jaydebeapi as jd  
#import pyodbc as po
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import datetime as dt
import psycopg2 as ybconn
import re

In [2]:
import getpass
from safe_password import readpw
import psycopg2 as ybconn
import pandas as pd
from datetime import date
from YB_load import yb_load, yb_load_file


# database on server for connection string below
dbase = 'py1usta1'

# connection string to server
conn = ybconn.connect(user=getpass.getuser(), password=readpw("Yellowbrick"), 
                      host='orlpybvip01.catmktg.com', port='5432', 
                      database = f'{dbase}')
conn.set_session(autocommit=True)
curs = conn.cursor()

print('Connection 2 -- done')

Connection 2 -- done


## **1) PARAMATERS:**

In [3]:

dbase = 'py1usta1'                  # database on server for connection string below

#PARAMATERS
 
analyst = 'mjuan'    #To avoid collisions with the ownership of some tables in YB

export_path = "/analytical_services/Public/moricejuan/"

report_name_for_export =  "2026_WG_March_Beauty_Event_S25G10"     # Desired name of the final Excel file - will concatenate with "VMR Custom Analysis" and Sunday week ending date down below

brand_nm = report_name_for_export      #If you are gonna use more than one word, separate them with "_" e.g. 'Catalina_Marketing'

lmc_list_id = "4165bfa2-b4fe-4f88-ba28-9f2c9f186f1e"

campaign_nm = report_name_for_export

brand_nbr = [1]             # LIST of promoted brand number(s) from upc list: comma separated between brackets e.g. [1] or [1,2,3]
cat_nbr = [1,2
          ]             # LIST of category brand number(s) from upc list: comma separated between brackets e.g. [1] or [1,2,3]

BL_CODES = [ 
    
    'BL_4303_0667'

]
  
    # coupons used in analysis: separated by white space, tabs, new lines, etc or commas inside of the triple quotes

Announcement = [ 
''
]

min_threshold_statement = 'dollars >= 25'       # examples:   'dollars >= 15'  or 'units >= 5'

threshold_unit = 'dollars'

pre_weeks  = 52           # Default: 52, number of pre-period weeks -- Kroger dom, we are allowed to use 52
redemption_days = 14      # Default: 14 (2 weeks), number of redepmtion days for the reward prints, from the Cost Estimate

segment_type = 1       #You can set the segment definition based on:
                             #"1. UPC hierarchy groups",
                             #"2. trademark_brand_desc",
                             #"3. cat_desc", 
                             #"4. CUSTOM BRAND'S DESCRIPTION FROM FILE PROVIDED"
                             #"5. RETAILER's DESCRIPTIONS from trade_item_owner_hierarchy_v table (except Kroger)",
                             #"6. KROGER's OWN DESCRIPTIONS from nz_hierarchy table"
        

#If you chose segment_type = 4, the code will take the custom Excel file (.xlsx) with both Brand's descriptions and cat_desc field from the next path:
custom_brand_desc_path = "/analytical_services/Public/moricejuan/Unilever_mapping_file.xlsx"        
        
# If you chose segment_type = 5, default is 2. You can set which description field from trade_item_owner_hierarchy_v to use.
label_level = 2

#DOUBLE-CHECK that sharepoint

In [4]:
print(report_name_for_export)

2026_WG_March_Beauty_Event_S25G10


In [5]:
############################## DON'T CHANGE ################################## 

import lmc_list_upc_2 as lmc
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText


#Assign date+time variable so tables do not clash in Yellowbrick
from datetime import datetime,timedelta
rnd=(datetime.now().strftime('%y%m%d%H%M%S')) #pulls todays date + time as a string. examp: 2254334774

LMC_BaseURL='https://listmanager.catalinamarketing.com'
lmc_list_focus_brands = lmc.LMC_PandasGetAllUPCs(LMC_BaseURL, lmc_list_id)
lmc_list_focus_brands['cmc_cat_nbr'] = round(lmc_list_focus_brands['cmc_cat_nbr'])
# lmc_list_focus_brands[cmc_cat_nbr]=lmc_list_focus_brands[cmc_cat_nbr].astype(float)
record_count= lmc_list_focus_brands['upc_cd'].count()
distinct_count = lmc_list_focus_brands['upc_cd'].nunique()
print(lmc_list_focus_brands)
# lmc_list_focus_brands.columns #brings back all the column names only, easier to view

# Check for duplicate upcs within the entire file
record_count= lmc_list_focus_brands['upc_cd'].count()
distinct_count = lmc_list_focus_brands['upc_cd'].nunique()
check_dups = pd.DataFrame([{'Total Records':record_count,'Distinct Count':distinct_count}])
print(check_dups.style)

# List of upc groups with upc group descriptions and upc counts
upc_group_list = lmc_list_focus_brands[['group_number','group_name','upc_cd']].groupby(['group_number','group_name']).count()
print(upc_group_list)

# Check if any eq measure are null (which will interfere with List Import and would need to be updated first)
check_eq_meas = lmc_list_focus_brands[['tot_wgt_meas','upc_cd']].groupby(['tot_wgt_meas']).count()
print(check_eq_meas)

# Check if any eq wgt amt are null (which will interfere with List Import and would need to be updated first)
check_eq_wgt_amt = lmc_list_focus_brands[['tot_wgt_amt','upc_cd']].groupby(['tot_wgt_amt']).count()
print(check_eq_wgt_amt)

# Check if any category nbr are null (which will interfere with List Import and would need to be updated first)
check_cat_nbr = lmc_list_focus_brands[['cmc_cat_nbr','upc_cd']].groupby(['cmc_cat_nbr']).count()
print(check_cat_nbr)

# Check if any upc group names have comma, if yes then will cause the hierarchy load to FAIL
lmc_list_focus_brands['has_commas'] = lmc_list_focus_brands.group_name.str.contains(",")
comma_check=lmc_list_focus_brands[['group_number','group_name','has_commas']].drop_duplicates()
comma_check.has_commas[comma_check.has_commas == True] = '!!!!!!!!!!!HAS COMMAS!!!!!!!!!!'
comma_check.has_commas[comma_check.has_commas == False] = 'NO'

# IF ANY HAVE COMMAS, can get rid of commas in main file by uncommenting this, then rerun this cell to check
# lmc_list_focus_brands['group_name'] = lmc_list_focus_brands['group_name'].astype(str).apply(lambda x: x.replace(',','_'))
print(comma_check)


# Check the highest # of digits of all upcs. If any have 13 or more digits his will cause list import to fail.
max(lmc_list_focus_brands['upc_cd'].str.len())

# Prep upc file to load
load_file = lmc_list_focus_brands[['upc_cd','cmc_cat_nbr','group_number','group_name','tot_wgt_amt','tot_wgt_meas']].copy()
load_file['cmc_cat_nbr'] = load_file['cmc_cat_nbr'].fillna(1) #replace null categories with 1
load_file['tot_wgt_amt'] = load_file['tot_wgt_amt'].fillna(1) #replace null eq amt with 1
load_file['tot_wgt_meas'] = load_file['tot_wgt_meas'].fillna('CT') #replace null eq meas with CT
load_file['upc_length']=lmc_list_focus_brands['upc_cd'].str.len() # calculates number of digits for each upc_cd
load_file = load_file[load_file['upc_length'] <= 12] # filters out any upcs that are more than 12 digits and that would cause list import to fail
del load_file['upc_length'] #drop the upc_length column because we are done with it

# add columns for upload
load_file.insert(2,'NULL7',np.nan)
load_file.insert(5,'NULL8',np.nan)
load_file.insert(6,'NULL9',np.nan)
load_file.insert(7,'NULL10',np.nan)

load_file = load_file.rename(columns={'cmc_cat_nbr': 'cat_nbr'}) #rename to cat_nbr
load_file = load_file.rename(columns={'group_number': 'brand_nbr'}) #rename to brand_nbr
load_file = load_file.rename(columns={'group_name': 'brand_name'}) #rename to brand_nm
load_file = load_file.rename(columns={'tot_wgt_amt': 'eq_vol'}) #rename to eq_vol
load_file = load_file.rename(columns={'tot_wgt_meas': 'eq_meas'}) #rename to eq_meas

load_file = load_file.astype({'upc_cd': 'int64','cat_nbr': 'int64','brand_nbr': 'int64','eq_vol': 'float64'}) #format all fields

print(load_file)

upc_list_df = load_file

upc_file = upc_list_df[['brand_nbr', 'brand_name','upc_cd']]
upc_file = upc_file.rename(columns={'brand_nbr':'brand_nbr', 'brand_name':'brand_desc','upc_cd':'trade_item_cd'})
upc_file['brand_desc'] = upc_file['brand_desc'].replace(' ', '_')
upc_file = upc_file.reset_index(drop=True, inplace=False)

upc_file['brand_desc'] = upc_file['brand_desc'].str.replace('\xa0','')
upc_file['brand_desc'] = upc_file['brand_desc'].str.replace('\u200b','')

print(upc_file)


                  upc_cd                         upc_descr  \
id                                                           
1018100005    1018100005  +PBFSS SS LPBLM PCTBE N/AV  .5OZ   
1018103265    1018103265  +PCOFR SS SKMZR PBPMP CCN 13.5OZ   
1018103270    1018103270  +PCOFR SS SKMZR PLSPR REG  5.1OZ   
1018103271    1018103271  +PCOFR SS SKMZR PBPMP BZLN 6.5OZ   
1018103275    1018103275  +PCOFR SS SKMZR PLTBE REG  3.4OZ   
...                  ...                               ...   
99999435811  99999435811  +ENGRZ  BLGT4 N/AV           1CT   
9999964677    9999964677  +OSPHE N/AV SOP PLBTP PFS 47.2OZ   
9999964680    9999964680  +OSPHE N/AV SOP PLBTP N/AV  36OZ   
99999999998  99999999998  +CPRA1  SOP BX N/AV      N 5.3OZ   
99999999999  99999999999  +7SLC* SS INALG BLPKB REG    4CT   

                                   recipe_txt  brand_nbr  \
id                                                         
1018100005   +PBFSS SS LPBLM PCTBE N/AV  .5OZ        0.0   
1018103265   

<ipython-input-5-7346fccba3ed>:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comma_check.has_commas[comma_check.has_commas == False] = 'NO'


            group_number       group_name has_commas
id                                                  
1018100005             1  Promoted Beauty         NO
100000151              2  Category Beauty         NO
                  upc_cd  cat_nbr  NULL7  brand_nbr       brand_name  NULL8  \
id                                                                            
1018100005    1018100005     7503    NaN          1  Promoted Beauty    NaN   
1018103265    1018103265     7501    NaN          1  Promoted Beauty    NaN   
1018103270    1018103270     7501    NaN          1  Promoted Beauty    NaN   
1018103271    1018103271     7501    NaN          1  Promoted Beauty    NaN   
1018103275    1018103275     7501    NaN          1  Promoted Beauty    NaN   
...                  ...      ...    ...        ...              ...    ...   
99999435811  99999435811     1890    NaN          2  Category Beauty    NaN   
9999964677    9999964677     3411    NaN          2  Category Beauty    NaN  

upc_file.loc[upc_file['trade_item_cd']==9110832025].brand_desc.values[0]

file_nm = '12.22 Emerson Wellness Brand.xlsx'
file_path = "/analytical_services/Public/moricejuan/" + file_nm

group = pd.read_excel(file_path, sheet_name='Group')
group = group[['Group Number', 'Group Name']]

upc_file = pd.read_excel(file_path, sheet_name='UPC')
upc_file = upc_file[['Group Number', 'UPC']]
upc_file = upc_file.merge(group, on='Group Number', how='left')
upc_file = upc_file.rename(columns={'Group Number':'brandnbr', 'Group Name':'branddesc','UPC':'tradeitemcd'})
upc_file

In [6]:
yb_load(Df = upc_file,
            table_name = f'''VMR_{campaign_nm}_upclmc_{analyst}''',
            userid = getpass.getuser(),
            passwd = readpw("Yellowbrick"),
            append = False,
            database= f'py1usta1')


check_upload = pd.read_sql(f"""select count(*) from VMR_{campaign_nm}_upclmc_{analyst}""", conn)
result = 'Uploaded number of records: ' + str(check_upload.values[0])
print('Uploaded number of records: ' + str(check_upload.values[0]))
print("")

Checking that the save and YBtools locations

Checking and removing if a table by the same name exists


Creating table


Here is your new table structure


***************************************************************

	 FIELD: 	 Pandas -> YB 

	 brandnbr: 	 int64 -> BIGINT
	 branddesc: 	 object -> VARCHAR( 15 )
	 tradeitemcd: 	 int64 -> BIGINT
***************************************************************

Saving data frame to the drive for loading into YB

Bulk loading has commenced

removing pandas saved file
Uploaded number of records: [696402]



In [7]:
############################## DON'T CHANGE ################################## 

all_combos = False         

#PowerPoint final deck:
template_path = "/analytical_services/Public/moricejuan/VMR_pptx_tests/VMR_Scorecard_Template - Copy.pptx"
#The code will take the PPTX template to build the final deck from this path. Don't change it!


#Modified from K.Mertens's J&J custom VMR code.

# double confirm that there is an export path ... if not, add ./ for current location
if export_path == "" :
    export_path = "./"
else :
    export_path = export_path

## Converting "BL-..." format to "USA-BLIP-BL...":


for i in range(len(BL_CODES)):
    
    if BL_CODES[i][0] == 'B' and BL_CODES[i][1] == 'L' and BL_CODES[i][2] == '_':
        
        BL_CODES[i] = str(''.join(('USA-BLIP-',BL_CODES[i])) )

if Announcement != ['']:
    
    for i in range(len(Announcement)):

        if Announcement[i][0] == 'B' and Announcement[i][1] == 'L' and Announcement[i][2] == '_':

            Announcement[i] = str(''.join(('USA-BLIP-',Announcement[i])) )


        
        
# connection string to server - this is quick connection to check UPCs
conn = ybconn.connect(user=getpass.getuser(), password=readpw("Yellowbrick"), 
                      host='orlpybvip01.catmktg.com', port='5432', 
                      database = f'{dbase}')
conn.set_session(autocommit=True)
curs = conn.cursor()

 

df_upcs_quick = pd.read_sql(f'''

    select 000 as list_nbr
      ,brandnbr as brand_nbr
      ,branddesc as brand_desc
      ,count(distinct tradeitemcd) as nbr_of_upcs
    from VMR_{campaign_nm}_upclmc_{analyst}
    group by 1,2,3
''',conn).sort_values(by=['list_nbr','brand_nbr']).reset_index().drop(columns=['index'])


brand_nbr_str = ",".join(str(i) for i in brand_nbr)
cat_nbr_str = ",".join(str(i) for i in cat_nbr)

BL_CODES = str(BL_CODES) 
BL_CODES = BL_CODES.replace("[","").replace("]","")

Announcement = str(Announcement) 
Announcement = Announcement.replace("[","").replace("]","")

#Segment Definition:

place_holder_1 = ''
place_holder_2 = ''
place_holder_3 = ''
place_holder_4 = ''
place_holder_5 = ''


if segment_type == 1:
    segment_def = 'brand_desc'

if segment_type == 2:
    segment_def = 'trademark_brand_desc'
    
if segment_type == 3:
    segment_def = 'cat_desc'
    
if segment_type == 4:
    
    upload_descr = pd.read_excel(custom_brand_desc_path)
    upload_descr = upload_descr.replace(np.nan, 'All Other Categories', regex=True)
    columns_nm = list(upload_descr.columns)
    upload_descr = upload_descr.rename(columns={columns_nm[0]:'custombranddescr',columns_nm[1]:'catdesc'})
    
    
    yb_load(Df = upload_descr,
            table_name = f'''VMR_{brand_nm}_brand_descriptions_{analyst}''',
            userid = getpass.getuser(),
            passwd = readpw("Yellowbrick"),
            append = False,
            database= f'{dbase}')
    
    place_holder_1 = f''', f.custombranddescr as custom_brand_desc '''
    place_holder_2 = f'''inner join VMR_{brand_nm}_brand_descriptions_{analyst} f ON (f.catdesc = p.cat_desc)'''
    segment_def = f'''custom_brand_desc'''
    

if segment_type == 5:
    
    lb_def = f'''l{label_level}'''
    
    place_holder_3 = f''',z.owner_nm'''
    
    place_holder_4 = f''',z.trade_item_hier_{lb_def}_desc'''
    
    place_holder_5 = f'''inner join trade_item_owner_hierarchy_v z on p.trade_item_key = z.trade_item_key'''
        
    segment_def = f'''trade_item_hier_{lb_def}_desc'''
    
    
if segment_type == 6:
    
    lb_def = f'''l{label_level}'''
    
    place_holder_4 = f''',z.tradeitemhier{lb_def}desc'''
    
    place_holder_5 = f'''inner join py1ussa1.public.nz_hierarchy z on p.trade_item_key = z.tradeitemkey'''
        
    segment_def = f'''tradeitemhier{lb_def}desc'''
    
    

print('UPC Quick Check: ')
print(df_upcs_quick)
print("")
print('Quick Check: ')
print('The BL codes to perfom the analysis are: ' + str(BL_CODES))
print("Focus brand(s):",brand_nbr_str)
print("Category brands:",cat_nbr_str)
print("Segment definition string:",segment_def)
print(" ")

if segment_type == 4:
    print("Custom Brand's descriptions uploaded:")
    print(upload_descr)
    

UPC Quick Check: 
   list_nbr  brand_nbr       brand_desc  nbr_of_upcs
0         0          1  Promoted Beauty        12663
1         0          2  Category Beauty       683739

Quick Check: 
The BL codes to perfom the analysis are: 'USA-BLIP-BL_4303_0667'
Focus brand(s): 1
Category brands: 1,2
Segment definition string: brand_desc
 


In [8]:
#Renaming dictionary: (taken from K.Mertens J&J VMR code)

renaming_dict = {
    'ord_event_key' : 'ord_event_key',
    'cnsmr_id_key' : 'cnsmr_id_key',
    'analysis_period_p1p2': 'Analysis Period', 
    'analysis_period_rec52': 'Analysis Period', 
    'cal_sun_wk_ending_dt': 'WE Date',
    'nbr_of_upcs': 'Raw # of UPCs included',
    'brand_desc': 'Brand',
    'sur_seg': 'Purchasing Segment',
    'price_seg': 'Pricing Segment',
    'fin_cmit_contract_nbr': 'Contract Nbr',
    'promo_src_id_txt': 'BL',
    'analysis_periods':'Analysis Periods',
    'start':'Start', 'end':'End',
    'count_distinct_trips':'Count distinct trips',
    'dollar_sales':'Dollar Sales',
    'dollars_per_trip':'Dollars Per Trip',
    '%_change_dollars_per_trip':'% Change Dollars Per Trip',
    'units':'Units',
    'dollars':'Dollars',
    'trips':'Trips',
    'units_per_trip':'Units Per Trip',
    'dollars_per_trip_yago_period':'Dollars Per Trip YAGO Period',
    'dollars_per_trip_vmr_period':'Dollars Per Trip VMR Period',
    'units_per_trip_yago_period':'Units Per Trip YAGO Period',
    'units_per_trip_vmr_period':'Units Per Trip VMR Period',
    'dollars_per_trip_pre_period':'Dollars Per Trip Pre Period',
    'units_per_trip_pre_period':'Units Per Trip Pre Period',
   
}

## **2) DIMESIONAL TABLES AND DATE CALCULATIONS:**

2a) Creating Basic Dimensional tables:

In [9]:
#Modified from K.Mertens's J&J custom VMR code.

#FILTERING BY REWARD PRINTS (BLs):

curs.execute(f'''

    DROP table if exists VMR_{brand_nm}_promo_filter;
    CREATE temp table VMR_{brand_nm}_promo_filter as
    
        SELECT distinct pv.promo_src_id_txt
            ,pv.promo_src_id
            ,pv.promo_varnt_key
            ,pv.promo_key
            
        FROM promotion_variant_v pv 
        
        WHERE pv.promo_src_id_txt in ({BL_CODES})
        
        DISTRIBUTE ON (promo_varnt_key);
        
''')

#FILTERING BY TOUCH POINT FIELDS:

curs.execute(f'''

    DROP table if exists VMR_{brand_nm}_touchpoint_filter;       
    CREATE temp table VMR_{brand_nm}_touchpoint_filter as
        
        SELECT tp.touchpoint_key 
               ,tp.acct_specific_rtlr_nbr 
               ,tp.acct_specific_rtlr_nm 
               ,tp.site_key 
               ,tp.ntwk_id 
               ,tp.ntwk_nm 
               ,tp.csdb_chn_nbr
               
        FROM touchpoint_v tp 

       -- WHERE acct_specific_rtlr_nbr = -1

        DISTRIBUTE ON(touchpoint_key);      
        
''')

#GETTING DETAILS ABOUT EVERY TRIP FILTERING BY BLS:

curs.execute(f''' 

    DROP table if exists VMR_{brand_nm}_print_dt_actual;
    CREATE temp table VMR_{brand_nm}_print_dt_actual as

        SELECT
             p.promo_src_id_txt
            ,p.promo_src_id
            ,a.event_typ_cd
            ,a.ord_date_key
            ,d.cal_dt
            ,tp.ntwk_id
            ,tp.ntwk_nm
            ,tp.acct_specific_rtlr_nbr
            ,tp.acct_specific_rtlr_nm
            ,tp.site_key
            ,tp.csdb_chn_nbr
            ,tp.touchpoint_key 
            ,sum(a.tot_selected_qty) as prints

        FROM ord_promo_varnt_cnsmr_ne_v a 

            INNER JOIN VMR_{brand_nm}_promo_filter p ON (a.promo_varnt_key=p.promo_varnt_key)
            INNER JOIN VMR_{brand_nm}_touchpoint_filter tp ON (a.ord_touchpoint_key=tp.touchpoint_key)
            INNER JOIN date_v d ON (d.date_key = a.ord_date_key)

        WHERE a.event_typ_cd in ('IS-SPRINT-T','IS-PRINT-T','STCNTL_CMPGN','STCNTL_SITE','IS-SPRINT-vd','LNG_TRM_CNTL')
             -- and cal_dt > '2024-11-18'

        GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12

        DISTRIBUTE ON(ord_date_key);
    
''')

#CREATING UPC TABLE:

curs.execute(f'''
    drop table if exists VMR_{brand_nm}_upc_filter;       
    create temp table VMR_{brand_nm}_upc_filter as
        select 0 as list_nbr
              ,(b.brandnbr) as brand_nbr
              ,(b.branddesc) as brand_desc
              ,p.trademark_brand_desc
              ,p.trade_item_key
              ,b.tradeitemcd as trade_item_cd
              ,p.cat_nbr
              ,p.cat_desc
              {place_holder_3}
              {place_holder_4}
              {place_holder_1}
        from trade_item_v p 
              inner join VMR_{campaign_nm}_upclmc_{analyst} b on (p.trade_item_cd = b.tradeitemcd)
              {place_holder_2}
              {place_holder_5}
            
        distribute replicate;
''')


#GETTING PROMO DETAILS FOR SUMMARY TABLE:

curs.execute(f''' 

    DROP table if exists VMR_{brand_nm}_promo_check;
    CREATE temp table VMR_{brand_nm}_promo_check as
    
        SELECT distinct fin_cmit_contract_nbr 
               ,fin_cmit_contract_nm
               ,promo_src_id_txt
               ,promo_desc_txt
              ,promo_funding_typ_cd
              ,promo_discnt_val_amt
              ,rolling_wk_vld_qty
        
        FROM promotion_variant_v pv 
        
        WHERE pv.promo_src_id_txt in ({BL_CODES})
        
    DISTRIBUTE REPLICATE;
    
''')

#CREATING PROMO SUMMARY TABLE:

curs.execute(f''' 

    DROP table if exists VMR_{brand_nm}_promo_summary;
    CREATE temp table VMR_{brand_nm}_promo_summary as

        SELECT p.fin_cmit_contract_nm
               ,p.fin_cmit_contract_nbr
               ,p.promo_src_id_txt
               ,p.promo_desc_txt
               ,min(cal_dt) as actual_start
               ,max(cal_dt) as actual_end
               ,sum(prints) as total_reward_prints
              
        FROM VMR_{brand_nm}_print_dt_actual a
        
            INNER JOIN VMR_{brand_nm}_promo_check p on (a.promo_src_id_txt = p.promo_src_id_txt)
            
        GROUP BY 1,2,3,4

        DISTRIBUTE REPLICATE;
    
''')


#Dimesional table that pull-outs print dates for every chain:

curs.execute(f''' 
    DROP table if exists VMR_{brand_nm}_actual_dates_by_chain;
    CREATE temp table VMR_{brand_nm}_actual_dates_by_chain as
    
            SELECT csdb_chn_nbr
                  ,min(cal_dt) as actual_start
                  ,max(cal_dt) as actual_end
                  ,min(ord_date_key) as min_actual_date_key
                  ,max(ord_date_key) as max_actual_date_key
            
            FROM VMR_{brand_nm}_print_dt_actual a
            
            GROUP BY 1
            
            DISTRIBUTE REPLICATE;
''',conn)


#Dimensional table that pull-outs the printing stores from chains filtered in the last dimensiona table:

curs.execute(f''' 
    DROP table if exists VMR_{brand_nm}_printing_stores;
    CREATE temp table VMR_{brand_nm}_printing_stores as 
            
            SELECT distinct tp.site_key,tp.touchpoint_key, tp.acct_specific_rtlr_nm, a.min_actual_date_key, a.max_actual_date_key

            FROM touchpoint_v tp 

                INNER JOIN VMR_{brand_nm}_actual_dates_by_chain a on tp.csdb_chn_nbr = a.csdb_chn_nbr

            DISTRIBUTE ON(touchpoint_key);
''')

In [10]:
 pd.read_sql(f''' SELECT *
 FROM VMR_{brand_nm}_print_dt_actual
 order by cal_dt ASC
''',conn)

,promo_src_id_txt,promo_src_id,event_typ_cd,ord_date_key,cal_dt,ntwk_id,ntwk_nm,acct_specific_rtlr_nbr,acct_specific_rtlr_nm,site_key,csdb_chn_nbr,touchpoint_key,prints
0,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260228,2026-02-28,79,Walgreens,39,Walgreens Boots Alliance Inc,113400013,398,906179409,1
1,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260228,2026-02-28,79,Walgreens,39,Walgreens Boots Alliance Inc,104700016,398,816178529,1
2,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260228,2026-02-28,79,Walgreens,39,Walgreens Boots Alliance Inc,93900020,398,927578526,1
3,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260228,2026-02-28,79,Walgreens,39,Walgreens Boots Alliance Inc,117200026,398,825978392,1
4,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260228,2026-02-28,79,Walgreens,39,Walgreens Boots Alliance Inc,71600018,398,793279179,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
510608,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260328,2026-03-28,79,Walgreens,39,Walgreens Boots Alliance Inc,60700019,387,799279108,1
510609,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260328,2026-03-28,79,Walgreens,39,Walgreens Boots Alliance Inc,183900022,389,826577491,1
510610,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260329,2026-03-29,79,Walgreens,39,Walgreens Boots Alliance Inc,46300014,391,801878373,1
510611,USA-BLIP-BL_4303_0667,671817089,IS-SPRINT-T,20260329,2026-03-29,79,Walgreens,39,Walgreens Boots Alliance Inc,176600017,389,789680343,1


In [11]:
 pd.read_sql(f''' SELECT promo_src_id_txt, max(cal_dt), min(cal_dt)
 FROM VMR_{brand_nm}_print_dt_actual
 group by 1
''',conn)

,promo_src_id_txt,max,min
0,USA-BLIP-BL_4303_0667,2026-03-29,2026-02-28


In [12]:
pd.read_sql(f''' SELECT * FROM VMR_{brand_nm}_upc_filter
''',conn)

,list_nbr,brand_nbr,brand_desc,trademark_brand_desc,trade_item_key,trade_item_cd,cat_nbr,cat_desc
0,0,2,Category Beauty,NOBRAND,724349661820006,72434966182,9799,MISCELLANEOUS - UNDEFINED
1,0,2,Category Beauty,LOREAL,71249315120006,7124931512,7107,HAIR COLORING
2,0,2,Category Beauty,SIMPLY FOOT,71603131140006,7160313114,9202,BEAUTY TOOLS
3,0,2,Category Beauty,WEGMANS,77890513730006,7789051373,2630,FACIAL TISSUE
4,0,2,Category Beauty,FX,22796874450006,2279687445,7114,HAIR CARE - STYLING
...,...,...,...,...,...,...,...,...
696383,0,2,Category Beauty,H E B,41220818220006,4122081822,7358,DIETARY SUPPLEMENT - HERBAL
696384,0,2,Category Beauty,H E B,41220826490006,4122082649,9799,MISCELLANEOUS - UNDEFINED
696385,0,2,Category Beauty,H E B,41220957010006,4122095701,1890,HOUSEHOLD SUPPLIES - OTHER
696386,0,2,Category Beauty,H E B,41220978080006,4122097808,9799,MISCELLANEOUS - UNDEFINED


2b) Identifying number of retailers involved to define static parameters:

In [13]:
nbr_retailers = pd.read_sql(f''' SELECT count(distinct acct_specific_rtlr_nm)
                                 FROM VMR_{brand_nm}_printing_stores
''',conn)

nbr_retailers = nbr_retailers['count'].values[0]
print("Number of retailers for this analysis: " + str(nbr_retailers))


retailers = pd.read_sql(f''' SELECT distinct acct_specific_rtlr_nm FROM VMR_{brand_nm}_printing_stores''',conn)
retailers = list(retailers['acct_specific_rtlr_nm'])
retailers = str(retailers) 
retailers = retailers.replace("[","").replace("]","")
retailers = retailers.replace("'","")
print("Retailers involved in this analysis:", retailers)

if segment_type == 5:
    
    retail_owner_df = pd.read_sql(f'''SELECT acct_specific_rtlr_nm, owner_nm
                                      FROM ord_trd_itm_cnsmr_fact_ne_v a
                                      INNER JOIN VMR_{brand_nm}_upc_filter b ON (a.trade_item_key = b.trade_item_key)
                                      INNER JOIN VMR_{brand_nm}_printing_stores c ON (a.ord_touchpoint_key = c.touchpoint_key)
                                      GROUP BY 1,2
    ''',conn)

    #print(retail_owner_df)

    owner_list = []

    retail_unique = list(retail_owner_df['acct_specific_rtlr_nm'].unique())

    for i in range(len(retail_unique)): 
        for j in range(len(retail_owner_df['owner_nm'])):
            if retail_owner_df['owner_nm'][j] in retail_owner_df['acct_specific_rtlr_nm'][i]:   
                owner_list.append(retail_owner_df['owner_nm'][j])
                
                
    owner_list = str(owner_list) 
    owner_list = owner_list.replace("[","").replace("]","")
    owner_list = owner_list.replace("'","")
    
    place_holder_6 = f'''and owner_nm = '{owner_list}' '''

    print("Retailers descriptions source:", owner_list)


# static variables:  static_1 --> at least X trips; static_2 --> every Y-week block; 
# static_3 --> Z number of time blocks;  static_4 --> through lag week number

if nbr_retailers < 3:
    static_1 = 1
    static_2 = 52
    print('Static: ',static_1,'trips every',static_2,'weeks')

else:
    static_1 = 2
    static_2 = 8
    print('Static: ',static_1,'trips every',static_2,'weeks')

    
if int(nbr_retailers) == 1 and retailers == '"Kroger/Roundys/Harris Teeter"':
    category_show_parameter = False
else:
    category_show_parameter = True

Number of retailers for this analysis: 1
Retailers involved in this analysis: Walgreens Boots Alliance Inc
Static:  1 trips every 52 weeks


In [14]:
#Getting dominance report for the BLs used:

qry_parent_prints= f''' 


    select 
    b.promo_src_id_txt as blip,
    d.acct_specific_rtlr_nbr,
    d.acct_specific_rtlr_nm,
    sum(a.tot_selected_qty) as prints

    from ord_promo_varnt_cnsmr_ne_v a 
    inner join promotion_variant_v b on 
    a.promo_varnt_key=b.promo_varnt_key  
    inner join touchpoint_v d on 
    a.ord_touchpoint_key=d.touchpoint_key 

    where a.event_typ_cd in ('IS-SPRINT-T','IS-PRINT-T') and
    b.promo_src_id_txt in ({BL_CODES})

    group by 1,2,3

    '''
df_parent_prints = pd.read_sql(qry_parent_prints, conn)

# parent pivot for prints
df_parent_pivot = pd.pivot_table(df_parent_prints,'prints',index=['acct_specific_rtlr_nbr','acct_specific_rtlr_nm'],columns='blip',aggfunc='sum',margins=True,fill_value=0)
df_parent_pivot.style.format("{:,.0f}")

# calculate column %s for blips by parent
df_parent_pivot_2 = df_parent_pivot.div(df_parent_pivot.iloc[-1,:], axis=1)
df_parent_pivot_fin = df_parent_pivot_2.style.format("{:,.2%}")
df_parent_pivot_2 = df_parent_pivot_2.reset_index()

if len(df_parent_pivot_2.loc[(df_parent_pivot_2['acct_specific_rtlr_nbr']==-2) & (df_parent_pivot_2['All']>=0.45)])>0:
    dominant_retailer = df_parent_pivot_2.loc[(df_parent_pivot_2['acct_specific_rtlr_nbr']==-2) & (df_parent_pivot_2['All']>=0.45)].acct_specific_rtlr_nm.values[0]
    dominance = 'Kroger'
    print("Kroger Dominance")
elif len(df_parent_pivot_2.loc[((df_parent_pivot_2['acct_specific_rtlr_nbr']!=-2) & (df_parent_pivot_2['acct_specific_rtlr_nbr']!='All')) & (df_parent_pivot_2['All']>=0.7)])>0:
    dominant_retailer = df_parent_pivot_2.loc[((df_parent_pivot_2['acct_specific_rtlr_nbr']!=-2) & (df_parent_pivot_2['acct_specific_rtlr_nbr']!='All')) & (df_parent_pivot_2['All']>=0.7)].acct_specific_rtlr_nm.values[0]
    dominance = 'Retailer'
    print("Retailer Dominance")
else:
    dominance = 'None'
    print("No Dominance")

df_parent_pivot_fin

Retailer Dominance


,blip,USA-BLIP-BL_4303_0667,All
acct_specific_rtlr_nbr,acct_specific_rtlr_nm,,
39,Walgreens Boots Alliance Inc,100.00%,100.00%
All,,100.00%,100.00%


In [15]:
#When we don't want to apply Kroger Dominance

dominance = 'None'
#

2c) Defining type of segment will be used to perform the analysis based on parameter selection made by the analyst:

In [16]:
if segment_type == 1:
    df_segments_2 = pd.read_sql(f''' select {segment_def} as segment, count(*) as nbr_of_upcs, brand_nbr as segm_nbr from VMR_{brand_nm}_upc_filter where brand_nbr in ({brand_nbr_str}) group by 1,3 order by 3 ''',conn) 
    print(df_segments_2)
    
if segment_type == 2:
    df_segments_2 = pd.read_sql(f''' select {segment_def} as segment, count(*) as nbr_of_upcs from VMR_{brand_nm}_upc_filter where brand_nbr in ({brand_nbr_str}) group by 1 order by 1 ''',conn) 
    df_segments_2['segm_nbr'] = list(range(1,len(df_segments_2)+1))
    print(df_segments_2)
    
if segment_type == 3:
    df_segments_2 = pd.read_sql(f''' select {segment_def} as segment, count(*) as nbr_of_upcs from VMR_{brand_nm}_upc_filter where brand_nbr in ({brand_nbr_str}) group by 1 order by 1 ''',conn) 
    df_segments_2['segm_nbr'] = list(range(1,len(df_segments_2)+1))
    print(df_segments_2)    
    
if segment_type == 4:
    df_segments_2 = pd.read_sql(f''' select {segment_def} as segment, count(*) as nbr_of_upcs from VMR_{brand_nm}_upc_filter where brand_nbr in ({brand_nbr_str}) group by 1 order by 1 ''',conn) 
    df_segments_2['segm_nbr'] = list(range(1,len(df_segments_2)+1))
    print(df_segments_2)    

if segment_type == 5:
    df_segments_2 = pd.read_sql(f''' select {segment_def} as segment, count(*) as nbr_of_upcs from VMR_{brand_nm}_upc_filter where brand_nbr in ({brand_nbr_str}) and owner_nm = '{owner_list}' group by 1 order by 1 ''',conn) 
    df_segments_2['segm_nbr'] = list(range(1,len(df_segments_2)+1))
    print(df_segments_2)    
    
if segment_type == 6:
    df_segments_2 = pd.read_sql(f''' select {segment_def} as segment, count(*) as nbr_of_upcs from VMR_{brand_nm}_upc_filter where brand_nbr in ({brand_nbr_str}) group by 1 order by 1 ''',conn) 
    df_segments_2['segm_nbr'] = list(range(1,len(df_segments_2)+1))
    print(df_segments_2)    

           segment  nbr_of_upcs  segm_nbr
0  Promoted Beauty        12649         1


In [17]:
if segment_type == 5:

    curs.execute(f'''
        drop table if exists VMR_{brand_nm}_upc_filter;       
        create temp table VMR_{brand_nm}_upc_filter as
            select 0 as list_nbr
                  ,(b.brandnbr) as brand_nbr
                  ,(b.branddesc) as brand_desc
                  ,p.trademark_brand_desc
                  ,p.trade_item_key
                  ,b.tradeitemcd as trade_item_cd
                  ,p.cat_nbr
                  ,p.cat_desc
                  {place_holder_3}
                  {place_holder_4}
                  {place_holder_1}
            from trade_item_v p 
                  inner join VMR_{campaign_nm}_upclmc_{analyst} b on (p.trade_item_cd = b.tradeitemcd)
                  {place_holder_2}
                  {place_holder_5}
            
                  {place_holder_6}

            distribute replicate;
    ''')

2d) Displaying the offer's summary:

In [18]:
df_promo_summary = pd.read_sql(f''' select * from VMR_{brand_nm}_promo_summary''',conn)
df_promo_summary = df_promo_summary.rename(columns = {'fin_cmit_contract_nbr': 'Contract Nbr','promo_src_id_txt': 'BL',
'fin_cmit_contract_nm': 'Fin Cmit Contract Nm', 'promo_desc_txt':'Promo Desc Txt', 'actual_start': 'Actual Start',
'actual_end':'Actual End', 'total_reward_prints':'Total Reward Prints'})
df_promo_summary

,Fin Cmit Contract Nm,Contract Nbr,BL,Promo Desc Txt,Actual Start,Actual End,Total Reward Prints
0,March Beauty Event S$25G$10-FY26- CAT,452858,USA-BLIP-BL_4303_0667,Ad03_TR_500_Spend $25 or more on promoted Beau...,2026-02-28,2026-03-29,2502676


In [19]:
df_promo_summary['Total Reward Prints'].sum()

2502676

2e) Date calculations for the analysis:

In [20]:
#Modified code from K.Mertens's J&J custom VMR code.

#reward_print_start = dt.date(2025, 7, 13)

#Reward print period from ord_date_key data:

reward_print_start = df_promo_summary['Actual Start'].min()
reward_print_end = df_promo_summary['Actual End'].max()

weeks_per_period_final = round(((reward_print_end - reward_print_start + dt.timedelta(days=1))/dt.timedelta(days=7)),1)


nbr_days = reward_print_end - reward_print_start
nbr_days = nbr_days.days

#number of weeks the VMR print period lasts


#VMR pre-period dates: (using the same program lenght as the reward print period)

vmr_pre_period_start = reward_print_start - dt.timedelta(days = nbr_days+1)
vmr_pre_period_end = reward_print_start - dt.timedelta(days=1)


#52 Prior-Period: (using 52 parameter)

prior_period_start = reward_print_start - dt.timedelta(weeks = pre_weeks) #pre_weeks before the reward print period starts
prior_period_end = reward_print_start - dt.timedelta(days=1) #one day before the reward print period starts


#VMR Period YAGO (same length as VMR Period):

yago_print_start = reward_print_start - dt.timedelta(weeks = 52) #52 weeks before the reward print period starts
yago_print_end = reward_print_end - dt.timedelta(weeks = 52) #52 weeks before the reward print period starts 


#Reward Period + Redemption days:

redemption_start = reward_print_start  #the BL could be redeemed the same day it was printed
redemption_end = reward_print_end + dt.timedelta(days = redemption_days)  #the last possible redemption could be the last printing day + redemption_days


#Reward Period + 4 WEEK

post_4wk_start = reward_print_end + dt.timedelta(days = 1) #the BL could be redeemed the same day it was printed
post_4wk_end = reward_print_end + dt.timedelta(days = nbr_days+1)  
#post_4wk_end = reward_print_end + dt.timedelta(days = 108)  


#Pre weeks for PPTX Trend

vmr_pre26wk_start =  reward_print_start - dt.timedelta(weeks = 26)
vmr_pre26wk_end = reward_print_start - dt.timedelta(days = 1)


yago_26wk_start = vmr_pre26wk_start - dt.timedelta(weeks = 52)

#Lasta date of data:
last_data_dt =  (dt.date.today()- dt.timedelta(days = 2))


#Static calculations

curr_date = dt.date.today()
sun_offset = 1 + curr_date.weekday()
rec_sun = curr_date - dt.timedelta(days=sun_offset)

look_back = round(((curr_date - prior_period_start)/dt.timedelta(days=1) + 1)/7,1)

static_3 = int(round(((reward_print_end - prior_period_start)/dt.timedelta(days=1) + 1)/(7*static_2),2))
static_4 = int(round((((rec_sun - reward_print_end)/dt.timedelta(days=1) + 1)/7),0)) 


In [21]:
post_4wk_end

datetime.date(2026, 4, 28)

In [22]:
reward_print_start

datetime.date(2026, 2, 28)

In [23]:
print(curr_date)
print(yago_26wk_start)

2026-04-14
2024-08-31


2f) Dimensional table for dates:

In [24]:
#Dates dimensional table

curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_date_filter;
    CREATE temp table VMR_{brand_nm}_date_filter as
    
    
        SELECT distinct date_key,cal_dt,cal_sun_wk_ending_dt,cal_sun_wk_ending_rank_nbr
              ,CASE 
                    when cal_dt between '{reward_print_start}' and '{reward_print_end}' then 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
                    when cal_dt between '{vmr_pre_period_start}' and '{vmr_pre_period_end}' then 'VMR Pre-period ({round(weeks_per_period_final)} weeks)'
                    when cal_dt between '{yago_print_start}' and '{yago_print_end}' then 'Period Year Ago ({round(weeks_per_period_final)} weeks)'
                    end as analysis_period
              ,CASE when cal_dt between '{prior_period_start}' and '{prior_period_end}' then 'Prior-period ({round(pre_weeks)} weeks)'
                    end as prior_period
              ,CASE when cal_dt between '{redemption_start}' and '{redemption_end}' then 'Redemption Period ({redemption_days} days)'
                    end as redemption_period
              ,CASE when cal_dt between '{post_4wk_start}' and '{post_4wk_end}' then 'Post Period ({round(weeks_per_period_final)} weeks)'
                    end as post_period
                    
        FROM date_v
        
        WHERE cal_dt between '{yago_26wk_start}' and '{curr_date}'
        
        
    DISTRIBUTE REPLICATE;

''')


In [25]:
#Getting summarize table with all the dates

df_date_check = pd.read_sql(f'''

    WITH union_table AS 
    
    (SELECT CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period - TY ({round(weeks_per_period_final)} weeks)'
                WHEN analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period - Pre Period ({round(weeks_per_period_final)} weeks)'
                WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period - YAGO ({round(weeks_per_period_final)} weeks)'
                ELSE null END as analysis_periods
    , min(cal_dt) as start, max(cal_dt) as end 
    FROM VMR_{brand_nm}_date_filter
    WHERE analysis_period is not null
    GROUP BY 1
    UNION
    SELECT CASE WHEN redemption_period = 'Redemption Period ({redemption_days} days)' THEN 'Reward Period + ({redemption_days} days)'
                ELSE null END as analysis_periods
    , min(cal_dt) as start, max(cal_dt) as end 
    FROM VMR_{brand_nm}_date_filter
    WHERE redemption_period = 'Redemption Period ({redemption_days} days)'
    GROUP BY 1
    UNION
    SELECT CASE WHEN prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN '{pre_weeks} wk Prior Period'
           ELSE null END analysis_periods
    , min(cal_dt) as start, max(cal_dt) as end 
    FROM VMR_{brand_nm}_date_filter
    WHERE prior_period = 'Prior-period ({round(pre_weeks)} weeks)'
    GROUP BY 1
    UNION
    SELECT CASE WHEN post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN 'Post Period ({round(weeks_per_period_final)} weeks)'
           ELSE null END analysis_periods
    , min(cal_dt) as start, max(cal_dt) as end 
    FROM VMR_{brand_nm}_date_filter
    WHERE post_period = 'Post Period ({round(weeks_per_period_final)} weeks)'
    GROUP BY 1
    
    )
    
    SELECT *
    FROM union_table
    ORDER BY CASE WHEN analysis_periods = '{pre_weeks} wk Prior Period' THEN 1
               WHEN analysis_periods = 'VMR Period - YAGO ({round(weeks_per_period_final)} weeks)' THEN 2
               WHEN analysis_periods = 'VMR Period - Pre Period ({round(weeks_per_period_final)} weeks)' THEN 3
               WHEN analysis_periods = 'VMR Period - TY ({round(weeks_per_period_final)} weeks)' THEN 4
               WHEN analysis_periods = 'Redemption Period ({redemption_days} days)' THEN 5
               WHEN analysis_periods = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN 6
               END
    ''',conn)

df_date_check = df_date_check.rename(columns={'analysis_periods':'Analysis Periods','start':'Start', 'end':'End'})
df_date_check

,Analysis Periods,Start,End
0,52 wk Prior Period,2025-03-01,2026-02-27
1,VMR Period - YAGO (4 weeks),2025-03-01,2025-03-30
2,VMR Period - Pre Period (4 weeks),2026-01-29,2026-02-27
3,VMR Period - TY (4 weeks),2026-02-28,2026-03-29
4,Post Period (4 weeks),2026-03-30,2026-04-14
5,Reward Period + (14 days),2026-02-28,2026-04-12


2g) Shopper Consistent calculation parameters:

In [26]:
import datetime as dt 
import numpy as np

Current_beg = str(yago_print_start.strftime('%Y-%m-%d'))   # The beginning date of current period; Must be 'YYYY-MM-DD' format!
Current_end = str(post_4wk_end.strftime('%Y-%m-%d')) # The ending date of current period; Must be 'YYYY-MM-DD' format!

# Convert Dates from String type to Date type
Current_beg = dt.datetime.fromisoformat(Current_beg).date()
Current_end = dt.datetime.fromisoformat(Current_end).date()

# Prepare Static Parameters
curr_date = dt.date.today()
sun_offset = 1 + curr_date.weekday()
rec_sun = curr_date - dt.timedelta(days=sun_offset)

static_3 = int(np.floor(((Current_end - Current_beg)/dt.timedelta(days=1) + 1)/(7*static_2)))
static_4 = max(int(np.floor((((rec_sun - Current_end)/dt.timedelta(days=1) + 1)/7))), 0)

# Convert Date type to Numeric type
Current_beg = str(Current_beg)[:10].replace('-','')
Current_end = str(Current_end)[:10].replace('-','')

print('Numeric Begin Date of Current Period:',Current_beg)
print('Numeric End Date of Current Period:',Current_end)

print('\n','Static Information: Based on Current_beg through Current_end', sep='')
print('Last Sunday:',rec_sun.strftime("%A %B %d %Y"))
print('Static: ',static_1,'trips every',static_2,'weeks')
print('Static: ',static_3,'time blocks ending',static_4,'weeks back','\n')


#Dimensional table for shopper-consistency

curs.execute(f''' 
DROP table if exists shopper_consistent_{analyst};
        CREATE temp table shopper_consistent_{analyst} as
                select distinct cnsmr_id_key
                from consumer_id_v
                where abuser_ind= 'N' 
                     and unident_ind = 'N'
                     and really_bad_abuser_ind='N'
                     and cnsmr_id_key>0
                     and isconsistshop(shop_ord_hst_bmap,{static_1},{static_2},{static_3},{static_4})
                DISTRIBUTE ON (cnsmr_id_key);
''')


Numeric Begin Date of Current Period: 20250301
Numeric End Date of Current Period: 20260428

Static Information: Based on Current_beg through Current_end
Last Sunday: Sunday April 12 2026
Static:  1 trips every 52 weeks
Static:  1 time blocks ending 0 weeks back 



## **3) GETTING ALL THE REWARD TRANSACTIONS (ORD_EVENT_KEY) AND REWARD ID'S FOR VMR PERIOD:**

In [27]:
#ALL REWARD TRANSACTIONS FOR VMR PERIOD:

curs.execute(f'''    
    
--Getting all the reward transactions:

    DROP table if exists VMR_{brand_nm}_reward_trans;
    CREATE table VMR_{brand_nm}_reward_trans as
        
          SELECT o.ord_event_key,
                 cal_dt as reward_date,
                 sum(o.purch_qty) as units,
                 sum(o.purch_amt) as dollars
          
          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_printing_stores s ON (o.ord_touchpoint_key = s.touchpoint_key)
          
          WHERE b.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
                and o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str})
                and o.ord_date_key between s.min_actual_date_key and s.max_actual_date_key
        
        GROUP BY 1,2
        
        HAVING {min_threshold_statement}
        
        DISTRIBUTE RANDOM;
        
''')

curs.execute(f'''    
GRANT ALL ON TABLE VMR_{brand_nm}_reward_trans TO public
''')



print(f'''VMR_{brand_nm}_reward_trans''')

curs.execute(f'''    

--Getting the list with the all distinct reward ord_date_key:

    DROP table if exists VMR_{brand_nm}_reward_events;
    CREATE temp table VMR_{brand_nm}_reward_events as
    
        SELECT distinct ord_event_key

        FROM VMR_{brand_nm}_reward_trans

        DISTRIBUTE RANDOM;
        
        --GRANT ALL ON TABLE VMR_{brand_nm}_reward_events TO public
''')

#ALL REWARD ID's FOR VMR PERIOD:

curs.execute(f'''   

--Getting all the Reward transactions and IDs:

    DROP table if exists VMR_{brand_nm}_reward_ids;
    CREATE temp table VMR_{brand_nm}_reward_ids as
        
          SELECT o.ord_event_key,
                 o.ord_designated_cnsmr_id_key, 
                 b.cal_dt as reward_date,
                 sum(o.purch_qty) as units,
                 sum(o.purch_amt) as dollars
          
          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_printing_stores s ON (o.ord_touchpoint_key = s.touchpoint_key)
                 INNER JOIN consumer_id_v p ON (o.ord_designated_cnsmr_id_key = p.cnsmr_id_key)

          
          WHERE p.abuser_ind= 'N' 
                and p.unident_ind = 'N'
                and p.really_bad_abuser_ind='N'
                and p.cnsmr_id_key>0 and
                b.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
                and o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str})
                and o.ord_date_key between s.min_actual_date_key and s.max_actual_date_key
        
        GROUP BY 1,2,3
        
        having {min_threshold_statement}
        
        
        DISTRIBUTE ON (ord_event_key);
        
        
        
--Getting the list with the all distinct Reward ID's:

    DROP table if exists VMR_{brand_nm}_reward_ids_csmr;
    CREATE temp table VMR_{brand_nm}_reward_ids_csmr as
    
    SELECT distinct ord_designated_cnsmr_id_key
    
    FROM VMR_{brand_nm}_reward_ids
    
    DISTRIBUTE RANDOM;
    
    
    --GRANT ALL ON TABLE VMR_{brand_nm}_reward_ids_csmr TO public

        
''')

VMR_2026_WG_March_Beauty_Event_S25G10_reward_trans


In [28]:
total_reward_ids = pd.read_sql(f''' SELECT COUNT(ord_designated_cnsmr_id_key) FROM VMR_{brand_nm}_reward_ids_csmr''',conn)
print("The total number of trackable reward ID's is: " + str(total_reward_ids['count'].values[0]))
total_reward_ids = total_reward_ids['count'].values[0]

The total number of trackable reward ID's is: 2099785


In [29]:
consistent_reward_ids = pd.read_sql(f''' SELECT COUNT(ord_designated_cnsmr_id_key) 
FROM VMR_{brand_nm}_reward_ids_csmr o
INNER JOIN shopper_consistent_{analyst} f ON (o.ord_designated_cnsmr_id_key = f.cnsmr_id_key)
''',conn)
print("The total number of consistent reward ID's is: " + str(consistent_reward_ids['count'].values[0]))

The total number of consistent reward ID's is: 2099622


In [30]:
total_reward_trans = pd.read_sql(f''' SELECT COUNT(ord_event_key) FROM VMR_{brand_nm}_reward_events''',conn)
print("The total number of reward transactions is: " + str(total_reward_trans['count'].values[0]))

total_reward_trans_var = total_reward_trans

total_reward_trans = str(total_reward_trans['count'].values[0])

The total number of reward transactions is: 2967989


## **4) GETTING DOLLARS AND UNITS FOR YAGO, PRE-PERIOD AND VMR PERIOD FOR ALL TRANSACTIONS OCCURED (FOR ANY SHOPPER):**

4a) Comparison between VMR period and Yago trips (for any shopper and all transactions):

In [31]:
curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_YAGO_trend;
    CREATE temp table VMR_{brand_nm}_YAGO_trend as
    
        SELECT  
               CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period'
                    ELSE 'NA' END as analysis_periods,
               count(distinct a.ord_event_key) as count_distinct_trips,
               sum(a.purch_amt) as dollar_sales,
               dollar_sales/count_distinct_trips as dollars_per_trip,
               sum(a.purch_qty) as units_sales
              
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              and brand_nbr IN ({brand_nbr_str})
              and a.purch_amt > 0
              and a.purch_qty > 0
        
        GROUP BY 1
    
        UNION
        
        
        SELECT  
               CASE WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN 'YAGO Period'
                    ELSE 'NA' END as analysis_periods,
               count(distinct a.ord_event_key) as count_distinct_trips,
               sum(a.purch_amt) as dollar_sales,
               dollar_sales/count_distinct_trips as dollars_per_trip,
               sum(a.purch_qty) as units_sales
              
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            
        WHERE analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)'  
              and brand_nbr IN ({brand_nbr_str})
              and a.purch_amt > 0
              and a.purch_qty > 0
        
        GROUP BY 1
        
        ORDER BY 1 DESC
        
    DISTRIBUTE REPLICATE;

''')

In [32]:
vmr_trend_yago_summary = pd.read_sql(f'''
    SELECT analysis_periods,
           sum(count_distinct_trips) as count_distinct_trips,
           sum(dollar_sales) as dollar_sales,
           sum(dollar_sales)/sum(count_distinct_trips) as dollars_per_trip,
           sum(units_sales) as units_sales,
           round(sum(units_sales)::float/sum(count_distinct_trips)::float,1) as units_per_trip
    FROM VMR_{brand_nm}_YAGO_trend
    WHERE analysis_periods != 'NA'
    GROUP BY 1
    ORDER BY CASE WHEN analysis_periods = 'YAGO Period' THEN 1
                  WHEN analysis_periods = 'VMR Period' THEN 2
                  END
    ''',conn)


#Calculating %change to insert them in the summary table:

if vmr_trend_yago_summary.loc[0, 'analysis_periods'] != 'YAGO Period':
    
    vmr_trend_yago_summary = vmr_trend_yago_summary.rename(columns = {

    'analysis_periods':'Analysis Periods', 'count_distinct_trips':'Count Distinct Trips', 'dollar_sales':'Dollar Sales', 'dollars_per_trip':'Dollars per Trip', '%_change_dollars_per_trip':'% Change Dollars per Trip',
    'units_sales':'Units Sold', 'units_per_trip':'Units per Trip', '%_change_units_per_trip':'% Change Units per Trip'

    })
    
    dollars_moved = round(vmr_trend_yago_summary.loc[0,'Dollar Sales'])
    dollars_moved = "{:,}".format(dollars_moved)
    
    units_moved = round(vmr_trend_yago_summary.loc[0,'Units Sold'])
    units_moved = "{:,}".format(units_moved)

    dollars_per_trip_chg_yago = 0
    
    units_per_trip_chg_yago = 0
    
    dollars_chg_yago = 0
    units_chg_yago = 0
    
    print("NOTE: There is no YAGO data for this UPC list.")
    print("")
    print(vmr_trend_yago_summary)

    
else:
    
    change_dollars_per_trip = (vmr_trend_yago_summary.loc[1,'dollars_per_trip'] - vmr_trend_yago_summary.loc[0,'dollars_per_trip'])/vmr_trend_yago_summary.loc[0,'dollars_per_trip']
    vmr_trend_yago_summary['%_change_dollars_per_trip'] = ['-', change_dollars_per_trip]
    
    change_units_per_trip = (vmr_trend_yago_summary.loc[1,'units_per_trip'] - vmr_trend_yago_summary.loc[0,'units_per_trip'])/vmr_trend_yago_summary.loc[0,'units_per_trip']
    vmr_trend_yago_summary['%_change_units_per_trip'] = ['-', change_units_per_trip]

    vmr_trend_yago_summary = vmr_trend_yago_summary.rename(columns = {

    'analysis_periods':'Analysis Periods', 'count_distinct_trips':'Count Distinct Trips', 'dollar_sales':'Dollar Sales', 'dollars_per_trip':'Dollars per Trip', '%_change_dollars_per_trip':'% Change Dollars per Trip',
    'units_sales':'Units Sold', 'units_per_trip':'Units per Trip', '%_change_units_per_trip':'% Change Units per Trip'

    })

    #Extracting variables for slides

    dollars_moved = round(vmr_trend_yago_summary.loc[1,'Dollar Sales'])
    dollars_moved = "{:,}".format(dollars_moved)
    
    units_moved = round(vmr_trend_yago_summary.loc[0,'Units Sold'])
    units_moved = "{:,}".format(units_moved)
    
    dollars_chg_yago = round(((vmr_trend_yago_summary.loc[1,'Dollar Sales'] - vmr_trend_yago_summary.loc[0,'Dollar Sales'])/vmr_trend_yago_summary.loc[0,'Dollar Sales'])*100)
    units_chg_yago = round(((vmr_trend_yago_summary.loc[1,'Units Sold'] - vmr_trend_yago_summary.loc[0,'Units Sold'])/vmr_trend_yago_summary.loc[0,'Units Sold'])*100)

    dollars_per_trip_chg_yago = round(vmr_trend_yago_summary.loc[1,'% Change Dollars per Trip']*100)
    
    units_per_trip_chg_yago = round(vmr_trend_yago_summary.loc[1,'% Change Units per Trip']*100)

    print(vmr_trend_yago_summary)

  Analysis Periods  Count Distinct Trips  Dollar Sales  Dollars per Trip  \
0      YAGO Period              13574255  2.390810e+08         17.612829   
1       VMR Period              13533312  2.507479e+08         18.528196   

   Units Sold  Units per Trip % Change Dollars per Trip  \
0    26092326             1.9                         -   
1    26899195             2.0                  0.051972   

  % Change Units per Trip  
0                       -  
1                0.052632  


In [33]:
print(dollars_per_trip_chg_yago)
print(dollars_chg_yago)
print(units_per_trip_chg_yago)
print(units_chg_yago)

5
5
5
3


### Prior 26 weeks

In [34]:
curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_26wk_trend;
    CREATE temp table VMR_{brand_nm}_26wk_trend as
    
        SELECT  
               CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period'
                    ELSE 'NA' END as analysis_periods,
               count(distinct a.ord_event_key) as count_distinct_trips,
               sum(a.purch_amt) as dollar_sales,
               dollar_sales/count_distinct_trips as dollars_per_trip,
               sum(a.purch_qty) as units_sales
              
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              and brand_nbr IN ({brand_nbr_str})
              and a.purch_amt > 0
              and a.purch_qty > 0
        
        GROUP BY 1
    
        UNION
        
        
        SELECT  
               'Prior 26 wks' as analysis_periods,
               count(distinct a.ord_event_key) as count_distinct_trips,
               sum(a.purch_amt) as dollar_sales,
               dollar_sales/count_distinct_trips as dollars_per_trip,
               sum(a.purch_qty) as units_sales
              
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            
        WHERE b.cal_dt between '{vmr_pre26wk_start}' and '{vmr_pre26wk_end}'
              and brand_nbr IN ({brand_nbr_str})
              and a.purch_amt > 0
              and a.purch_qty > 0
        
        GROUP BY 1
        
        ORDER BY 1 DESC
        
    DISTRIBUTE REPLICATE;

''')

In [35]:
vmr_trend_26wk_summary = pd.read_sql(f'''
    SELECT analysis_periods,
           sum(count_distinct_trips) as count_distinct_trips,
           sum(dollar_sales) as dollar_sales,
           sum(dollar_sales)/sum(count_distinct_trips) as dollars_per_trip,
           sum(units_sales) as units_sales,
           round(sum(units_sales)::float/sum(count_distinct_trips)::float,1) as units_per_trip
    FROM VMR_{brand_nm}_26wk_trend
    WHERE analysis_periods != 'NA'
    GROUP BY 1
    ORDER BY CASE WHEN analysis_periods = 'Prior 26 wks' THEN 1
                  WHEN analysis_periods = 'VMR Period' THEN 2
                  END
    ''',conn)


#Calculating %change to insert them in the summary table:

if vmr_trend_26wk_summary.loc[0, 'analysis_periods'] != 'Prior 26 wks':
    
    
    vmr_trend_26wk_summary = vmr_trend_26wk_summary.rename(columns = {

    'analysis_periods':'Analysis Periods', 'count_distinct_trips':'Count Distinct Trips', 'dollar_sales':'Dollar Sales', 'dollars_per_trip':'Dollars per Trip', '%_change_dollars_per_trip':'% Change Dollars per Trip',
    'units_sales':'Units Sold', 'units_per_trip':'Units per Trip', '%_change_units_per_trip':'% Change Units per Trip'

    })

    dollars_per_trip_chg_26wk = 0
    units_per_trip_chg_26wk = 0
    
    dollars_chg_26wk = 0
    units_chg_26wk = 0
    
    print("NOTE: There is no Prior 26wk data for this UPC list.")
    print("")
    print(vmr_trend_26wk_summary)

    
    
else:

    vmr_trend_26wk_summary = vmr_trend_26wk_summary.rename(columns = {

        'analysis_periods':'Analysis Periods', 'count_distinct_trips':'Count Distinct Trips', 'dollar_sales':'Dollar Sales', 'dollars_per_trip':'Dollars per Trip', '%_change_dollars_per_trip':'% Change Dollars per Trip',
        'units_sales':'Units Sold', 'units_per_trip':'Units per Trip', '%_change_units_per_trip':'% Change Units per Trip'
    })

    dollars_moved_26wk = round(vmr_trend_26wk_summary.loc[0,'Dollar Sales'])
    dollars_moved_26wk = "{:,}".format(dollars_moved_26wk)

    units_moved_26wk = round(vmr_trend_26wk_summary.loc[0,'Units Sold'])
    units_moved_26wk = "{:,}".format(units_moved_26wk)

    dollars_per_trip_chg_26wk = 0

    units_per_trip_chg_26wk = 0

    dollars_chg_26wk = 0
    units_chg_26wk = 0

    print("")
    print(vmr_trend_26wk_summary)



  Analysis Periods  Count Distinct Trips  Dollar Sales  Dollars per Trip  \
0     Prior 26 wks              76707608  1.355303e+09         17.668432   
1       VMR Period              13533312  2.507479e+08         18.528196   

   Units Sold  Units per Trip  
0   148362140             1.9  
1    26899195             2.0  


4b) Comparison between VMR period and Pre-Period trips (for any shopper and all transactions):

In [36]:
curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_vmr_pre_period_trend;
    CREATE temp table VMR_{brand_nm}_vmr_pre_period_trend as
    
        SELECT  
               CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period'
                    ELSE 'NA' END as analysis_periods,
               count(distinct a.ord_event_key) as count_distinct_trips,
               sum(a.purch_amt) as dollar_sales,
               dollar_sales/count_distinct_trips as dollars_per_trip,
               sum(a.purch_qty) as units_sales
              
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              and brand_nbr IN ({brand_nbr_str})
              and a.purch_amt > 0
              and a.purch_qty > 0
        
        GROUP BY 1
        
        UNION
        
        SELECT  
               CASE WHEN analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Pre-Period'
                    ELSE 'NA' END as analysis_periods,
               count(distinct a.ord_event_key) as count_distinct_trips,
               sum(a.purch_amt) as dollar_sales,
               dollar_sales/count_distinct_trips as dollars_per_trip,
               sum(a.purch_qty) as units_sales
              
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            LEFT JOIN VMR_{brand_nm}_reward_events e ON (e.ord_event_key = a.ord_event_key)
            
        WHERE analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)'
              and brand_nbr IN ({brand_nbr_str})
              and a.purch_amt > 0
              and a.purch_qty > 0
        
        GROUP BY 1
        
        ORDER BY 1 DESC
        
    DISTRIBUTE REPLICATE;

''')

In [37]:
vmr_trend_pre_period_summary = pd.read_sql(f'''
    SELECT analysis_periods,
           sum(count_distinct_trips) as count_distinct_trips,
           sum(dollar_sales) as dollar_sales,
           sum(dollar_sales)/sum(count_distinct_trips) as dollars_per_trip,
           sum(units_sales) as units_sales,
           round(sum(units_sales)::float/sum(count_distinct_trips)::float,1) as units_per_trip
    FROM VMR_{brand_nm}_vmr_pre_period_trend
    WHERE analysis_periods != 'NA'
    GROUP BY 1
    ORDER BY CASE WHEN analysis_periods = 'VMR Pre-Period' THEN 1
                  WHEN analysis_periods = 'VMR Period' THEN 2
                  END
    ''',conn)


#Calculating %change to insert them in the summary table:

if vmr_trend_pre_period_summary.loc[0, 'analysis_periods'] != 'VMR Pre-Period':
    
    
    vmr_trend_pre_period_summary = vmr_trend_pre_period_summary.rename(columns = {

    'analysis_periods':'Analysis Periods', 'count_distinct_trips':'Count Distinct Trips', 'dollar_sales':'Dollar Sales', 'dollars_per_trip':'Dollars per Trip', '%_change_dollars_per_trip':'% Change Dollars per Trip',
    'units_sales':'Units Sold', 'units_per_trip':'Units per Trip', '%_change_units_per_trip':'% Change Units per Trip'

    })
    

    dollars_per_trip_chg_pre_period = 0
    
    units_per_trip_chg_pre_period = 0
    
    dollars_chg_pre_period = 0
    units_chg_pre_period = 0
    
    print("NOTE: There is no PrePeriod data for this UPC list.")
    print("")
    print(vmr_trend_pre_period_summary)

    
else:

    change_dollars_per_trip = ((vmr_trend_pre_period_summary.loc[1,'dollars_per_trip'] - vmr_trend_pre_period_summary.loc[0,'dollars_per_trip'])/vmr_trend_pre_period_summary.loc[0,'dollars_per_trip'])
    vmr_trend_pre_period_summary['%_change_dollars_per_trip'] = ['-', change_dollars_per_trip]

    change_units_per_trip = (vmr_trend_pre_period_summary.loc[1,'units_per_trip'] - vmr_trend_pre_period_summary.loc[0,'units_per_trip'])/vmr_trend_pre_period_summary.loc[0,'units_per_trip']
    vmr_trend_pre_period_summary['%_change_units_per_trip'] = ['-', change_units_per_trip]


    vmr_trend_pre_period_summary = vmr_trend_pre_period_summary.rename(columns = {

    'analysis_periods':'Analysis Periods', 'count_distinct_trips':'Count Distinct Trips', 'dollar_sales':'Dollar Sales', 'dollars_per_trip':'Dollars per Trip', '%_change_dollars_per_trip':'% Change Dollars per Trip',
    'units_sales':'Units Sold', 'units_per_trip':'Units per Trip', '%_change_units_per_trip':'% Change Units per Trip'

    })

    dollars_chg_pre_period = round((vmr_trend_pre_period_summary.loc[1,'Dollar Sales'] - vmr_trend_pre_period_summary.loc[0,'Dollar Sales'])/vmr_trend_pre_period_summary.loc[0,'Dollar Sales'])

    dollars_per_trip_chg_pre_period = round(vmr_trend_pre_period_summary.loc[1,'% Change Dollars per Trip']*100)

    units_per_trip_chg_pre_period = round(vmr_trend_pre_period_summary.loc[1,'% Change Units per Trip']*100)


    vmr_trend_pre_period_summary

In [38]:
vmr_trend_pre_period_summary

,Analysis Periods,Count Distinct Trips,Dollar Sales,Dollars per Trip,Units Sold,Units per Trip,% Change Dollars per Trip,% Change Units per Trip
0,VMR Pre-Period,13032116,2.354315e+08,18.065483,25094079,1.9,-,-
1,VMR Period,13533312,2.507479e+08,18.528196,26899195,2.0,0.025613,0.052632


## **5) GETTING TOTAL DOLLARS FROM REWARD TRIPS BY SEGMENT DURING VMR PERIOD (FOR ANY SHOPPER):**

5a) Getting the total dollars by segment for Reward trips during VMR period (for any shopper):

In [39]:
curs.execute(f'''    
    
 --Getting Units and Dollars for all the reward transactions by segment:
        
    DROP table if exists VMR_{brand_nm}_reward_segments;
    CREATE temp table VMR_{brand_nm}_reward_segments as
        
          SELECT {segment_def},
                 sum(o.purch_qty) as units,
                 sum(o.purch_amt) as dollars
          
          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_reward_events r ON (o.ord_event_key = r.ord_event_key)
          
          WHERE b.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
                and o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str})
                
        
        GROUP BY 1
        
                
        DISTRIBUTE RANDOM;
    

''')

In [40]:
total_vmr_details_by_brand = pd.read_sql(f'''

    SELECT *
    FROM VMR_{brand_nm}_reward_segments
    ORDER BY dollars DESC

    ''',conn)


total_vmr_details_by_brand["units"] = total_vmr_details_by_brand["units"].fillna(0)
total_vmr_details_by_brand["dollars"] = total_vmr_details_by_brand["dollars"].fillna(0)

total_vmr_details_by_brand["% Units"] = (total_vmr_details_by_brand["units"])/(total_vmr_details_by_brand["units"].sum())
total_vmr_details_by_brand["% Dollars"] = (total_vmr_details_by_brand["dollars"])/(total_vmr_details_by_brand["dollars"].sum())

total_vmr_details_by_brand[segment_def] = total_vmr_details_by_brand[segment_def].fillna('Null')
total_vmr_details_by_brand["% Units"] = total_vmr_details_by_brand["% Units"].fillna(0)
total_vmr_details_by_brand["% Dollars"] = total_vmr_details_by_brand["% Dollars"].fillna(0)

total_vmr_details_by_brand.loc['Total']= total_vmr_details_by_brand.sum()
total_vmr_details_by_brand.reset_index(drop=True)
total_vmr_details_by_brand.at['Total',segment_def]='Total'

dollars_moved_vmr = total_vmr_details_by_brand.loc['Total', 'dollars']
units_moved_vmr = total_vmr_details_by_brand.loc['Total', 'units']

total_vmr_details_by_brand = total_vmr_details_by_brand.rename(columns= {segment_def: "Segment", 'units':'Units', 'dollars':'Dollars'})
total_vmr_details_by_brand

,Segment,Units,Dollars,% Units,% Dollars
0,Promoted Beauty,11040784,1.228082e+08,1.0,1.0
Total,Total,11040784,1.228082e+08,1.0,1.0


In [41]:
total_vmr_details_by_brand

,Segment,Units,Dollars,% Units,% Dollars
0,Promoted Beauty,11040784,1.228082e+08,1.0,1.0
Total,Total,11040784,1.228082e+08,1.0,1.0


total_vmr_details_by_brand = pd.read_sql(f'''

    SELECT *
    FROM VMR_{brand_nm}_reward_segments
    ORDER BY dollars DESC

    ''',conn)

total_vmr_details_by_brand

In [42]:
curs.execute(f'''

        DROP table if exists VMR_{brand_nm}_bls_levels;
        CREATE temp table VMR_{brand_nm}_bls_levels as

            select a.PROMO_SRC_ID_TXT as mclu_blip, a.promo_src_id as mclu_nbr,
                   a.promo_dist_start_dt as start_dt, a.promo_dist_stop_dt as stop_dt,
                   CASE when a.promo_discnt_val_amt>0 then a.promo_discnt_val_amt else NULL end as cpn_val_amt,
                   CASE WHEN a.cnsmr_curr_beh_purch_rqmt_qty>0 THEN a.cnsmr_curr_beh_purch_rqmt_qty 
                       ELSE a.cnsmr_TRG_CLASS_purch_rqmt_qty END as min_qty,
                   avg(b.promo_red_handling_fee_rt) as handling_fee,
                   avg(c.avg_imprsn_prc_rt) as impression_cost
    from promotion_v a
      inner join promotion_cost_v b on (a.promo_key=b.promo_key)
      inner join (select promo_key, avg_imprsn_prc_rt 
                    from promo_chanl_prfrm_cum_aggr_v 
                    where dist_chanl_cd='IN-STORE' 
                                    and imprsn_typ_cd='IN-STORE PRINT') c on (a.promo_key=c.promo_key)
    where a.PROMO_SRC_ID_TXT in ({BL_CODES})
    --where a.PROMO_SRC_ID_TXT in ('USA-BLIP-BL_0223_8916', 'USA-BLIP-BL_3049_5392'	)
    group by 1,2,3,4,5,6
    order by min_qty
    
    distribute random;

''')
 

bl_data = pd.read_sql(f''' select * from VMR_{brand_nm}_bls_levels''',conn)
print(bl_data)

               mclu_blip   mclu_nbr    start_dt     stop_dt  cpn_val_amt  \
0  USA-BLIP-BL_4303_0667  671817089  2026-03-01  2026-03-28         10.0   

   min_qty  handling_fee  impression_cost  
0        0           0.0            0.065  


In [43]:
bl_data = pd.read_sql(f''' select * from VMR_{brand_nm}_bls_levels''',conn)
print(bl_data)


min_thres = list(bl_data['min_qty'].dropna().unique())
min_thres = sorted(min_thres)
print(min_thres)


if threshold_unit == 'units':
    
    for i in range(len(min_thres)):
    
        if i == 0:
            if min_thres[i] == max(min_thres):
                plchldr1 = f'''CASE WHEN sum(o.purch_qty) >= {str(min_thres[i])} then 'Buy {str(min_thres[i])}+' ''' + "\n"
            else:
                plchldr1 = f'''CASE WHEN sum(o.purch_qty) >= {str(min_thres[i])} and sum(o.purch_qty) < {str(min_thres[i+1])} then 'Buy {str(min_thres[i])}-{str(min_thres[i+1])}' ''' + "\n"
                        
                
        else:
            if min_thres[i] == max(min_thres):
                plchldr1 =  plchldr1 + f'''WHEN sum(o.purch_qty) >= {str(min_thres[i])} then 'Buy {str(min_thres[i])}+' ''' + "\n"
            else:
                plchldr1 = plchldr1 + f'''WHEN sum(o.purch_qty) >= {str(min_thres[i])} and sum(o.purch_qty) < {str(min_thres[i+1])} then 'Buy {str(min_thres[i])}-{str(min_thres[i+1])}' ''' + "\n"
            

if threshold_unit == 'dollars':
    
    for i in range(len(min_thres)):
    
        if i == 0:
            if min_thres[i] == max(min_thres):
                plchldr1 = f'''CASE WHEN sum(o.purch_amt) >= {str(min_thres[i])} then 'Buy ${str(min_thres[i])}+' ''' + "\n"
            else:
                plchldr1 = f'''CASE WHEN sum(o.purch_amt) >= {str(min_thres[i])} and sum(o.purch_amt) < {str(min_thres[i+1])} then 'Buy ${str(min_thres[i])}-${str(min_thres[i+1])}' ''' + "\n"
        else:
            if min_thres[i] == max(min_thres):
                plchldr1 =  plchldr1 + f'''WHEN sum(o.purch_amt) >= {str(min_thres[i])} then 'Buy ${str(min_thres[i])}+' ''' + "\n"
            else:
                plchldr1 = plchldr1 + f'''WHEN sum(o.purch_amt) >= {str(min_thres[i])} and sum(o.purch_amt) < {str(min_thres[i+1])} then 'Buy ${str(min_thres[i])}-${str(min_thres[i+1])}' ''' + "\n"
         

plchldr1 = plchldr1 + f'''ELSE 'All Other Transactions' END AS level'''

print(plchldr1)

               mclu_blip   mclu_nbr    start_dt     stop_dt  cpn_val_amt  \
0  USA-BLIP-BL_4303_0667  671817089  2026-03-01  2026-03-28         10.0   

   min_qty  handling_fee  impression_cost  
0        0           0.0            0.065  
[0]
CASE WHEN sum(o.purch_amt) >= 0 then 'Buy $0+' 
ELSE 'All Other Transactions' END AS level


bl_data = pd.read_sql(f''' select * from VMR_{brand_nm}_bls_levels''',conn)
print(bl_data)


min_thres = [10]
print(min_thres)


if threshold_unit == 'units':
    
    for i in range(len(min_thres)):
    
        if i == 0:
            if min_thres[i] == max(min_thres):
                plchldr1 = f'''CASE WHEN sum(o.purch_qty) >= {str(min_thres[i])} then 'Buy {str(min_thres[i])}+' ''' + "\n"
            else:
                plchldr1 = f'''CASE WHEN sum(o.purch_qty) >= {str(min_thres[i])} and sum(o.purch_qty) < {str(min_thres[i+1])} then 'Buy {str(min_thres[i])}-{str(min_thres[i+1])}' ''' + "\n"
                        
                
        else:
            if min_thres[i] == max(min_thres):
                plchldr1 =  plchldr1 + f'''WHEN sum(o.purch_qty) >= {str(min_thres[i])} then 'Buy {str(min_thres[i])}+' ''' + "\n"
            else:
                plchldr1 = plchldr1 + f'''WHEN sum(o.purch_qty) >= {str(min_thres[i])} and sum(o.purch_qty) < {str(min_thres[i+1])} then 'Buy {str(min_thres[i])}-{str(min_thres[i+1])}' ''' + "\n"
            

if threshold_unit == 'dollars':
    
    for i in range(len(min_thres)):
    
        if i == 0:
            if min_thres[i] == max(min_thres):
                plchldr1 = f'''CASE WHEN sum(o.purch_amt) >= {str(min_thres[i])} then 'Buy ${str(min_thres[i])}+' ''' + "\n"
            else:
                plchldr1 = f'''CASE WHEN sum(o.purch_amt) >= {str(min_thres[i])} and sum(o.purch_amt) < {str(min_thres[i+1])} then 'Buy ${str(min_thres[i])}-${str(min_thres[i+1])}' ''' + "\n"
        else:
            if min_thres[i] == max(min_thres):
                plchldr1 =  plchldr1 + f'''WHEN sum(o.purch_amt) >= {str(min_thres[i])} then 'Buy ${str(min_thres[i])}+' ''' + "\n"
            else:
                plchldr1 = plchldr1 + f'''WHEN sum(o.purch_amt) >= {str(min_thres[i])} and sum(o.purch_amt) < {str(min_thres[i+1])} then 'Buy ${str(min_thres[i])}-${str(min_thres[i+1])}' ''' + "\n"
         

plchldr1 = plchldr1 + f'''ELSE 'All Other Transactions' END AS level'''

print(plchldr1)

bl_data = pd.read_sql(f''' select * from VMR_{brand_nm}_bls_levels''',conn)
print(bl_data)



if threshold_unit == 'units':
    
    plchldr1 = f'''CASE WHEN sum(o.purch_amt) >= 25 AND sum(o.purch_amt) <= 34.99 then 'Buy $25-34.99' ELSE 'All Other Transactions' END AS level '''        

if threshold_unit == 'dollars':
    
    
    plchldr1 = f'''CASE WHEN sum(o.purch_amt) >= 30 AND sum(o.purch_amt) <= 44.99999 then 'Buy $30-44.99' ELSE 'All Other Transactions' END AS level'''
    
   
print(plchldr1)

In [44]:
curs.execute(f'''    
            
    DROP table if exists VMR_{brand_nm}_reward_levels;
    CREATE temp table VMR_{brand_nm}_reward_levels as
    
        
          SELECT o.ord_event_key,'reward period' as period,
          {plchldr1}, 
          sum(o.purch_qty) as units, sum(o.purch_amt) as dollars
          

          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_printing_stores a ON (o.ord_touchpoint_key = a.touchpoint_key)
                 INNER JOIN VMR_{brand_nm}_reward_events r ON (o.ord_event_key = r.ord_event_key)
                 --INNER JOIN VMR_{brand_nm}_reward_ids_csmr z ON (z.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)

          
          WHERE b.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' and
                o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str})
                and cal_dt between '{reward_print_start}' and '{reward_print_end}'

            
          GROUP BY 1,2
          
          
           HAVING level != 'All Other Transactions'
          
          
          UNION
          
          
          
          SELECT o.ord_event_key,'reward period' as period,
          {plchldr1}, 
          sum(o.purch_qty) as units, sum(o.purch_amt) as dollars
          

          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_printing_stores a ON (o.ord_touchpoint_key = a.touchpoint_key)                 
                 --INNER JOIN VMR_{brand_nm}_reward_ids_csmr z ON (z.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)

          WHERE b.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' and
                o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str})
                and cal_dt between '{reward_print_start}' and '{reward_print_end}'

            
          GROUP BY 1,2
          
          HAVING level = 'All Other Transactions'
          
          
          UNION
        
          
          SELECT ord_event_key,'pre-period' as period,
          {plchldr1}, 
          sum(o.purch_qty) as units, sum(o.purch_amt) as dollars
          

          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_printing_stores a ON (o.ord_touchpoint_key = a.touchpoint_key)
                 --INNER JOIN VMR_{brand_nm}_reward_ids_csmr z ON (z.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)
          
          WHERE b.analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' and
                o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str}) and
                cal_dt between '{vmr_pre_period_start}' and '{vmr_pre_period_end}' 

            
          GROUP BY 1,2
          
          
          
          UNION
          
          
          
          SELECT ord_event_key,'YAGO period' as period,
          {plchldr1}, 
          sum(o.purch_qty) as units, sum(o.purch_amt) as dollars
          

          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_printing_stores a ON (o.ord_touchpoint_key = a.touchpoint_key)
                 --INNER JOIN VMR_{brand_nm}_reward_ids_csmr z ON (z.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)

          
          WHERE b.analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' and
                o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str}) and
                cal_dt between '{yago_print_start}' and '{yago_print_end}'

            
          GROUP BY 1,2
          
          
          
          UNION
          
          
          SELECT ord_event_key,'prior 52wk' as period,
          {plchldr1}, 
          sum(o.purch_qty) as units, sum(o.purch_amt) as dollars
          

          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_printing_stores a ON (o.ord_touchpoint_key = a.touchpoint_key)
                 --INNER JOIN VMR_{brand_nm}_reward_ids_csmr z ON (z.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)

          
          WHERE b.prior_period = 'Prior-period ({round(pre_weeks)} weeks)' and
                o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str}) and
                cal_dt between '{prior_period_start}' and '{prior_period_end}'


            
          GROUP BY 1,2
          
          
          
        
            DISTRIBUTE RANDOM;
    
''')


curs.execute(f'''    
            
    DROP table if exists VMR_{brand_nm}_reward_levels_end;
    CREATE temp table VMR_{brand_nm}_reward_levels_end as
    
        
          SELECT level, period,
                 sum(dollars) as dollars
                 , count(distinct ord_event_key) as trips, sum(units) as units
                 
          
          FROM VMR_{brand_nm}_reward_levels
          
          GROUP BY 1,2
          
          DISTRIBUTE RANDOM;
    
''')

In [45]:
pd.read_sql(f'''

 SELECT level, period,
                 sum(dollars) as dollars
                 , count(distinct ord_event_key) as trips, sum(units) as units
                 
          
          FROM VMR_{brand_nm}_reward_levels
          
          GROUP BY 1,2
          

    ''',conn)

,level,period,dollars,trips,units
0,Buy $0+,YAGO period,2.390810e+08,13574255,26092326
1,Buy $0+,prior 52wk,2.797224e+09,159531030,307586002
2,Buy $0+,reward period,1.228082e+08,2967989,11040784
3,Buy $0+,pre-period,2.354315e+08,13032116,25094079


5b) Getting dollars and trips for different level buyers for Reward transactions (any shoppers):

In [46]:
level_ct = pd.read_sql(f'''

    SELECT level, dollars, trips, units
    FROM VMR_{brand_nm}_reward_levels_end
    WHERE period = 'reward period'
    
    UNION
    
    SELECT 'Grand Total' as level, sum(dollars) as dollars, sum(trips) as trips, sum(units) as units
    FROM VMR_{brand_nm}_reward_levels_end
    WHERE period = 'reward period'
    
    
    ORDER BY 3 ASC

    ''',conn)

level_ct

,level,dollars,trips,units
0,Buy $0+,1.228082e+08,2967989,11040784
1,Grand Total,1.228082e+08,2967989,11040784


In [47]:
level_ct_yago = pd.read_sql(f'''

    SELECT level, dollars, trips, units
    FROM VMR_{brand_nm}_reward_levels_end
    WHERE period = 'YAGO period'
    
    UNION
    
    SELECT 'Grand Total' as level, sum(dollars) as dollars, sum(trips) as trips, sum(units) as units
    FROM VMR_{brand_nm}_reward_levels_end
    WHERE period = 'YAGO period'
    
    
    ORDER BY 3 ASC

    ''',conn)

level_ct_yago

,level,dollars,trips,units
0,Buy $0+,239081041.1,13574255,26092326
1,Grand Total,239081041.1,13574255,26092326


In [48]:
level_ct_prior = pd.read_sql(f'''

    SELECT level, dollars, trips, units
    FROM VMR_{brand_nm}_reward_levels_end
    WHERE period = 'prior 52wk'

    UNION
    
    SELECT 'Grand Total' as level, sum(dollars) as dollars, sum(trips) as trips, sum(units) as units
    FROM VMR_{brand_nm}_reward_levels_end
    WHERE period = 'prior 52wk'
    
    
    ORDER BY 3 ASC

    ''',conn)

level_ct_prior

,level,dollars,trips,units
0,Grand Total,2.797224e+09,159531030,307586002
1,Buy $0+,2.797224e+09,159531030,307586002


In [49]:
level_ct_prep = pd.read_sql(f'''

    SELECT level, dollars, trips, units
    FROM VMR_{brand_nm}_reward_levels_end
    WHERE period = 'pre-period'
   
   UNION
    
    SELECT 'Grand Total' as level, sum(dollars) as dollars, sum(trips) as trips, sum(units) as units
    FROM VMR_{brand_nm}_reward_levels_end
    WHERE period = 'pre-period'
    
    
    ORDER BY 3 ASC
    ''',conn)

level_ct_prep

,level,dollars,trips,units
0,Grand Total,2.354315e+08,13032116,25094079
1,Buy $0+,2.354315e+08,13032116,25094079


In [50]:
if int(level_ct.loc[(level_ct['level']!= 'All Other Transactions') & (level_ct['level']!= 'Grand Total')].dollars.sum()) == int(total_vmr_details_by_brand['Dollars'].max()):
    print('The Dollars moved for this campaign DO MATCH with the dollars by reward level')
else:
    print('The Dollars moved for this campaign DO NOT match with the dollars by reward level')

The Dollars moved for this campaign DO MATCH with the dollars by reward level


## **6) GETTING DOLLARS PER TRIP BY BRAND FOR VMR PERIOD, VMR PRE-PERIOD AND YAGO (FOR REWARD ID's):**

6a) Getting dollars per trip and units per trip for each period by segment (for consistent Reward ID's):

In [51]:
analyze_start = yago_print_start.strftime('%Y-%m-%d')
analyze_end = reward_print_end.strftime('%Y-%m-%d') 

curs.execute(f'''   

    DROP table if exists VMR_{brand_nm}_participants_vmr_period_segment;
    CREATE temp table VMR_{brand_nm}_participants_vmr_period_segment as
    
    
        SELECT CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period'
                    WHEN analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Pre-Period'
                    WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN 'YAGO Period'
                    END as analysis_periods,
               {segment_def},
               sum(a.purch_amt) as dollars,
               count(distinct a.ord_event_key) as trips,
               sum(a.purch_qty) as units,
               round((dollars/trips),2)::float as dollars_per_trip,
               round((units::float/trips::float),1) as units_per_trip
               
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr d ON (a.ord_designated_cnsmr_id_key = d.ord_designated_cnsmr_id_key)

            
        WHERE brand_nbr in ({brand_nbr_str})
              and purch_qty >0
              and purch_amt >0
              and b.cal_dt >= '{analyze_start}'
            and b.cal_dt <= '{analyze_end}'
            and analysis_period IN ('VMR Print Period ({round(weeks_per_period_final)} weeks)',
                                    'VMR Pre-period ({round(weeks_per_period_final)} weeks)',
                                    'Period Year Ago ({round(weeks_per_period_final)} weeks)')
            
        
        GROUP BY 1,2
                
        
        UNION
        
        
        SELECT 'Prior 26wk' as analysis_periods,
               {segment_def},
               sum(a.purch_amt) as dollars,
               count(distinct a.ord_event_key) as trips,
               sum(a.purch_qty) as units,
               round((dollars/trips),2)::float as dollars_per_trip,
               round((units::float/trips::float),1) as units_per_trip
               
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr d ON (a.ord_designated_cnsmr_id_key = d.ord_designated_cnsmr_id_key)

            
        WHERE brand_nbr in ({brand_nbr_str})
              and purch_qty >0
              and purch_amt >0
              and b.cal_dt >= '{vmr_pre26wk_start}'
            and b.cal_dt <= '{vmr_pre26wk_end}'
            
        
        GROUP BY 1,2
                
        
    DISTRIBUTE REPLICATE;

''')

In [52]:
vmr_period_segment_sum = pd.read_sql(f'''

    SELECT *
    FROM VMR_{brand_nm}_participants_vmr_period_segment
    WHERE analysis_periods is not null
    ORDER BY 1,2,3 DESC
    
    ''',conn)

vmr_period_segment_sum

,analysis_periods,brand_desc,dollars,trips,units,dollars_per_trip,units_per_trip
0,Prior 26wk,Promoted Beauty,1.730937e+08,7576251,17813730,22.85,2.4
1,VMR Period,Promoted Beauty,1.104637e+08,3265956,10388473,33.82,3.2
2,VMR Pre-Period,Promoted Beauty,3.040137e+07,1327033,3116936,22.91,2.3
3,YAGO Period,Promoted Beauty,2.732577e+07,1192477,2802491,22.92,2.4


6b) Getting the distinct number of trips for each period to calculte %change:

In [53]:
unique_trips_by_period = pd.read_sql(f'''

WITH prior_table as (

SELECT  
   CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period'
        WHEN analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Pre-Period'
        WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN 'YAGO Period'
        ELSE 'NA' END as analysis_periods,
   count(distinct a.ord_event_key) as distinct_trips

FROM ord_trd_itm_cnsmr_fact_ne_v a 

    INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
    INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
    INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
    INNER JOIN VMR_{brand_nm}_reward_ids_csmr d ON (a.ord_designated_cnsmr_id_key = d.ord_designated_cnsmr_id_key)


WHERE analysis_period IN ('VMR Print Period ({round(weeks_per_period_final)} weeks)', 'VMR Pre-period ({round(weeks_per_period_final)} weeks)', 
      'Period Year Ago ({round(weeks_per_period_final)} weeks)')
      and brand_nbr IN ({brand_nbr_str})
      and a.purch_amt > 0
      and a.purch_qty > 0
      
GROUP BY 1


UNION


SELECT  
   'Prior 26wk' as analysis_periods,
   count(distinct a.ord_event_key) as distinct_trips

FROM ord_trd_itm_cnsmr_fact_ne_v a 

    INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
    INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
    INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
    INNER JOIN VMR_{brand_nm}_reward_ids_csmr d ON (a.ord_designated_cnsmr_id_key = d.ord_designated_cnsmr_id_key)


WHERE b.cal_dt BETWEEN '{vmr_pre26wk_start}' and '{vmr_pre26wk_end}' 
      and brand_nbr IN ({brand_nbr_str})
      and a.purch_amt > 0
      and a.purch_qty > 0
      
GROUP BY 1

)

SELECT *
FROM prior_table
ORDER BY CASE WHEN analysis_periods = 'YAGO Period' THEN 1
              WHEN analysis_periods = 'VMR Pre-Period' THEN 2
              WHEN analysis_periods = 'VMR Period' THEN 3
              WHEN analysis_periods = 'Prior 26wk' THEN 4
              END

    ''', conn)

unique_trips_by_period

,analysis_periods,distinct_trips
0,YAGO Period,1192476
1,VMR Pre-Period,1327033
2,VMR Period,3265956
3,Prior 26wk,7576251


6c) Getting the comparison in dollars per trip and units per trip for VMR Period and YAGO period (for consistent Reward ID's):

In [54]:
#Getting VMR and YAGO data from segment table

if dominance != 'Kroger':

    if unique_trips_by_period.loc[0, 'analysis_periods'] != 'YAGO Period':

        print("NOTE: There is no YAGO data for this UPC list.")

    else:

        vmr_data = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period'), [segment_def, 'dollars_per_trip', 'units_per_trip']]
        vmr_data.rename(columns = {'dollars_per_trip':'dollars_per_trip_vmr_period', 'units_per_trip':'units_per_trip_vmr_period'}, inplace = True)

        vmr_yago = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='YAGO Period'), [segment_def, 'dollars_per_trip', 'units_per_trip']]
        vmr_yago.rename(columns = {'dollars_per_trip':'dollars_per_trip_yago_period', 'units_per_trip':'units_per_trip_yago_period'}, inplace = True)


        #Getting comparison table VMR vs YAGO and calculating %_change for each segment

        comparison_yago = pd.merge(vmr_data, vmr_yago, how="left", on=[segment_def])
        comparison_yago = comparison_yago.reindex(sorted(comparison_yago.columns), axis=1)

        comparison_yago.insert(3, '% Change Dollars',(comparison_yago["dollars_per_trip_vmr_period"]-comparison_yago["dollars_per_trip_yago_period"])/(comparison_yago["dollars_per_trip_yago_period"]))
        comparison_yago.insert(6, '% Change Units',(comparison_yago["units_per_trip_vmr_period"]-comparison_yago["units_per_trip_yago_period"])/(comparison_yago["units_per_trip_yago_period"]))


        #Calculating total %_change dollars and units for the whole brand

        total_trips_vmr = unique_trips_by_period.loc[unique_trips_by_period['analysis_periods']=='VMR Period'].distinct_trips.values[0]
        total_trips_yago = unique_trips_by_period.loc[unique_trips_by_period['analysis_periods']=='YAGO Period'].distinct_trips.values[0]

        total_dollars_vmr = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period')].dollars.sum()
        total_dollars_yago = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='YAGO Period')].dollars.sum()
        change_total_dollar_trip_vmr = ((total_dollars_vmr/total_trips_vmr)-(total_dollars_yago/total_trips_yago))/(total_dollars_yago/total_trips_yago)

        total_units_vmr = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period')].units.sum()
        total_units_yago = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='YAGO Period')].units.sum()
        change_total_units_trip_vmr = ((total_units_vmr/total_trips_vmr)-(total_units_yago/total_trips_yago))/(total_units_yago/total_trips_yago)

        comparison_yago = comparison_yago.rename(columns={segment_def: "Segment"})

        columns_titles = ["Segment","dollars_per_trip_yago_period","dollars_per_trip_vmr_period", "% Change Dollars", "units_per_trip_yago_period", "units_per_trip_vmr_period", "% Change Units"]
        comparison_yago= comparison_yago.reindex(columns=columns_titles)

        comparison_yago.loc['Total']= ['', (total_dollars_yago/total_trips_yago), (total_dollars_vmr/total_trips_vmr), change_total_dollar_trip_vmr, (total_units_yago/total_trips_yago), (total_units_vmr/total_trips_vmr), change_total_units_trip_vmr  ]
        comparison_yago.at['Total','Segment']='Total'

        comparison_yago = comparison_yago.reset_index(drop = True)

        comparison_yago = comparison_yago.rename(columns = {

        'dollars_per_trip_yago_period':'Dollars per Trip - YAGO Period', 
        'dollars_per_trip_vmr_period':'Campaign Dollars per Trip',
        'units_per_trip_yago_period':'Units per Trip - YAGO Period', 
        'units_per_trip_vmr_period':'Campaign Units per Trip'

        })

        print(comparison_yago)


           Segment  Dollars per Trip - YAGO Period  Campaign Dollars per Trip  \
0  Promoted Beauty                       22.920000                  33.820000   
1            Total                       22.915153                  33.822788   

   % Change Dollars  Units per Trip - YAGO Period  Campaign Units per Trip  \
0          0.475567                      2.400000                 3.200000   
1          0.476001                      2.350145                 3.180837   

   % Change Units  
0        0.333333  
1        0.353464  


AGREGAR AL AUTOMATE
unique_trips_by_period.loc[unique_trips_by_period['analysis_periods']=='VMR Period'].distinct_trips.values[0]

In [55]:
#Getting VMR and YAGO data from segment table

if dominance == 'Kroger':


    vmr_data = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period'), [segment_def, 'dollars_per_trip', 'units_per_trip']]
    vmr_data.rename(columns = {'dollars_per_trip':'dollars_per_trip_vmr_period', 'units_per_trip':'units_per_trip_vmr_period'}, inplace = True)

    vmr_yago = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='Prior 26wk'), [segment_def, 'dollars_per_trip', 'units_per_trip']]
    vmr_yago.rename(columns = {'dollars_per_trip':'dollars_per_trip_yago_period', 'units_per_trip':'units_per_trip_yago_period'}, inplace = True)


    #Getting comparison table VMR vs YAGO and calculating %_change for each segment

    comparison_yago = pd.merge(vmr_data, vmr_yago, how="left", on=[segment_def])
    comparison_yago = comparison_yago.reindex(sorted(comparison_yago.columns), axis=1)

    comparison_yago.insert(3, '% Change Dollars',(comparison_yago["dollars_per_trip_vmr_period"]-comparison_yago["dollars_per_trip_yago_period"])/(comparison_yago["dollars_per_trip_yago_period"]))
    comparison_yago.insert(6, '% Change Units',(comparison_yago["units_per_trip_vmr_period"]-comparison_yago["units_per_trip_yago_period"])/(comparison_yago["units_per_trip_yago_period"]))


    #Calculating total %_change dollars and units for the whole brand

    total_trips_vmr = unique_trips_by_period.loc[2, 'distinct_trips']
    total_trips_yago = unique_trips_by_period.loc[3, 'distinct_trips']

    total_dollars_vmr = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period')].dollars.sum()
    total_dollars_yago = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='Prior 26wk')].dollars.sum()
    change_total_dollar_trip_vmr = ((total_dollars_vmr/total_trips_vmr)-(total_dollars_yago/total_trips_yago))/(total_dollars_yago/total_trips_yago)

    total_units_vmr = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period')].units.sum()
    total_units_yago = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='Prior 26wk')].units.sum()
    change_total_units_trip_vmr = ((total_units_vmr/total_trips_vmr)-(total_units_yago/total_trips_yago))/(total_units_yago/total_trips_yago)

    comparison_yago = comparison_yago.rename(columns={segment_def: "Segment"})

    columns_titles = ["Segment","dollars_per_trip_yago_period","dollars_per_trip_vmr_period", "% Change Dollars", "units_per_trip_yago_period", "units_per_trip_vmr_period", "% Change Units"]
    comparison_yago= comparison_yago.reindex(columns=columns_titles)

    comparison_yago.loc['Total']= ['', (total_dollars_yago/total_trips_yago), (total_dollars_vmr/total_trips_vmr), change_total_dollar_trip_vmr, (total_units_yago/total_trips_yago), (total_units_vmr/total_trips_vmr), change_total_units_trip_vmr  ]
    comparison_yago.at['Total','Segment']='Total'

    comparison_yago = comparison_yago.reset_index(drop = True)

    comparison_yago = comparison_yago.rename(columns = {

    'dollars_per_trip_yago_period':'Dollars per Trip - Prior 26wk', 
    'dollars_per_trip_vmr_period':'Campaign Dollars per Trip',
    'units_per_trip_yago_period':'Units per Trip - Prior 26wk', 
    'units_per_trip_vmr_period':'Campaign Units per Trip'

    })

    print(comparison_yago)


In [56]:
if dominance != 'Kroger':
    
    if unique_trips_by_period.loc[0, 'analysis_periods'] == 'YAGO Period':

        table_slide = comparison_yago[['Segment','Campaign Dollars per Trip', 'Dollars per Trip - YAGO Period', '% Change Dollars', 'Campaign Units per Trip', 'Units per Trip - YAGO Period','% Change Units']]
        table_slide['Campaign Dollars per Trip'] = round(table_slide['Campaign Dollars per Trip'],2)
        table_slide['Campaign Units per Trip'] = round(table_slide['Campaign Units per Trip'],1)
        x_high = table_slide
        table_slide = table_slide.T.reset_index().T.reset_index(drop=True)

        table_slide.loc[:, table_slide.columns[1:]] = table_slide.loc[:, table_slide.columns[1:]].fillna(0)
        table_slide 
    else:
        table_slide = pd.DataFrame()

In [57]:
all_ids_new_seg = pd.DataFrame()
len(all_ids_new_seg)

0

In [58]:
if dominance != 'Kroger':

    if unique_trips_by_period.loc[0, 'analysis_periods'] == 'YAGO Period':

        table_slide_s = comparison_yago[['Segment','Campaign Dollars per Trip', '% Change Dollars', 'Campaign Units per Trip', '% Change Units']]
        table_slide_s['Campaign Dollars per Trip'] = round(table_slide_s['Campaign Dollars per Trip'],2)
        table_slide_s['Campaign Units per Trip'] = round(table_slide_s['Campaign Units per Trip'],1)
        table_slide_s = table_slide_s.T.reset_index().T.reset_index(drop=True)

        table_slide_s.loc[:, table_slide_s.columns[1:]] = table_slide_s.loc[:, table_slide_s.columns[1:]].fillna(0)
        table_slide_s 
    else:
        table_slide_s = pd.DataFrame()

<ipython-input-58-ed659bd101c9>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_slide_s['Campaign Dollars per Trip'] = round(table_slide_s['Campaign Dollars per Trip'],2)
<ipython-input-58-ed659bd101c9>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  table_slide_s['Campaign Units per Trip'] = round(table_slide_s['Campaign Units per Trip'],1)


In [59]:
if dominance == 'Kroger':
    table_slide = comparison_yago[['Segment','Campaign Dollars per Trip', 'Dollars per Trip - Prior 26wk', '% Change Dollars', 'Campaign Units per Trip', 'Units per Trip - Prior 26wk','% Change Units']]
    table_slide['Campaign Dollars per Trip'] = round(table_slide['Campaign Dollars per Trip'],2)
    table_slide['Campaign Units per Trip'] = round(table_slide['Campaign Units per Trip'],1)
    x_high = table_slide
    table_slide = table_slide.T.reset_index().T.reset_index(drop=True)

    table_slide.loc[:, table_slide.columns[1:]] = table_slide.loc[:, table_slide.columns[1:]].fillna(0)
    table_slide 

    table_slide_s = comparison_yago[['Segment','Campaign Dollars per Trip', '% Change Dollars', 'Campaign Units per Trip', '% Change Units']]
    table_slide_s['Campaign Dollars per Trip'] = round(table_slide_s['Campaign Dollars per Trip'],2)
    table_slide_s['Campaign Units per Trip'] = round(table_slide_s['Campaign Units per Trip'],1)
    table_slide_s = table_slide_s.T.reset_index().T.reset_index(drop=True)

    table_slide_s.loc[:, table_slide_s.columns[1:]] = table_slide_s.loc[:, table_slide_s.columns[1:]].fillna(0)
    print(table_slide_s)
        

6d) Getting the comparison in dollars per trip and units per trip for VMR Period and VMR Pre-Period period (for consistent Reward ID's):

In [60]:
#Getting VMR and YAGO data from segment table

if len(unique_trips_by_period.loc[(unique_trips_by_period['analysis_periods'] == 'VMR Pre-Period')]) != 0:

    vmr_data = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period'), [segment_def, 'dollars_per_trip', 'units_per_trip']]
    vmr_data.rename(columns = {'dollars_per_trip':'dollars_per_trip_vmr_period', 'units_per_trip':'units_per_trip_vmr_period'}, inplace = True)

    vmr_preperiod = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Pre-Period'), [segment_def, 'dollars_per_trip', 'units_per_trip']]
    vmr_preperiod.rename(columns = {'dollars_per_trip':'dollars_per_trip_pre_period', 'units_per_trip':'units_per_trip_pre_period'}, inplace = True)


    #Getting comparison table VMR vs YAGO and calculating %_change for each segment

    comparison_preperiod = pd.merge(vmr_data, vmr_preperiod, how="left", on=[segment_def])
    comparison_preperiod = comparison_preperiod.reindex(sorted(comparison_preperiod.columns), axis=1)

    comparison_preperiod.insert(3, '% Change Dollars',(comparison_preperiod["dollars_per_trip_vmr_period"]-comparison_preperiod["dollars_per_trip_pre_period"])/(comparison_preperiod["dollars_per_trip_pre_period"]))
    comparison_preperiod.insert(6, '% Change Units',(comparison_preperiod["units_per_trip_vmr_period"]-comparison_preperiod["units_per_trip_pre_period"])/(comparison_preperiod["units_per_trip_pre_period"]))


    #Calculating total %_change dollars and units for the whole brand

    total_trips_vmr = unique_trips_by_period.loc[(unique_trips_by_period['analysis_periods'] == 'VMR Period'), 'distinct_trips'].values[0]
    total_trips_pperiod = unique_trips_by_period.loc[(unique_trips_by_period['analysis_periods'] == 'VMR Pre-Period'), 'distinct_trips'].values[0]

    total_dollars_vmr = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period')].dollars.sum()
    total_dollars_pperiod = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Pre-Period')].dollars.sum()
    change_total_dollar_trip_vmr = ((total_dollars_vmr/total_trips_vmr)-(total_dollars_pperiod/total_trips_pperiod))/(total_dollars_pperiod/total_trips_pperiod)

    total_units_vmr = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Period')].units.sum()
    total_units_pperiod = vmr_period_segment_sum.loc[(vmr_period_segment_sum['analysis_periods']=='VMR Pre-Period')].units.sum()
    change_total_units_trip_vmr = ((total_units_vmr/total_trips_vmr)-(total_units_pperiod/total_trips_pperiod))/(total_units_pperiod/total_trips_pperiod)

    comparison_preperiod = comparison_preperiod.rename(columns={segment_def: "Segment"})

    columns_titles = ["Segment","dollars_per_trip_pre_period","dollars_per_trip_vmr_period", "% Change Dollars", "units_per_trip_pre_period", "units_per_trip_vmr_period", "% Change Units"]
    comparison_preperiod= comparison_preperiod.reindex(columns=columns_titles)

    comparison_preperiod.loc['Total']= ['', (total_dollars_pperiod/total_trips_pperiod), (total_dollars_vmr/total_trips_vmr), change_total_dollar_trip_vmr, (total_units_pperiod/total_trips_pperiod), (total_units_vmr/total_trips_vmr), change_total_units_trip_vmr  ]
    comparison_preperiod.at['Total',"Segment"]='Total'

    comparison_preperiod = comparison_preperiod.reset_index(drop = True)

    comparison_preperiod = comparison_preperiod.rename(columns = {

    'dollars_per_trip_pre_period':'Dollars per Trip - Pre Period', 
    'dollars_per_trip_vmr_period':'Campaign Dollars per Trip',
    'units_per_trip_pre_period':'Units per Trip - Pre Period', 
    'units_per_trip_vmr_period':'Campaign Units per Trip'

    })


    comparison_preperiod

agregar en la pasada len(unique_trips_by_period.loc[(unique_trips_by_period['analysis_periods'] == 'VMR Pre-Period')])


In [61]:
if len(unique_trips_by_period.loc[(unique_trips_by_period['analysis_periods'] == 'VMR Pre-Period')]) != 0:

    table_slide_2 = comparison_preperiod[['Segment','Campaign Dollars per Trip', 'Dollars per Trip - Pre Period','% Change Dollars', 'Campaign Units per Trip', 'Units per Trip - Pre Period','% Change Units']]
    table_slide_2['Campaign Dollars per Trip'] = round(table_slide_2['Campaign Dollars per Trip'],2)
    table_slide_2['Campaign Units per Trip'] = round(table_slide_2['Campaign Units per Trip'],1)
    table_slide_2 = table_slide_2.T.reset_index().T.reset_index(drop=True)
    table_slide_2.loc[:, table_slide_2.columns[1:]] = table_slide_2.loc[:, table_slide_2.columns[1:]].fillna(0)
    table_slide_2

6e) Getting the comparison in dollars per trip and units per trip for VMR Period and VMR 52 WK Pre-Period period (for consistent Reward ID's):

In [62]:
analyze_start_2 = prior_period_start.strftime('%Y-%m-%d')

curs.execute(f'''   

    DROP table if exists VMR_{brand_nm}_participants_vmr_period_segment_pre52;
    CREATE temp table VMR_{brand_nm}_participants_vmr_period_segment_pre52 as
    
    
        SELECT CASE WHEN brand_nbr in ({brand_nbr_str}) and analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period'
                    WHEN brand_nbr in ({brand_nbr_str}) and prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN '52wk Pre-Period'
                    END as analysis_periods,
               {segment_def},
               sum(a.purch_amt) as dollars,
               count(distinct a.ord_event_key) as trips,
               sum(a.purch_qty) as units,
               round((dollars/trips),2)::float as dollars_per_trip,
               round((units::float/trips::float),1) as units_per_trip
               
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr d ON (a.ord_designated_cnsmr_id_key = d.ord_designated_cnsmr_id_key)

            
        WHERE 
              purch_qty >0
              and purch_amt >0
              and b.cal_dt >= '{analyze_start_2}'
            and b.cal_dt <= '{analyze_end}'
            and prior_period IN ('Prior-period ({round(pre_weeks)} weeks)') OR 
                analysis_period IN ('VMR Print Period ({round(weeks_per_period_final)} weeks)')            
        
        GROUP BY 1,2
        
        ORDER BY 2 DESC
        
    DISTRIBUTE REPLICATE;

''')

In [63]:
vmr_period_segment_sum_pre52 = pd.read_sql(f'''

    SELECT *
    FROM VMR_{brand_nm}_participants_vmr_period_segment_pre52
    WHERE analysis_periods is not null 
    ORDER BY 1,2,3 DESC
    
    ''',conn)

vmr_period_segment_sum_pre52

,analysis_periods,brand_desc,dollars,trips,units,dollars_per_trip,units_per_trip
0,52wk Pre-Period,Promoted Beauty,3.377358e+08,14880268,34962146,22.70,2.3
1,VMR Period,Promoted Beauty,1.104634e+08,3273896,10412960,33.74,3.2


In [64]:
unique_trips_by_period_52wk = pd.read_sql(f'''

SELECT  
   CASE WHEN prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN 'VMR 52wk PrePeriod'
        ELSE 'NA' END as analysis_periods,
   count(distinct a.ord_event_key) as distinct_trips

FROM ord_trd_itm_cnsmr_fact_ne_v a 

    INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
    INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
    INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
    INNER JOIN VMR_{brand_nm}_reward_ids_csmr d ON (a.ord_designated_cnsmr_id_key = d.ord_designated_cnsmr_id_key)


WHERE prior_period IN ('Prior-period ({round(pre_weeks)} weeks)')
      and brand_nbr IN ({brand_nbr_str})
      and a.purch_amt > 0
      and a.purch_qty > 0
      
GROUP BY 1


    ''', conn)

unique_trips_by_period_52wk

,analysis_periods,distinct_trips
0,VMR 52wk PrePeriod,14880257


In [65]:
#Getting VMR and Pre 52wk data from segment table

if unique_trips_by_period_52wk.loc[0, 'analysis_periods'] != 'VMR 52wk PrePeriod':
    
    print("NOTE: There is no Pre 52 wk data for this UPC list.")
    
else:


    vmr_data = vmr_period_segment_sum_pre52.loc[(vmr_period_segment_sum_pre52['analysis_periods']=='VMR Period'), [segment_def, 'dollars_per_trip', 'units_per_trip']]
    vmr_data.rename(columns = {'dollars_per_trip':'dollars_per_trip_vmr_period', 'units_per_trip':'units_per_trip_vmr_period'}, inplace = True)

    vmr_preperiod_52wk = vmr_period_segment_sum_pre52.loc[(vmr_period_segment_sum_pre52['analysis_periods']=='52wk Pre-Period'), [segment_def, 'dollars_per_trip', 'units_per_trip']]
    vmr_preperiod_52wk.rename(columns = {'dollars_per_trip':'dollars_per_trip_pre_period', 'units_per_trip':'units_per_trip_pre_period'}, inplace = True)


    #Getting comparison table VMR vs YAGO and calculating %_change for each segment

    comparison_preperiod_52wk = pd.merge(vmr_data, vmr_preperiod_52wk, how="left", on=[segment_def])
    comparison_preperiod_52wk = comparison_preperiod_52wk.reindex(sorted(comparison_preperiod_52wk.columns), axis=1)

    comparison_preperiod_52wk.insert(3, '% Change Dollars',(comparison_preperiod_52wk["dollars_per_trip_vmr_period"]-comparison_preperiod_52wk["dollars_per_trip_pre_period"])/(comparison_preperiod_52wk["dollars_per_trip_pre_period"]))
    comparison_preperiod_52wk.insert(6, '% Change Units',(comparison_preperiod_52wk["units_per_trip_vmr_period"]-comparison_preperiod_52wk["units_per_trip_pre_period"])/(comparison_preperiod_52wk["units_per_trip_pre_period"]))


    #Calculating total %_change dollars and units for the whole brand

    total_trips_vmr = unique_trips_by_period.loc[(unique_trips_by_period['analysis_periods'] == 'VMR Period'), 'distinct_trips'].values[0]
    total_trips_pperiod = unique_trips_by_period_52wk.loc[(unique_trips_by_period_52wk['analysis_periods'] == 'VMR 52wk PrePeriod'), 'distinct_trips'].values[0]

    total_dollars_vmr = vmr_period_segment_sum_pre52.loc[(vmr_period_segment_sum_pre52['analysis_periods']=='VMR Period')].dollars.sum()
    total_dollars_pperiod = vmr_period_segment_sum_pre52.loc[(vmr_period_segment_sum_pre52['analysis_periods']=='52wk Pre-Period')].dollars.sum()
    change_total_dollar_trip_vmr = ((total_dollars_vmr/total_trips_vmr)-(total_dollars_pperiod/total_trips_pperiod))/(total_dollars_pperiod/total_trips_pperiod)

    total_units_vmr = vmr_period_segment_sum_pre52.loc[(vmr_period_segment_sum_pre52['analysis_periods']=='VMR Period')].units.sum()
    total_units_pperiod = vmr_period_segment_sum_pre52.loc[(vmr_period_segment_sum_pre52['analysis_periods']=='52wk Pre-Period')].units.sum()
    change_total_units_trip_vmr = ((total_units_vmr/total_trips_vmr)-(total_units_pperiod/total_trips_pperiod))/(total_units_pperiod/total_trips_pperiod)

    comparison_preperiod_52wk = comparison_preperiod_52wk.rename(columns={segment_def: "Segment"})

    columns_titles = ["Segment","dollars_per_trip_pre_period","dollars_per_trip_vmr_period", "% Change Dollars", "units_per_trip_pre_period", "units_per_trip_vmr_period", "% Change Units"]
    comparison_preperiod_52wk= comparison_preperiod_52wk.reindex(columns=columns_titles)

    comparison_preperiod_52wk.loc['Total']= ['', (total_dollars_pperiod/total_trips_pperiod), (total_dollars_vmr/total_trips_vmr), change_total_dollar_trip_vmr, (total_units_pperiod/total_trips_pperiod), (total_units_vmr/total_trips_vmr), change_total_units_trip_vmr  ]
    comparison_preperiod_52wk.at['Total',"Segment"]='Total'

    comparison_preperiod_52wk = comparison_preperiod_52wk.reset_index(drop = True)

    comparison_preperiod_52wk = comparison_preperiod_52wk.rename(columns = {

    'dollars_per_trip_pre_period':'Dollars per Trip - Prior 52wk', 
    'dollars_per_trip_vmr_period':'Campaign Dollars per Trip',
    'units_per_trip_pre_period':'Units per Trip - Prior 52wk', 
    'units_per_trip_vmr_period':'Campaign Units per Trip'

    })

    table_slide_1 = comparison_preperiod_52wk[['Segment','Campaign Dollars per Trip', 'Dollars per Trip - Prior 52wk', '% Change Dollars', 'Campaign Units per Trip', 'Units per Trip - Prior 52wk','% Change Units']]
    table_slide_1['Campaign Dollars per Trip'] = round(table_slide_1['Campaign Dollars per Trip'],2)
    table_slide_1['Campaign Units per Trip'] = round(table_slide_1['Campaign Units per Trip'],1)
    table_slide_1 = table_slide_1.T.reset_index().T.reset_index(drop=True)
    table_slide_1.loc[:, table_slide_1.columns[1:]] = table_slide_1.loc[:, table_slide_1.columns[1:]].fillna(0)


    print(comparison_preperiod_52wk)

           Segment  Dollars per Trip - Prior 52wk  Campaign Dollars per Trip  \
0  Promoted Beauty                      22.700000                  33.740000   
1            Total                      22.696907                  33.822673   

   % Change Dollars  Units per Trip - Prior 52wk  Campaign Units per Trip  \
0          0.486344                     2.300000                 3.200000   
1          0.490189                     2.349566                 3.188334   

   % Change Units  
0        0.391304  
1        0.356989  


In [66]:
table_slide_1

,0,1,2,3,4,5,6
0,Segment,Campaign Dollars per Trip,Dollars per Trip - Prior 52wk,% Change Dollars,Campaign Units per Trip,Units per Trip - Prior 52wk,% Change Units
1,Promoted Beauty,33.74,22.7,0.486344,3.2,2.3,0.391304
2,Total,33.82,22.696907,0.490189,3.2,2.349566,0.356989


In [67]:
vmr_data = vmr_period_segment_sum_pre52.loc[(vmr_period_segment_sum_pre52['analysis_periods']=='VMR Period'), [segment_def, 'dollars', 'units']]
vmr_data.rename(columns = {'dollars':'dollars_vmr_period', 'units':'units_vmr_period'}, inplace = True)

vmr_preperiod_52wk = vmr_period_segment_sum_pre52.loc[(vmr_period_segment_sum_pre52['analysis_periods']=='52wk Pre-Period'), [segment_def, 'dollars', 'units']]
vmr_preperiod_52wk.rename(columns = {'dollars':'dollars_vmr_preperiod', 'units':'units_vmr_preperiod'}, inplace = True)

#Getting comparison table VMR vs YAGO and calculating %_change for each segment

comparison_preperiod_52wk_chart = pd.merge(vmr_data, vmr_preperiod_52wk, how="left", on=[segment_def])
comparison_preperiod_52wk_chart = comparison_preperiod_52wk_chart.reindex(sorted(comparison_preperiod_52wk_chart.columns), axis=1)

comparison_preperiod_52wk_chart.insert(3, '% Change Dollars',(comparison_preperiod_52wk_chart["dollars_vmr_period"]-comparison_preperiod_52wk_chart["dollars_vmr_preperiod"])/(comparison_preperiod_52wk_chart["dollars_vmr_preperiod"]))
comparison_preperiod_52wk_chart.insert(6, '% Change Units',(comparison_preperiod_52wk_chart["units_vmr_period"]-comparison_preperiod_52wk_chart["units_vmr_preperiod"])/(comparison_preperiod_52wk_chart["units_vmr_preperiod"]))

total_vmr_details_segm_chart = comparison_preperiod_52wk_chart[segment_def]
total_vmr_details_units_chart = comparison_preperiod_52wk_chart["% Change Units"]
total_vmr_details_dollars_chart = comparison_preperiod_52wk_chart["% Change Dollars"]

comparison_preperiod_52wk_chart

,brand_desc,dollars_vmr_period,dollars_vmr_preperiod,% Change Dollars,units_vmr_period,units_vmr_preperiod,% Change Units
0,Promoted Beauty,1.104634e+08,3.377358e+08,-0.67293,10412960,34962146,-0.702165


## **7) SIZE BASKETS DURING VMR PERIOD AND VMR REDEMPTION PERIOD:**

7a) Clasifying all the trips occured between the VMR Start and the End of Redemption Period as: Reward Trips, AO Trips during VMR Period, Redemption Trips and AO Trips during Redemption Period.

In [68]:
#Modified code from Arthur Li's previous code.

#Create New date filter to include redemption period:

curs.execute(f'''

    DROP table if exists VMR_{brand_nm}_red_date_filter;
    CREATE temp table VMR_{brand_nm}_red_date_filter as
    
        SELECT distinct date_key, 
               cal_dt, 
               cal_sun_wk_ending_dt, 
               cal_sun_wk_ending_rank_nbr
        FROM date_v
        WHERE cal_dt between '{reward_print_start}' and '{redemption_end}'
        
    DISTRIBUTE REPLICATE;
    
''')    
    
    
#Select Redemption trips:


curs.execute(f'''
    
    DROP table if exists VMR_{brand_nm}_redemption_trips; 
    CREATE temp table VMR_{brand_nm}_redemption_trips as 
        
        SELECT distinct a.ord_event_key
        
        FROM ORD_RED_CNSMR_FACT_NE_V a
        
            INNER JOIN VMR_{brand_nm}_red_date_filter d ON (a.ord_date_key = d.date_key)
            INNER JOIN VMR_{brand_nm}_promo_filter pv ON (a.promo_varnt_key=pv.promo_varnt_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
        
        DISTRIBUTE ON (ord_event_key);
''')


#curs.execute(f'''
    
 #   DROP table if exists VMR_{brand_nm}_redemption_trips; 
  #  CREATE temp table VMR_{brand_nm}_redemption_trips as 
    
    
   #         SELECT distinct a.ord_event_key
                
    #        FROM gs1_databar_redemption_fact_v a
            
     #       INNER JOIN VMR_{brand_nm}_red_date_filter d ON (a.ord_date_key = d.date_key)
      #      INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            
       #     WHERE a.gs1_offer_cd in (043101) 
        #        and a.gs1_company_prefix_cd in (0605030) 
            
    
        #DISTRIBUTE ON (ord_event_key);
#''')
        
        
#Select All trips during Reward & Redemption Period with labelling Reward, Redemption and AO Trips:

curs.execute(f'''

    DROP table if exists VMR_{brand_nm}_redemption_period;
    CREATE temp table VMR_{brand_nm}_redemption_period as 
        
        SELECT a.ord_event_key, 
               a.tot_ord_amt,
               CASE WHEN r.ord_event_key is not null then 'Reward Trip'
                    WHEN r.ord_event_key is null and rd.ord_event_key is null and d.cal_dt between '{reward_print_start}' and '{reward_print_end}' then 'AO Trips - Reward Period'
                    ELSE 'NA' END as reward_p_flag,
                
               CASE WHEN r.ord_event_key is null and rd.ord_event_key is not null then 'Redemption Trip'
                    WHEN r.ord_event_key is null and rd.ord_event_key is null then 'AO Trips - Redemption Period'
                    ELSE 'NA' END as redemp_p_flag
        
        FROM ord_sum_cnsmr_fact_ne_v a
        
            INNER JOIN VMR_{brand_nm}_red_date_filter d ON (a.ord_date_key = d.date_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            LEFT JOIN VMR_{brand_nm}_reward_events r ON (a.ord_event_key = r.ord_event_key)
            LEFT JOIN VMR_{brand_nm}_redemption_trips rd ON (rd.ord_event_key = a.ord_event_key)
            
        DISTRIBUTE RANDOM;   
       
''')

In [69]:
nbr_reward_trips = pd.read_sql(f'''

    SELECT COUNT(distinct a.ord_event_key) as trips,
           SUM(a.tot_ord_amt) as dollars,
           dollars::float/trips::float as avg_basket_size          
    FROM ord_sum_cnsmr_fact_ne_v a
        INNER JOIN VMR_{brand_nm}_redemption_period b ON (b.ord_event_key = a.ord_event_key)
    WHERE reward_p_flag = 'Reward Trip'
        
    ''',conn)

print('The number of reward trips are: '+ str(nbr_reward_trips['trips'].values[0]))
print('The total dollars for all the reward trips is: '+ str(nbr_reward_trips['dollars'].values[0]))
print('The avg basket dollars for all the reward trips is: '+ str(nbr_reward_trips['avg_basket_size'].values[0]))

The number of reward trips are: 2967989
The total dollars for all the reward trips is: 189263206.73
The avg basket dollars for all the reward trips is: 63.7681631333539


In [70]:
nbr_reward_promoted_trips = pd.read_sql(f'''

    SELECT COUNT(distinct a.ord_event_key) as trips,
           SUM(a.purch_amt) as dollars,
           dollars::float/{nbr_reward_trips['trips'].values[0]}::float as avg_basket_size
    FROM ord_trd_itm_cnsmr_fact_ne_v a
        INNER JOIN VMR_{brand_nm}_redemption_period b ON (b.ord_event_key = a.ord_event_key)
        INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
          
        WHERE brand_nbr in ({brand_nbr_str}) and reward_p_flag = 'Reward Trip'

        
    ''',conn)

print('The total dollars of promoted product in reward trips is: '+ str(nbr_reward_promoted_trips['dollars'].values[0]))
print('The avg basket dollars of promoted product in reward trips is: '+ str(nbr_reward_promoted_trips['avg_basket_size'].values[0]))
print('The % promoted product accounted in reward baskets: '+ str(round(nbr_reward_promoted_trips['avg_basket_size'].values[0]/nbr_reward_trips['avg_basket_size'].values[0]*100)))


The total dollars of promoted product in reward trips is: 122808075.76
The avg basket dollars of promoted product in reward trips is: 41.3775373695792
The % promoted product accounted in reward baskets: 65


In [71]:
nbr_ao_reward_trips = pd.read_sql(f'''

    SELECT count(distinct a.ord_event_key)
    FROM ord_trd_itm_cnsmr_fact_ne_v a
        INNER JOIN VMR_{brand_nm}_redemption_period b ON (b.ord_event_key = a.ord_event_key)
        INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
    WHERE reward_p_flag != 'NA'
          and c.brand_nbr IN ({brand_nbr_str})

    ''',conn)


print('The number of all brand transactions during Reward period is: '+ str(nbr_ao_reward_trips['count'].values[0]))
pct_reward_trips = round((total_reward_trans_var['count'].values[0]/nbr_ao_reward_trips['count'].values[0]),5)
print('The pct of reward trips among all brand transactions during VMR campaign is: '+ str(pct_reward_trips*100) + '%')

The number of all brand transactions during Reward period is: 13496561
The pct of reward trips among all brand transactions during VMR campaign is: 21.991%


In [72]:
#Same than above, just for double checking numbers

nbr_ao_reward_trips = pd.read_sql(f'''

    SELECT count(distinct a.ord_event_key)
    
    FROM ord_trd_itm_cnsmr_fact_ne_v a
    
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            LEFT JOIN VMR_{brand_nm}_reward_events r ON (a.ord_event_key = r.ord_event_key)
            LEFT JOIN VMR_{brand_nm}_redemption_trips rd ON (rd.ord_event_key = a.ord_event_key)
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)

    WHERE b.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' and 
     r.ord_event_key IS NULL and rd.ord_event_key IS NULL and brand_nbr IN ({brand_nbr_str})

    ''',conn)

print('The number of reward trips during VMR Campaign is: '+ str(total_reward_trans_var['count'].values[0]))
print('The number of all brand trips during Reward period is: '+ str(nbr_ao_reward_trips['count'].values[0]))

pct_reward_trips = round((total_reward_trans_var['count'].values[0])/(nbr_ao_reward_trips['count'].values[0]+total_reward_trans_var['count'].values[0]),5)
print('The pct of reward trips among all brand transactions during VMR campaign is: '+ str(pct_reward_trips*100) + '%')

The number of reward trips during VMR Campaign is: 2967989
The number of all brand trips during Reward period is: 10528572
The pct of reward trips among all brand transactions during VMR campaign is: 21.991%


In [73]:
nbr_redemp_trips = pd.read_sql(f'''

    SELECT COUNT(distinct a.ord_event_key) as trips,
           SUM(tot_ord_amt) as dollars,
           dollars::float/trips::float as avg_basket_size          
    FROM ord_sum_cnsmr_fact_ne_v a
        INNER JOIN VMR_{brand_nm}_redemption_trips b ON (b.ord_event_key = a.ord_event_key)
        INNER JOIN consumer_id_v p ON (a.ord_designated_cnsmr_id_key = p.cnsmr_id_key)
        
    ''',conn)

print('The number of redemption trips are: '+ str(nbr_redemp_trips['trips'].values[0]))
print('The total dollars for all the redemption trips is: '+ str(nbr_redemp_trips['dollars'].values[0]))
print('The avg basket dollars for all the redemption trips is: '+ str(nbr_redemp_trips['avg_basket_size'].values[0]))

The number of redemption trips are: 334394
The total dollars for all the redemption trips is: 3050323.4
The avg basket dollars for all the redemption trips is: 9.12194417363948


In [74]:
nbr_redemp_promoted = pd.read_sql(f'''

    SELECT COUNT(distinct a.ord_event_key) as trips,
           SUM(a.purch_amt) as dollars,
           dollars::float/{nbr_redemp_trips['trips'].values[0]}::float as avg_basket_size
    FROM ord_trd_itm_cnsmr_fact_ne_v a
        INNER JOIN VMR_{brand_nm}_redemption_trips b ON (b.ord_event_key = a.ord_event_key)
        INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
        INNER JOIN consumer_id_v p ON (a.ord_designated_cnsmr_id_key = p.cnsmr_id_key)
          
        WHERE brand_nbr in ({brand_nbr_str})

    ''',conn)

print('The number of redemption trips that includes the promoted product is: '+ str(nbr_redemp_promoted['trips'].values[0]))
print('The total dollars of promoted product in the redemption baskets is: '+ str(nbr_redemp_promoted['dollars'].values[0]))
print('The avg dollars of promoted product in redemption baskets is: '+ str(nbr_redemp_promoted['avg_basket_size'].values[0]))

if nbr_redemp_promoted['trips'].values[0] != 0:

    print('The % promoted product accounted in redemption baskets: '+ str(round(nbr_redemp_promoted['avg_basket_size'].values[0]/nbr_redemp_trips['avg_basket_size'].values[0]*100)))

The number of redemption trips that includes the promoted product is: 159897
The total dollars of promoted product in the redemption baskets is: 2300762.33
The avg dollars of promoted product in redemption baskets is: 6.88039357763596
The % promoted product accounted in redemption baskets: 75


In [75]:
if nbr_redemp_promoted['trips'].values[0] != 0:

    #Getting summarize table:

    curs.execute(f'''

        DROP table if exists VMR_{brand_nm}_baskets_results; 
        CREATE temp table VMR_{brand_nm}_baskets_results as


        SELECT reward_p_flag as analysis_period, 
               sum(tot_ord_amt) as Dollars, 
               count(distinct ord_event_key) as Trips, 
               dollars::float/trips::float as avg_basket_size          
        FROM VMR_{brand_nm}_redemption_period
        WHERE reward_p_flag != 'NA'
        GROUP BY 1

        UNION

        SELECT redemp_p_flag as analysis_period, 
               sum(tot_ord_amt) as dollars, 
               count(distinct ord_event_key) as trips, 
               dollars::float/trips::float as avg_basket_size
        FROM VMR_{brand_nm}_redemption_period
        WHERE redemp_p_flag != 'NA'
        GROUP BY 1

        DISTRIBUTE RANDOM;   

    ''')    


    df_vmr_ttls_basket = pd.read_sql(f''' 

    SELECT * FROM VMR_{brand_nm}_baskets_results

    ORDER BY case WHEN analysis_period = 'Reward Trip' THEN 1
                  WHEN analysis_period = 'AO Trips - Reward Period' THEN 2
                  WHEN analysis_period = 'Redemption Trip' THEN 3
                  WHEN analysis_period = 'AO Trips - Redemption Period' THEN 4 END 
    ''',conn)

    df_vmr_ttls_basket.rename(columns = {'analysis_period':'Analysis period', 'avg_basket_size':'Average Basket Size', 'dollars':'Dollars', 'trips':'Trips'}, inplace = True)


    df_vmr_ttls_basket.loc[4] = ['% change reward trip', '', '', (df_vmr_ttls_basket.loc[0,'Average Basket Size']-df_vmr_ttls_basket.loc[1,'Average Basket Size'])/(df_vmr_ttls_basket.loc[1,'Average Basket Size'])]
    df_vmr_ttls_basket.loc[5]= ['% change redemption trip', '', '', (df_vmr_ttls_basket.loc[2,'Average Basket Size']-df_vmr_ttls_basket.loc[3,'Average Basket Size'])/(df_vmr_ttls_basket.loc[3,'Average Basket Size'])]

    df_vmr_ttls_basket.iloc[2], df_vmr_ttls_basket.iloc[4] = df_vmr_ttls_basket.iloc[4], df_vmr_ttls_basket.iloc[2]
    df_vmr_ttls_basket.iloc[4], df_vmr_ttls_basket.iloc[3] = df_vmr_ttls_basket.iloc[3], df_vmr_ttls_basket.iloc[4]


    print(df_vmr_ttls_basket)
    
else:
    
    curs.execute(f'''

        DROP table if exists VMR_{brand_nm}_baskets_results; 
        CREATE temp table VMR_{brand_nm}_baskets_results as


        SELECT reward_p_flag as analysis_period, 
               sum(tot_ord_amt) as Dollars, 
               count(distinct ord_event_key) as Trips, 
               dollars::float/trips::float as avg_basket_size          
        FROM VMR_{brand_nm}_redemption_period
        WHERE reward_p_flag != 'NA'
        GROUP BY 1

        DISTRIBUTE RANDOM;   

    ''')    


    df_vmr_ttls_basket = pd.read_sql(f''' 

    SELECT * FROM VMR_{brand_nm}_baskets_results

    ''',conn)

    df_vmr_ttls_basket.rename(columns = {'analysis_period':'Analysis period', 'avg_basket_size':'Average Basket Size', 'dollars':'Dollars', 'trips':'Trips'}, inplace = True)


    #df_vmr_ttls_basket.loc[2] = ['% change reward trip', '', '', (df_vmr_ttls_basket.loc[0,'Average Basket Size']-df_vmr_ttls_basket.loc[1,'Average Basket Size'])/(df_vmr_ttls_basket.loc[1,'Average Basket Size'])]

    #df_vmr_ttls_basket.iloc[2] = df_vmr_ttls_basket.iloc[4], df_vmr_ttls_basket.iloc[2]


    print(df_vmr_ttls_basket)
    
    print('The number of redemption trips is zero, so this section is not going to be shown.')

                Analysis period        Dollars      Trips  Average Basket Size
0                   Reward Trip   189263206.73    2967989            63.768163
1      AO Trips - Reward Period  3744078707.77  124943854            29.966089
2          % change reward trip                                       1.128011
3               Redemption Trip     1951770.93     311636             6.262983
4  AO Trips - Redemption Period  5703127028.39  187357387            30.439830
5      % change redemption trip                                      -0.794250


## **8) BRAND COMBINATIONS PURCHASED ON THE VMR REWARD TRIP (FOR TRACKABLE ID's):**

Evaluating the limit of segments to perform the combinations:
(It must be less than 53 segments. If not, the code will filter by those segemnts with % reward trips > 1)

In [76]:
df_segments = df_segments_2
nbr_segments = len(total_vmr_details_by_brand[total_vmr_details_by_brand['Segment'] != 'Total'])


#NEW:
if len(df_segments) > 52:
    
    nbr_rows = len(df_segments)-52
    df_segments = df_segments.head(len(df_segments) - nbr_rows )
    


if nbr_segments<52:
    
    df_segments = df_segments[['segment', 'segm_nbr']]
    
    yb_load(Df = df_segments,
        table_name = f'''VMR_{brand_nm}_segments_{analyst}''',
        userid = getpass.getuser(),
        passwd = readpw("Yellowbrick"),
        append = False,
        database= f'{dbase}')

    check_upload = pd.read_sql(f"""select count(*) from VMR_{brand_nm}_segments_{analyst}""", conn)
    print('Uploaded number of records: ' + str(check_upload.values[0]))
    
    df_segments = df_segments.rename(columns={'segment': 'Segment', 'segm_nbr':'Segm Nbr'})
    print(df_segments)
    
    
else:
    
    if segment_type == 1:
    
        df_segments = pd.read_sql(f'''

        SELECT {segment_def} as segment,
               brand_nbr as segm_nbr,
               count(distinct o.ord_event_key) as reward_trips,
               {total_reward_trans} as total_reward_trips,
               round((reward_trips::float/total_reward_trips::float)*100,2) as pct_reward_trips

        FROM ord_trd_itm_cnsmr_fact_ne_v o

        INNER JOIN VMR_{brand_nm}_upc_filter b ON (o.trade_item_key = b.trade_item_key)
        INNER JOIN VMR_{brand_nm}_reward_events c ON (o.ord_event_key = c.ord_event_key)

        GROUP BY 1,2

        HAVING pct_reward_trips > 1

        ORDER BY 2 ASC
        
        LIMIT 52

        ''',conn)

        
        df_segments['segm_nbr'] = range(1, len(df_segments) + 1)

        df_segments = df_segments[['segment', 'segm_nbr']]

        yb_load(Df = df_segments,
            table_name = f'''VMR_{brand_nm}_segments_{analyst}''',
            userid = getpass.getuser(),
            passwd = readpw("Yellowbrick"),
            append = False,
            database= f'{dbase}')

        check_upload = pd.read_sql(f"""select count(*) from VMR_{brand_nm}_segments_{analyst}""", conn)
        print('Uploaded number of records: ' + str(check_upload.values[0]))

        df_segments = df_segments.rename(columns={"segm_nbr": "Segm Nbr", "segment": "Segment"})
        print("The number of segments is over the max limit (53). Segments were filtered by reward trips > 1%")
        print(df_segments)
        
    else:
        
        df_segments = pd.read_sql(f'''

        SELECT {segment_def} as segment,
               count(distinct o.ord_event_key) as reward_trips,
               {total_reward_trans} as total_reward_trips,
               round((reward_trips::float/total_reward_trips::float)*100,2) as pct_reward_trips

        FROM ord_trd_itm_cnsmr_fact_ne_v o

        INNER JOIN VMR_{brand_nm}_upc_filter b ON (o.trade_item_key = b.trade_item_key)
        INNER JOIN VMR_{brand_nm}_reward_events c ON (o.ord_event_key = c.ord_event_key)

        GROUP BY 1

        HAVING pct_reward_trips > 1

        ORDER BY 1
        
        LIMIT 52

        ''',conn)


        df_segments['segm_nbr'] = list(range(1,len(df_segments)+1))
        df_segments = df_segments[['segment', 'segm_nbr']]
        
        

        yb_load(Df = df_segments,
            table_name = f'''VMR_{brand_nm}_segments_{analyst}''',
            userid = getpass.getuser(),
            passwd = readpw("Yellowbrick"),
            append = False,
            database= f'{dbase}')

        check_upload = pd.read_sql(f"""select count(*) from VMR_{brand_nm}_segments_{analyst}""", conn)
        print('Uploaded number of records: ' + str(check_upload.values[0]))

        df_segments = df_segments.rename(columns={"segm_nbr": "Segm Nbr", "segment": "Segment"})
        print("The number of segments is over the max limit (53). Segments were filtered by reward trips > 1%")
        print(df_segments)
        

Checking that the save and YBtools locations

Checking and removing if a table by the same name exists


Creating table


Here is your new table structure


***************************************************************

	 FIELD: 	 Pandas -> YB 

	 segment: 	 object -> VARCHAR( 15 )
	 segmnbr: 	 int64 -> BIGINT
***************************************************************

Saving data frame to the drive for loading into YB

Bulk loading has commenced

removing pandas saved file
Uploaded number of records: [1]
           Segment  Segm Nbr
0  Promoted Beauty         1


In [77]:
#Modified from K.Mertens's J&J custom VMR code.

#Functions to rename columns in the dataframes, Don't change it!

renaming_dict = {
    'ord_event_key' : 'ord_event_key',
    'cnsmr_id_key' : 'cnsmr_id_key',
    'analysis_period_p1p2': 'Analysis Period', 
    'analysis_period_rec52': 'Analysis Period', 
    'analysis_periods': 'Analysis Period', 
    'cal_sun_wk_ending_dt': 'WE Date',
    'nbr_of_upcs': 'Raw # of UPCs included',
    'brand_desc': 'Brand',
    'sur_seg': 'Purchasing Segment',
    'price_seg': 'Pricing Segment',
    'fin_cmit_contract_nbr': 'Contract Nbr',
    'promo_src_id_txt': 'BL',
    'segment':'Segment',
    'segm_nbr':'Segm Nbr'
}

format_mapping = {
    'Raw # of UPCs included' : "{:,.0f}",
    'Contract Nbr': "{:.0f}"
}


#def convert_to_uppercase(m):
 #   return m.group(1) + m.group(2).upper()

def convert_to_uppercase(s):
    words = s.split()
    capitalized_words = [word.capitalize() for word in words]
    return ' '.join(capitalized_words)


def sql_to_pretty_names_dict(dataframe_to_update):
    global renaming_dict, format_mapping
    
    list_sql_names = list(dataframe_to_update.columns)

    # Replace underscores with spaces   
    list_pretty_names = [sql_nm.replace("_", " ") for sql_nm in list_sql_names]

    # Replace lowercase upc with upper case
    list_pretty_names = [sql_nm.replace("upc", "UPC") for sql_nm in list_pretty_names]
    
    # Replace lowercase upc with upper case    
    list_pretty_names = [convert_to_uppercase(pretty_name) for pretty_name in list_pretty_names]
    list_pretty_names = [pretty_name.replace(r"Vol(?!ume)", "Volume") for pretty_name in list_pretty_names]
    list_pretty_names = [pretty_name.replace(r"(?<!EQ )Volume", "EQ Volume") for pretty_name in list_pretty_names]
    
    
    for i in range(len(list_sql_names)): 
        if list_sql_names[i] in renaming_dict : 
            continue 
        sql_name = list_sql_names[i]
        pretty_name = list_pretty_names[i]
        renaming_dict[sql_name] = pretty_name
        
        # auto-generate the format mapping for the new entries
        if "Share" in pretty_name or "Pct" in pretty_name or "Pen" in pretty_name :
            format_mapping[pretty_name] = "{:.1%}"
        elif "Dollars Per" in pretty_name or "Price" in pretty_name and not "Seg" in pretty_name:
            format_mapping[pretty_name] = "${:.2f}"
        elif "Per Trip" in pretty_name :
            format_mapping[pretty_name] = "{:.1f}"
        elif "Dollars" in pretty_name : 
            format_mapping[pretty_name] = "${:,.0f}"
        elif "UPCs" in pretty_name or "Volume" in pretty_name or "Units" in pretty_name or "Trips" in pretty_name or "Stores" in pretty_name or "Shopper" in pretty_name or "Prints" in pretty_name:
            format_mapping[pretty_name] = "{:,.0f}"
   

In [78]:
df_segments

,Segment,Segm Nbr
0,Promoted Beauty,1


In [79]:
#Modified code from K.Mertens's J&J custom VMR code.

###### First make the function to get the ascii letter A-Z,a-z

def get_letter_for_brand_nbr(nbr):
    '''
    Given a number 1-52, this returns A-Z, a-z.
    '''
    if nbr < 27 :
        return chr(nbr+ord("A")-1)
    elif nbr < 53 :
        return chr(nbr+ord("a")-27)
    raise IndexError(f"This function can only work with up to 52 brand_nbr items.  You gave me {nbr}")
    
###### Make a dictionary that maps from the Letter to the Brand Description

alpha_to_brand_desc = {}

idx = {name: i for i, name in enumerate(list( df_segments ),1)}
for row in df_segments.itertuples():
    curr_letter = get_letter_for_brand_nbr(row[idx['Segm Nbr']])
    curr_desc = row[idx['Segment']]
    alpha_to_brand_desc[curr_letter] = curr_desc

print(alpha_to_brand_desc)

{'A': 'Promoted Beauty'}


In [80]:
row[idx['Segm Nbr']]

1

In [81]:
#Creating a table that contains all the reward trips details: ord_event_key, cnmr_id, brand_nbr, brand_descr

curs.execute(f'''

    DROP table if exists VMR_{brand_nm}_purch_details_all_trips;
    CREATE temp table VMR_{brand_nm}_purch_details_all_trips as
        
          SELECT distinct ou.ord_event_key
                 ,d.segment as segment
                 ,d.segmnbr as segm_nbr                
                 ,b.cal_dt
                 ,b.cal_sun_wk_ending_dt
                 ,b.analysis_period
                 ,b.prior_period
                 ,c.brand_nbr
                 ,ou.ord_designated_cnsmr_id_key as cnsmr_id_key
                 ,sum(ou.purch_qty) as brand_units
                 ,sum(ou.purch_amt) as brand_dollars
          
          FROM ord_trd_itm_cnsmr_fact_ne_v ou
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (ou.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (ou.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_segments_{analyst} d ON (c.{segment_def} = d.segment)
                 
          
          WHERE ou.purch_amt > 0
                and ou.purch_qty > 0
                and c.brand_nbr in ({brand_nbr_str})
        
        GROUP BY 1,2,3,4,5,6,7,8,9
        
        DISTRIBUTE ON (ord_event_key);
        
''')

In [82]:
#Modified from K.Mertens's J&J custom VMR code.

if nbr_segments > 1 :
    
####  Download purchase details from YB, organized by ord_event_key\

    curs.execute(f'''
    
        DROP table if exists VMR_{brand_nm}_combos_results_{analyst}; 
        CREATE table VMR_{brand_nm}_combos_results_{analyst} as
    
        SELECT a.ord_event_key,
               a.segm_nbr,
               a.segment
              ,sum(a.brand_units) as units
              ,sum(a.brand_dollars) as dollars 
              
        FROM VMR_{brand_nm}_purch_details_all_trips a
        
        INNER JOIN VMR_{brand_nm}_reward_events rd ON (rd.ord_event_key = a.ord_event_key) 
        INNER JOIN consumer_id_v p ON (a.cnsmr_id_key = p.cnsmr_id_key)

          WHERE p.abuser_ind= 'N' 
                and p.unident_ind = 'N'
                and p.really_bad_abuser_ind='N'
                and p.cnsmr_id_key>0
                and analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
        GROUP by 1,2,3
    
        DISTRIBUTE RANDOM;
    
    
    ''')
    
    
    from YB_unload import yb_unload

    df_vmr_period_purch = yb_unload(getpass.getuser(), readpw("Yellowbrick"), table_name = f'''VMR_{brand_nm}_combos_results_{analyst}''', database = f'{dbase}')
    
    df_vmr_period_purch.sort_values(['ord_event_key','segm_nbr']).reset_index().drop(columns=['index'])
    df_vmr_period_purch_test = df_vmr_period_purch
    
    df_vmr_period_purch[['segm_nbr','segment']] = df_vmr_period_purch[['segm_nbr','segment']].astype("category")
    sql_to_pretty_names_dict(df_vmr_period_purch)                           # update the renaming dictionary to include these column names
    df_vmr_period_purch = df_vmr_period_purch.rename(columns=renaming_dict)
    df_vmr_period_purch = df_vmr_period_purch.sort_values(['ord_event_key','Segm Nbr']).reset_index().drop(columns=['index'])

In [83]:
if nbr_segments > 1 :
    
    #### This creates 3 dictionaries, to accumulate the number of tranactions, dollars, and units for each brand combination

    current_ord_event_key = 0

    current_brands = ""   # empty string
    current_amount = 0.0  # no dollars accumulated yet
    current_units = 0.0

    total_transactions = {}
    total_dollars = {}
    total_units = {}
    
    idx = {name: i for i, name in enumerate(list(df_vmr_period_purch),1)}
    
    #{'ord_event_key': 1, 'Brand Nbr': 2, 'Brand': 3, 'Units': 4, 'Dollars': 5}
    
    for row in df_vmr_period_purch.itertuples():
        
    #Pandas(Index=0, ord_event_key=509252267271, _2=3, Brand='Lip', Units=1, Dollars=1.83)
        
        #if we are about to start a new transaction, add the details of the old one to our accumulation
        
        if current_ord_event_key != row[idx['ord_event_key']] and current_ord_event_key != 0:
            if not current_brands in total_transactions:
                total_transactions[current_brands] = 1
                total_dollars[current_brands] = current_amount
                total_units[current_brands] = current_units
            else:
                total_transactions[current_brands] += 1
                total_dollars[current_brands] += current_amount
                total_units[current_brands] += current_units

            current_brands = ""   # reset this to an empty string
            current_amount = 0.0  # reset this to no dollars accumulated yet
            current_units = 0.0   # reset this to no units accumulated yet

        brand_letter = get_letter_for_brand_nbr(row[idx['Segm Nbr']])
        current_brands += brand_letter
        current_amount += row[idx['Dollars']]
        current_units  += row[idx['Units']]
        current_ord_event_key = row[idx['ord_event_key']]    

    if not current_brands in total_transactions:
        total_transactions[current_brands] = 1
        total_dollars[current_brands] = current_amount
        total_units[current_brands] = current_units
    else:
        total_transactions[current_brands] += 1
        total_dollars[current_brands] += current_amount
        total_units[current_brands] += current_units

####  Put the dictionaries into a dataframe, sorted by transactions    

    df_vmr_period_combos = pd.DataFrame(data=[total_transactions,total_units,total_dollars],index=['Trips','Units','Dollars']).T
    df_vmr_period_combos=df_vmr_period_combos.sort_values(['Trips'],ascending = False).reset_index()
    df_vmr_period_combos.rename(columns = {'index':'Segment'},inplace=True)
    df_vmr_period_combos['No. of Segments'] = df_vmr_period_combos['Segment'].str.len()
    df_vmr_period_combos['Pct of Trips'] = round(df_vmr_period_combos['Trips'] / df_vmr_period_combos['Trips'].sum(),5)
    df_vmr_period_combos['Pct of Units'] = round(df_vmr_period_combos['Units'] / df_vmr_period_combos['Units'].sum(),5)
    df_vmr_period_combos['Pct of Dollars'] = round(df_vmr_period_combos['Dollars'] / df_vmr_period_combos['Dollars'].sum(),5)
    df_vmr_period_combos['Units per Trip'] = round(df_vmr_period_combos['Units'] / df_vmr_period_combos['Trips'],1)
    df_vmr_period_combos['Dollars per Trip'] = round(df_vmr_period_combos['Dollars'] / df_vmr_period_combos['Trips'],2)

    sql_to_pretty_names_dict(df_vmr_period_combos)                           # update the renaming dictionary to include these column names
    df_vmr_period_combos = df_vmr_period_combos.rename(columns=renaming_dict) # rename the columns, over-writing the original DataFrame column names

    
####  Change the A-Z,a-z code back to the brand description in the UPC hierarchy (previously downloaded from YB)    

    for i in range(len(df_vmr_period_combos)):
        curr_letters = df_vmr_period_combos.at[i,'Segment']
        curr_brands = ""
        for letter in curr_letters: 
            curr_brands += alpha_to_brand_desc[letter] + ' + '
        curr_brands = curr_brands[:-3]                            #remove the last 3 characters of the string, to delete the final ' + ' out of the brand combination
        df_vmr_period_combos.at[i,'Segment'] = curr_brands

    
####  Total by sub-brand on the VMR reward earning trip
    
    df_vmr_period_totals = pd.read_sql(f''' 
        SELECT a.segm_nbr, a.segment
              ,count(distinct a.ord_event_key) as trips
              ,sum(a.brand_units) as units
              ,sum(a.brand_dollars) as dollars
        FROM VMR_{brand_nm}_purch_details_all_trips a
        INNER JOIN VMR_{brand_nm}_reward_events rd ON (rd.ord_event_key = a.ord_event_key)
        INNER JOIN consumer_id_v p ON (a.cnsmr_id_key = p.cnsmr_id_key)
        WHERE p.abuser_ind= 'N' 
                and p.unident_ind = 'N'
                and p.really_bad_abuser_ind='N'
                and p.cnsmr_id_key>0
                and a.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
        GROUP BY 1,2
        
        UNION ALL
        
        SELECT 999 as segm_nbr, 'Total' as segment
              ,count(distinct a.ord_event_key) as trips
              ,sum(a.brand_units) as units
              ,sum(a.brand_dollars) as dollars
        FROM VMR_{brand_nm}_purch_details_all_trips a
        INNER JOIN VMR_{brand_nm}_reward_events rd ON (rd.ord_event_key = a.ord_event_key)
        INNER JOIN consumer_id_v p ON (a.cnsmr_id_key = p.cnsmr_id_key)
        WHERE p.abuser_ind= 'N' 
                and p.unident_ind = 'N'
                and p.really_bad_abuser_ind='N'
                and p.cnsmr_id_key>0
                and a.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
        GROUP BY 1,2
    ''',conn).sort_values(['segm_nbr']).reset_index().drop(columns=['index','segm_nbr'])

    ttl_trips = df_vmr_period_totals[df_vmr_period_totals['segment'] == 'Total']['trips'].item()
    ttl_units = df_vmr_period_totals[df_vmr_period_totals['segment'] == 'Total']['units'].item()
    ttl_dollars = df_vmr_period_totals[df_vmr_period_totals['segment'] == 'Total']['dollars'].item()

    df_vmr_period_totals = df_vmr_period_totals.assign(pct_of_trips = lambda x: ( round((x['trips'] / ttl_trips),3)),
                                                        pct_of_units = lambda x: ( round((x['units'] / ttl_units),3)),
                                                        pct_of_dollars = lambda x: ( round((x['dollars'] / ttl_dollars),3)),
                                                       )

    print("Hey")
    print(df_vmr_period_totals)
    
    sql_to_pretty_names_dict(df_vmr_period_totals)                           # update the renaming dictionary to include these column names
    df_vmr_period_totals = df_vmr_period_totals.rename(columns=renaming_dict) # rename the columns, over-writing the original DataFrame column names
    
    ttl_row_num = df_vmr_period_totals.index[df_vmr_period_totals['Segment'] == 'Total'].tolist()
    df_vmr_period_totals.iloc[ttl_row_num,4:7] = [np.nan, np.nan, np.nan]
    
    
    df_vmr_period_combos = df_vmr_period_combos.sort_values('Pct Of Trips', ascending = False)

    
    if all_combos == False:
    
        rows = df_vmr_period_combos['Pct Of Trips'].shape[0]
        ao_rows = df_vmr_period_combos.loc[10:rows-1].sum()

    df_vmr_period_combos_all = df_vmr_period_combos
    df_vmr_period_combos_all_out = df_vmr_period_combos_all
    
    if all_combos == False:
    
        df_vmr_period_combos = df_vmr_period_combos.head(10)
        df_vmr_period_combos.loc[11] = ao_rows
        df_vmr_period_combos.at[11,'Segment']='AO combinations'

    df_vmr_period_combos = df_vmr_period_combos.rename(columns={"Segment": "Segment", "No. Of Segments": "No. of Segments" })
    df_vmr_period_combos_all = df_vmr_period_combos_all.rename(columns={"Segment": "Segment", "No. Of Segments": "No. of Segments" })
    df_vmr_period_combos_all_out = df_vmr_period_combos_all_out.rename(columns={"Segment": "Segment", "No. Of Segments": "No. of Segments" })
    
    df_vmr_period_combos = df_vmr_period_combos.reset_index(drop=True)
    df_vmr_period_combos_all = df_vmr_period_combos_all.reset_index(drop=True)
    df_vmr_period_combos_all_out = df_vmr_period_combos_all_out.reset_index(drop=True)
    df_vmr_period_totals = df_vmr_period_totals.reset_index(drop=True)
    
    combos_slide = df_vmr_period_combos[['Segment', 'Pct Of Trips']]
    combos_slide = combos_slide[combos_slide['Segment']!='AO combinations']
    
    print(df_vmr_period_combos)
    print(df_vmr_period_totals)
    
else :
    
    df_vmr_period_combos = []
    df_vmr_period_totals = []
    
    print('Only one segment / no sub-brands in promoted brand group.  Skipping Combinations Analysis.')


Only one segment / no sub-brands in promoted brand group.  Skipping Combinations Analysis.


results_insight = pd.read_sql(f'''

    WITH transactions as (
    select ord_event_key, COUNT(ord_event_key) as counts
    from VMR_{brand_nm}_combos_results_{analyst}
    GROUP BY 1
    HAVING counts = 2)
    
     SELECT trade_item_cd,
                 sum(o.purch_amt) as dollars
          
          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_printing_stores s ON (o.ord_touchpoint_key = s.touchpoint_key)
                 INNER JOIN transactions t ON (o.ord_event_key = t.ord_event_key)
          
          WHERE b.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
                and o.purch_amt > 0
                and o.purch_qty > 0
                and brand_nbr in ({brand_nbr_str})
                and o.ord_date_key between s.min_actual_date_key and s.max_actual_date_key
        
        GROUP BY 1
        
        ORDER BY 2 DESC
            
    
''',conn) 

results_insight

#results_insight.to_excel('/analytical_services/Public/moricejuan/mark_anthony_combination_insight.xlsx', index=False)

In [84]:
if nbr_segments > 1 :

    df_vmr_period_combos_all_out

## **9) SEGMENT ANALYSIS OF PURCHASES VMR REWARD TRIP (FOR CONSISTENT REWARD ID'S):**

In [85]:
#Modified from K.Mertens's J&J custom VMR code.

if nbr_segments > 1 :

####  Download purchase details from YB, organized by cnsmr_id_key

    curs.execute(f'''
        
        DROP table if exists VMR_{brand_nm}_segments_results_{analyst}; 
        CREATE table VMR_{brand_nm}_segments_results_{analyst} as
        
        SELECT a.cnsmr_id_key,
                CASE WHEN a.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' and rd.ord_event_key is not null THEN 'VMR Period'
                     WHEN a.prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN 'Prior-Period'
                     ELSE 'NA' END as analysis_periods, 
                a.segm_nbr,
                sum(a.brand_units) as units 
        FROM VMR_{brand_nm}_purch_details_all_trips a
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr d ON (a.cnsmr_id_key = d.ord_designated_cnsmr_id_key)
            INNER JOIN shopper_consistent_{analyst} s ON (a.cnsmr_id_key = s.cnsmr_id_key)
            LEFT JOIN VMR_{brand_nm}_reward_events rd ON (rd.ord_event_key = a.ord_event_key)
        WHERE analysis_periods != 'NA'
        GROUP by 1,2,3
        
        DISTRIBUTE RANDOM;
        
    ''')
    
    
    from YB_unload import yb_unload

    df_pre_vmr_cnsmrs = yb_unload(getpass.getuser(), readpw("Yellowbrick"), table_name = f'''VMR_{brand_nm}_segments_results_{analyst}''', database = f'{dbase}')
        
    df_pre_vmr_cnsmrs.sort_values(['analysis_periods','cnsmr_id_key','segm_nbr'],ascending = True).reset_index().drop(columns=['index'])
    
    
    df_pre_vmr_cnsmrs[['analysis_periods','segm_nbr']] = df_pre_vmr_cnsmrs[['analysis_periods','segm_nbr']].astype('category')
    df_pre_vmr_cnsmrs[['cnsmr_id_key']] = df_pre_vmr_cnsmrs[['cnsmr_id_key']].astype('str')

    sql_to_pretty_names_dict(df_pre_vmr_cnsmrs) # update the renaming dictionary to include these column names
    
    df_pre_vmr_cnsmrs = df_pre_vmr_cnsmrs.rename(columns=renaming_dict)   # rename the columns, over-writing the original DataFrame column names
    
        
    print("Look at me 1") #rem
    print(df_pre_vmr_cnsmrs) #rem
    
    df_pre_vmr_cnsmrs.sort_values(['Analysis Period', 'Segm Nbr'],ascending = True).reset_index().drop(columns=['index'])
    

#### This creates 2 dictionaries, to look at sub-brand purchasing before and at reward earning trip

    idx = {name: i for i, name in enumerate(list(df_pre_vmr_cnsmrs),1)}

    # Two dicts, accessible by analysis period
    consumer_behavior = {}
    consumer_behavior['Prior-Period'] = {}
    consumer_behavior['VMR Period'] = {}

    # Accumulate consumer behavhior before and during print
    for row in df_pre_vmr_cnsmrs.itertuples():
        # unpack the details from the row
        current_period = row[idx['Analysis Period']]
        current_consumer = row[idx['cnsmr_id_key']]
        brand_letter = get_letter_for_brand_nbr(row[idx['Segm Nbr']])

        # Assign consumer's brand purchase to the current period
        if not current_consumer in consumer_behavior[current_period]:
            consumer_behavior[current_period][current_consumer] = brand_letter
        else:
            consumer_behavior[current_period][current_consumer] += brand_letter

    N_prev_seg = 0
    N_new_seg = 0
    N_prev_new_seg = 0
    
    #creating empty lists where the consumer ids for each segment will be saved: (jmorice)
    ids_prev_seg = []
    ids_prev_new_seg = []
    ids_new_seg = []
    
    
    # Note: all consumers in pre-period are also in print period -- but sometimes, when we filter for purch_qty>0 and purch_amt>0.01, then the print period trip drops out.
    # This forces analysis to look at only existing brand buyers: the shoppers with purchases in both the pre-period and analysis period
    for consumer in consumer_behavior['Prior-Period']:
        # XXX skip if they are not in print period
        if not consumer in consumer_behavior['VMR Period']: continue

        # Construct sets for pre- and print-periods
        # Note: consider constructing these sets in the previous loop, rather than doing string stuff
        pre_set = set(list(consumer_behavior['Prior-Period'][consumer]))
        print_set = set(list(consumer_behavior['VMR Period'][consumer]))

        # If print set is entirely in previous, then this is "only previous" behavior
        if print_set.issubset(pre_set):
            N_prev_seg += 1
            ids_prev_seg.append(consumer)

        # no overlap is called "only new"
        elif print_set.isdisjoint(pre_set):
            N_new_seg += 1
            ids_new_seg.append(consumer)

        # must be combination
        else:
            N_prev_new_seg += 1
            ids_prev_new_seg.append(consumer)

            
    N_prev_seg_bump_up = (N_prev_seg)
    N_new_seg_bump_up = (N_new_seg)
    N_prev_new_seg_bump_up = (N_prev_new_seg)
    
    
    #Transforming the segment consumer ids's lists into dataframes:
    ids_prev_seg = pd.DataFrame(ids_prev_seg).rename(columns={0:'cnsmr_id_key'})
    ids_prev_new_seg = pd.DataFrame(ids_prev_new_seg).rename(columns={0:'cnsmr_id_key'})
    ids_new_seg = pd.DataFrame(ids_new_seg).rename(columns={0:'cnsmr_id_key'})

    
#### Put result into a dataframe and condense
    df_plus_one = pd.DataFrame(data=[N_prev_seg_bump_up,N_new_seg_bump_up + N_prev_new_seg_bump_up],index=['Bought the same segments as they did before','Added at least one new segment'])
    df_plus_one.rename(columns={df_plus_one.columns[0]: 'Nbr Of Shoppers'},inplace=True)
    df_plus_one['Pct Of Shoppers'] = round(df_plus_one['Nbr Of Shoppers'] / df_plus_one['Nbr Of Shoppers'].sum(),3)
    df_plus_one = df_plus_one.reset_index().rename(columns={'index': ''})
    
    sql_to_pretty_names_dict(df_plus_one)                           # update the renaming dictionary to include these column names
    
    print("Look at me") #rem
    print(df_plus_one) #rem
    
#### Same result into another dataframe with more detail
    df_new_existing_combos = pd.DataFrame(data=[N_prev_seg_bump_up,N_prev_new_seg_bump_up,N_new_seg_bump_up],index=['Bought the same segments as they did before','Both previous and new segment(s) on this trip','Only the new segment(s) on this trip'])
    df_new_existing_combos.rename(columns={df_new_existing_combos.columns[0]: 'Nbr Of Shoppers'},inplace=True)
    df_new_existing_combos['Pct Of Shoppers'] = round(df_new_existing_combos['Nbr Of Shoppers'] / df_new_existing_combos['Nbr Of Shoppers'].sum(),3)
    df_new_existing_combos = df_new_existing_combos.reset_index().rename(columns={'index': ''})

    sql_to_pretty_names_dict(df_new_existing_combos)                           # update the renaming dictionary to include these column names

    print("Look at me") #rem
    print(df_new_existing_combos) #rem
    
else :
    
    df_plus_one = []
    df_new_existing_combos = []   
    
    print('Only one segment / no sub-brands in promoted brand group.  Skipping Combinations Analysis.')    

Only one segment / no sub-brands in promoted brand group.  Skipping Combinations Analysis.


In [86]:
if nbr_segments > 1 :
     print(df_plus_one)

if nbr_segments > 1 :
    print(df_new_existing_combos)

##### 9b) Decomp by segment for Shoppers who bought a new segment during the reward trip:

In [87]:
if nbr_segments > 1 :

    if len(ids_new_seg)!=0 and len(ids_prev_new_seg)!=0:
        ids_new_seg_2 = ids_new_seg
        ids_prev_new_seg_2 = ids_prev_new_seg

        all_ids_new_seg = pd.concat([ids_new_seg, ids_prev_new_seg])
        all_ids_new_seg = all_ids_new_seg.reset_index(drop=True)

        yb_load(Df = all_ids_new_seg,
                table_name = f'''VMR_{brand_nm}_all_ids_new_seg_{analyst}''',
                userid = getpass.getuser(),
                passwd = readpw("Yellowbrick"),
                append = False,
                database= f'{dbase}')
        
        print('len(ids_new_seg)!=0 & len(ids_prev_new_seg)!=0')
        
        
    elif len(ids_new_seg)!=0 and len(ids_prev_new_seg)==0:
        ids_new_seg_2 = ids_new_seg
        all_ids_new_seg = ids_new_seg_2
        all_ids_new_seg = all_ids_new_seg.reset_index(drop=True)

        yb_load(Df = all_ids_new_seg,
                table_name = f'''VMR_{brand_nm}_all_ids_new_seg_{analyst}''',
                userid = getpass.getuser(),
                passwd = readpw("Yellowbrick"),
                append = False,
                database= f'{dbase}')
        
        print('len(ids_new_seg)!=0 & len(ids_prev_new_seg)==0')
        
        
    elif len(ids_new_seg)==0 and len(ids_prev_new_seg)!=0:
        ids_prev_new_seg_2 = ids_prev_new_seg
        all_ids_new_seg = ids_prev_new_seg_2
        all_ids_new_seg = all_ids_new_seg.reset_index(drop=True)

        yb_load(Df = all_ids_new_seg,
                table_name = f'''VMR_{brand_nm}_all_ids_new_seg_{analyst}''',
                userid = getpass.getuser(),
                passwd = readpw("Yellowbrick"),
                append = False,
                database= f'{dbase}')
        
        print('len(ids_new_seg)==0 & len(ids_prev_new_seg)!=0')
        
        
    else:
        all_ids_new_seg = pd.DataFrame()
        print("No new ids for this campaign.")


In [88]:
if nbr_segments > 1 :    
    
    if len(all_ids_new_seg) != 0:
    
        curs.execute(f'''       
            DROP table if exists VMR_{brand_nm}_new_buyer_flag;
            CREATE temp table VMR_{brand_nm}_new_buyer_flag as

               SELECT a.cnsmr_id_key,
                       a.segm_nbr,
                       a.segment,
                       count(distinct CASE WHEN a.analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' and rd.ord_event_key is not null THEN a.ord_event_key ELSE null END) AS trips_vmr_period,
                       count(distinct CASE WHEN a.prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.ord_event_key ELSE null END) AS trips_prior_period

                       FROM VMR_{brand_nm}_purch_details_all_trips a
                            INNER JOIN VMR_{brand_nm}_all_ids_new_seg_{analyst} n ON (a.cnsmr_id_key = n.cnsmridkey)
                           LEFT JOIN VMR_{brand_nm}_reward_events rd ON (rd.ord_event_key = a.ord_event_key)

                        GROUP BY 1,2,3
            DISTRIBUTE REPLICATE;

        ''')


        curs.execute(f'''       
            DROP table if exists VMR_{brand_nm}_new_seg_denom;
            CREATE temp table VMR_{brand_nm}_new_seg_denom as

               SELECT count(distinct cnsmr_id_key) as buyer_denom
               FROM VMR_{brand_nm}_new_buyer_flag 

            DISTRIBUTE REPLICATE;
        ''')


        curs.execute(f'''       
            DROP table if exists VMR_{brand_nm}_new_seg_numer;
            CREATE temp table VMR_{brand_nm}_new_seg_numer as

               SELECT segm_nbr, segment,
                      count(distinct cnsmr_id_key) as buyer_numer
               FROM VMR_{brand_nm}_new_buyer_flag
               where trips_prior_period=0 and trips_vmr_period>0
               GROUP BY 1,2
            DISTRIBUTE REPLICATE;
        ''')    

        new_seg_pct = pd.read_sql(f'''select a.segm_nbr, segment, buyer_numer as new_seg_buyer, 
                                      buyer_denom as ttl_new_buyer, new_seg_buyer::float / ttl_new_buyer::float as new_seg_pct
        from VMR_{brand_nm}_new_seg_numer a
        cross join VMR_{brand_nm}_new_seg_denom b
        order by 1''',conn)

        print(new_seg_pct)

    else:
        new_seg_pct = 0
    
        print(new_seg_pct)
    

In [89]:
if nbr_segments > 1 :
    
    if len(all_ids_new_seg) != 0:

        import matplotlib.pyplot as plt
        import matplotlib.dates as mdates
        import matplotlib.cbook as cbook
        import matplotlib as mpl
        from dateutil.relativedelta import relativedelta
        from matplotlib.dates import MO, TU, WE, TH, FR, SA, SU 
        import datetime


        pct_data_new_part = (new_seg_pct.new_seg_pct)*100
        segments_new_part =  new_seg_pct.segment

        #BAR CHART SETTINGS:

        fig,ax = plt.subplots()

        #if you want to reduce the width bar, you can prove other width value on the next sentence:
        bar = ax.bar(segments_new_part, pct_data_new_part, width = 0.5, color = 'salmon', label = '% of New segment buyers') #bar chart plotting

        for label in ax.get_xticklabels(which='major'): label.set(rotation=50, horizontalalignment='right')  #change X-axis data in 45 grades


        #X-Y AXIS LABELS AND LIMITS SETTINGS:
        ax.set_title('Decomp by segment for Shoppers who bought a new segment during the reward trip', color = '#1F2B44', weight = 'bold')
        ax.set_ylabel('% shoppers', weight = 'bold') #axis y_1 label
        ax.set_xlabel('Segment', weight = 'bold') #axis y_1 label
        plt.ylim(0, 100) 



        #GENERAL COLORS SETTINGS FOR AXIS AND TEXT:

        mpl.rcParams['text.color'] = '#1F2B44'
        ax.spines['bottom'].set_color('#1F2B44')
        ax.spines['top'].set_color('#1F2B44')
        ax.spines['left'].set_color('#1F2B44')
        ax.spines['right'].set_color('#1F2B44')

        ax.xaxis.label.set_color('#1F2B44')
        ax.yaxis.label.set_color('#1F2B44')
        ax.tick_params(axis='x', colors='#1F2B44')
        ax.tick_params(axis='y', colors='#1F2B44')



        #IMAGE SIZES AND OUTPUTS: Change if you need resize the chart output

        #if you want to change the chart size, you can change the dimensions in the next sentence:
        fig.set_size_inches(12, 5, forward=True) #image size
        ax.margins(y=0.3)

        plt.show()

## **11) PRE-52 WEEKS PROFILE (FOR CONSISTENT REWARD ID'S BASED ON DOLLARS):**

In [90]:

#Getting the consistent ID's who purchased the promoted brand during the recent 52 weeks before the reward print period.
#Getting #trips and dollars for both brand and categories

if dominance != 'Kroger':
    
    curs.execute(f'''

        DROP table if exists VMR_{brand_nm}_buyers_Pre52;
        CREATE temp table VMR_{brand_nm}_buyers_Pre52 as

            SELECT distinct o.ord_designated_cnsmr_id_key,
                   count(distinct CASE WHEN c.brand_nbr in ({brand_nbr_str}) THEN o.ord_event_key else null end) as b_trips,
                   count(distinct CASE WHEN c.brand_nbr in ({cat_nbr_str}) THEN o.ord_event_key else null end) as c_trips,
                   sum(CASE WHEN c.brand_nbr in ({brand_nbr_str}) THEN o.purch_amt else 0 end) as bdollars,
                   sum(CASE WHEN c.brand_nbr in ({cat_nbr_str}) THEN o.purch_amt else 0 end) as cdollars

             FROM ORD_TRD_ITM_CNSMR_FACT_NE_V o

                    INNER JOIN shopper_consistent_{analyst} s ON (o.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
                    INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                    INNER JOIN date_v b ON (o.ord_date_key = b.date_key)

             WHERE b.cal_dt BETWEEN '{prior_period_start}' and '{prior_period_end}' 
                   and purch_amt > 0
                   and purch_qty > 0

             GROUP BY 1


             DISTRIBUTE RANDOM; 

    ''')
    
else:
    
    curs.execute(f'''

        DROP table if exists VMR_{brand_nm}_buyers_Pre52;
        CREATE temp table VMR_{brand_nm}_buyers_Pre52 as

            SELECT distinct o.ord_designated_cnsmr_id_key,
                   count(distinct CASE WHEN c.brand_nbr in ({brand_nbr_str}) THEN o.ord_event_key else null end) as b_trips,
                   count(distinct CASE WHEN c.brand_nbr in ({cat_nbr_str}) THEN o.ord_event_key else null end) as c_trips,
                   sum(CASE WHEN c.brand_nbr in ({brand_nbr_str}) THEN o.purch_amt else 0 end) as bdollars,
                   sum(CASE WHEN c.brand_nbr in ({cat_nbr_str}) THEN o.purch_amt else 0 end) as cdollars

             FROM ORD_TRD_ITM_CNSMR_FACT_NE_V o

                    INNER JOIN shopper_consistent_{analyst} s ON (o.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
                    INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                    INNER JOIN date_v b ON (o.ord_date_key = b.date_key)

             WHERE b.cal_dt BETWEEN '{vmr_pre26wk_start}' and '{vmr_pre26wk_end}' 
                   and purch_amt > 0
                   and purch_qty > 0

             GROUP BY 1


             DISTRIBUTE RANDOM; 

    ''')


In [91]:
pd.read_sql(f''' SELECT COUNT(DISTINCT ord_designated_cnsmr_id_key) FROM VMR_{brand_nm}_buyers_Pre52''',conn)

,count
0,210449515


In [92]:
pre_52 = pd.read_sql(f''' SELECT * FROM VMR_{brand_nm}_buyers_Pre52 limit 10''',conn)
pre_52

,ord_designated_cnsmr_id_key,b_trips,c_trips,bdollars,cdollars
0,18673996700,2,2,5.78,5.78
1,434738774042,0,3,0.00,34.60
2,18714164216,1,1,7.49,14.98
3,434615211020,1,6,17.06,147.14
4,623238345,8,33,92.80,415.52
5,9472949903,26,53,340.35,1680.42
6,18704461110,0,2,0.00,39.54
7,416012972315,20,59,165.68,861.76
8,293518495528,3,11,34.57,130.46
9,312044087907,1,1,2.49,19.48


In [93]:
nbr_buyers_pre52 = pd.read_sql(f''' 
SELECT count(ord_designated_cnsmr_id_key)
FROM VMR_{brand_nm}_buyers_Pre52
WHERE b_trips > 0
''',conn)
print("Number of total promoted brand buyers during Pre-52 period: " + str(nbr_buyers_pre52['count'].values[0]))

Number of total promoted brand buyers during Pre-52 period: 94139076


#### For Brand Comsuption:

In [94]:
brand_dol50 = pd.read_sql(f''' SELECT percentile_cont(0.50) within group(order by bdollars) as pct FROM VMR_{brand_nm}_buyers_Pre52 where b_trips>1''',conn)
brand_dol50 = brand_dol50['pct'].values[0]
brand_dol75 = pd.read_sql(f''' SELECT percentile_cont(0.75) within group(order by bdollars) as pct FROM VMR_{brand_nm}_buyers_Pre52 where b_trips>1''',conn)
brand_dol75 = brand_dol75['pct'].values[0]
print("Brand Pct50: " + str(brand_dol50))
print("Brand Pct75: " + str(brand_dol75))

Brand Pct50: 44.46
Brand Pct75: 85.1


In [95]:
brand_consump = pd.read_sql(f''' 

SELECT CASE 
       WHEN b_trips=0 or  b.ord_designated_cnsmr_id_key is null THEN 'Never Buyer'
       WHEN b_trips=1 AND b.ord_designated_cnsmr_id_key is not null THEN '1x Buyer'
       WHEN b_trips>1 AND bdollars>0 AND bdollars<{brand_dol50} AND b.ord_designated_cnsmr_id_key is not null THEN 'Light Buyer (1%-50%)'
       WHEN b_trips>1 AND bdollars>={brand_dol50} AND bdollars<{brand_dol75} AND b.ord_designated_cnsmr_id_key is not null THEN 'Medium Buyer (50%-75%)'
       WHEN b_trips>1 AND bdollars>={brand_dol75} AND b.ord_designated_cnsmr_id_key is not null THEN 'Heavy Buyer (75%-100%)' END as brand_grp,
       COUNT(distinct a.ord_designated_cnsmr_id_key) as brand_ids
       
FROM VMR_{brand_nm}_reward_ids_csmr a
    LEFT JOIN VMR_{brand_nm}_buyers_Pre52 b ON (b.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
    --INNER JOIN shopper_consistent_{analyst} s ON (a.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
WHERE brand_grp is not null

GROUP BY 1
ORDER BY CASE WHEN brand_grp = 'Never Buyer' then 1
              WHEN brand_grp = '1x Buyer' then 2
              WHEN brand_grp = 'Light Buyer (1%-50%)' then 3
              WHEN brand_grp = 'Medium Buyer (50%-75%)' then 4
              WHEN brand_grp = 'Heavy Buyer (75%-100%)' then 5
              END

''',conn)

brand_consump['%_brand_ids'] = (brand_consump['brand_ids']/brand_consump['brand_ids'].sum())

brand_consump.loc[5] = brand_consump.sum()
brand_consump.at[5, 'brand_grp'] = 'Total'

brand_consump = brand_consump.rename(columns = {'brand_grp':'Brand Group', 'brand_ids':'Brand IDs', '%_brand_ids':'% Brand IDs'})

new_brand_buyers = "{:,}".format(brand_consump.loc[0,'Brand IDs'])

brand_consump

,Brand Group,Brand IDs,% Brand IDs
0,Never Buyer,297895,0.141869
1,1x Buyer,231847,0.110415
2,Light Buyer (1%-50%),208431,0.099263
3,Medium Buyer (50%-75%),281832,0.134219
4,Heavy Buyer (75%-100%),1079780,0.514234
5,Total,2099785,1.000000


In [96]:
curs.execute(f'''
    
        DROP table if exists VMR_{brand_nm}_brand_buyers_ids_{analyst}; 
        CREATE table VMR_{brand_nm}_brand_buyers_ids_{analyst} as
    
        SELECT a.ord_designated_cnsmr_id_key,
       CASE 
       WHEN b_trips=0 or  b.ord_designated_cnsmr_id_key is null THEN 'Never Buyer'
       WHEN b_trips=1 AND b.ord_designated_cnsmr_id_key is not null THEN '1x Buyer'
       WHEN b_trips>1 AND bdollars>0 AND bdollars<{brand_dol50} AND b.ord_designated_cnsmr_id_key is not null THEN 'Light'
       WHEN b_trips>1 AND bdollars>={brand_dol50} AND bdollars<{brand_dol75} AND b.ord_designated_cnsmr_id_key is not null THEN 'Medium'
       WHEN b_trips>1 AND bdollars>={brand_dol75} AND b.ord_designated_cnsmr_id_key is not null THEN 'Heavy' END as brand_grp
       
FROM VMR_{brand_nm}_reward_ids_csmr a
    LEFT JOIN VMR_{brand_nm}_buyers_Pre52 b ON (b.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
    --INNER JOIN shopper_consistent_{analyst} s ON (a.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
WHERE brand_grp is not null
GROUP BY 1,2
ORDER BY CASE WHEN brand_grp = 'Never Buyer' then 1
              WHEN brand_grp = '1x Buyer' then 2
              WHEN brand_grp = 'Light' then 3
              WHEN brand_grp = 'Medium' then 4
              WHEN brand_grp = 'Heavy' then 5
              END

        DISTRIBUTE RANDOM;
    
    
''')
    

#### For Category Comsuption:

In [97]:
cat_dol50 = pd.read_sql(f''' SELECT percentile_cont(0.50) within group(order by cdollars) as pct FROM VMR_{brand_nm}_buyers_Pre52 WHERE c_trips>1 ''',conn)
cat_dol50 = cat_dol50['pct'].values[0]
cat_dol75 = pd.read_sql(f''' SELECT percentile_cont(0.75) within group(order by cdollars) as pct FROM VMR_{brand_nm}_buyers_Pre52 WHERE c_trips>1''',conn)
cat_dol75 = cat_dol75['pct'].values[0]
print("Cat Pct50: " + str(cat_dol50))
print("Cat Pct75: " + str(cat_dol75))

Cat Pct50: 60.91
Cat Pct75: 136.15


In [98]:
cat_consump = pd.read_sql(f''' 

SELECT CASE 
       WHEN b.ord_designated_cnsmr_id_key is null THEN 'Never Buyer'
       WHEN c_trips=1 AND b.ord_designated_cnsmr_id_key is not null THEN '1x Buyer'
       WHEN c_trips>1 AND cdollars>0 AND cdollars<{cat_dol50} AND b.ord_designated_cnsmr_id_key is not null THEN 'Light Buyer (1%-50%)'
       WHEN c_trips>1 AND cdollars>={cat_dol50} AND cdollars<{cat_dol75} AND b.ord_designated_cnsmr_id_key is not null THEN 'Medium Buyer (50%-75%)'
       WHEN c_trips>1 AND cdollars>={cat_dol75} AND b.ord_designated_cnsmr_id_key is not null THEN 'Heavy Buyer (75%-100%)' END as category_grp,
       COUNT(distinct a.ord_designated_cnsmr_id_key) as category_ids
       
FROM VMR_{brand_nm}_reward_ids_csmr a
    LEFT JOIN VMR_{brand_nm}_buyers_Pre52 b ON (b.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
    INNER JOIN shopper_consistent_{analyst} s ON (a.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
WHERE category_grp is not null
GROUP BY 1
ORDER BY CASE WHEN category_grp = 'Never Buyer' then 1
              WHEN category_grp = '1x Buyer' then 2
              WHEN category_grp = 'Light Buyer (1%-50%)' then 3
              WHEN category_grp = 'Medium Buyer (50%-75%)' then 4
              WHEN category_grp = 'Heavy Buyer (75%-100%)' then 5
              END
 

''',conn)

cat_consump['%_category_ids'] = (cat_consump['category_ids']/cat_consump['category_ids'].sum())

cat_consump.loc[5] = cat_consump.sum()
cat_consump.at[5, 'category_grp'] = 'Total'

cat_consump = cat_consump.rename(columns = {'category_grp':'Category Group', 'category_ids':'Category IDs', '%_category_ids':'% Category IDs'})

cat_consump

,Category Group,Category IDs,% Category IDs
0,Never Buyer,161850,0.077085
1,1x Buyer,135821,0.064688
2,Light Buyer (1%-50%),204331,0.097318
3,Medium Buyer (50%-75%),337396,0.160694
4,Heavy Buyer (75%-100%),1260224,0.600215
5,Total,2099622,1.000000


## **12) GETTING REPURCHASE RATE FOR VMR PARTICIPANTS:**

In [99]:

curs.execute(f'''

    DROP table if exists VMR_{brand_nm}_repurch_details;
    CREATE temp table VMR_{brand_nm}_repurch_details as
        
          SELECT d.ord_designated_cnsmr_id_key,
                 e.brand_grp as brand_group,
                 min(d.reward_date) as reward_date
          
          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                 INNER JOIN VMR_{brand_nm}_reward_ids d ON (o.ord_designated_cnsmr_id_key = d.ord_designated_cnsmr_id_key)
                 INNER JOIN VMR_{brand_nm}_brand_buyers_ids_{analyst} e ON (o.ord_designated_cnsmr_id_key = e.ord_designated_cnsmr_id_key)
                 INNER JOIN shopper_consistent_{analyst} s ON (o.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
                 
          WHERE o.purch_amt > 0
                and o.purch_qty > 0
                and c.brand_nbr in ({brand_nbr_str})
                --and b.cal_dt > '{reward_print_start}'
                and b.cal_dt > reward_date               --Repurch date after the printing date
                and b.cal_dt between '{reward_print_start}' and '{post_4wk_end}'  --Repurch including N weeks after the reward purchase
        
        GROUP BY 1,2
        
        DISTRIBUTE ON (ord_designated_cnsmr_id_key);
        
''')

In [100]:
pd.read_sql(f'''
    SELECT *
    FROM VMR_{brand_nm}_repurch_details
    ''',conn)

,ord_designated_cnsmr_id_key,brand_group,reward_date
0,18696650503,Heavy,2026-03-17
1,24426728201,Heavy,2026-03-15
2,18668242363,Heavy,2026-03-19
3,107333456245,Heavy,2026-03-08
4,9479989068,Heavy,2026-03-10
...,...,...,...
778688,18675393709,Heavy,2026-03-04
778689,18659363688,Heavy,2026-03-26
778690,413553088917,Never Buyer,2026-03-25
778691,18699122145,Heavy,2026-03-26


In [101]:
pd.read_sql(f'''
    SELECT min(reward_date), max(reward_date)
    FROM VMR_{brand_nm}_repurch_details
    ''',conn)

,min,max
0,2026-02-28,2026-03-29


In [102]:
repurch_total_buyers = pd.read_sql(f'''
    SELECT count(distinct ord_designated_cnsmr_id_key)
    FROM VMR_{brand_nm}_repurch_details
    ''',conn)

repurch_total_buyers = repurch_total_buyers['count'].values[0]
print(f'''The total of VMR participants that repurchased during the post {weeks_per_period_final} weeks is: ''' + str(repurch_total_buyers))

total_reward_ids_m = pd.read_sql(f'''
    SELECT count(distinct ord_designated_cnsmr_id_key)
    FROM VMR_{brand_nm}_brand_buyers_ids_{analyst} a
    INNER JOIN shopper_consistent_{analyst} s ON (a.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
    ''',conn)

pct_repurch = (repurch_total_buyers/total_reward_ids_m['count'].values[0])
print("The repurchase percentage among all the VMR participants is: " + str(round(pct_repurch*100)))

The total of VMR participants that repurchased during the post 4.3 weeks is: 778693
The repurchase percentage among all the VMR participants is: 37


In [103]:
repurch_new_buyers = pd.read_sql(f'''
    SELECT count(distinct ord_designated_cnsmr_id_key)
    FROM VMR_{brand_nm}_repurch_details
    WHERE brand_group = 'Never Buyer'
    ''',conn)

total_new_buy = pd.read_sql(f'''
    SELECT count(distinct ord_designated_cnsmr_id_key)
    FROM VMR_{brand_nm}_brand_buyers_ids_{analyst} a
    INNER JOIN shopper_consistent_{analyst} s ON (a.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
    WHERE brand_grp = 'Never Buyer'
    ''',conn)

repurch_new_buyers = repurch_new_buyers['count'].values[0]
print(f'''The New Buyers participants that repurchased during the post {weeks_per_period_final} weeks is: ''' + str(repurch_new_buyers))

pct_repurch_new_buyers = repurch_new_buyers/total_new_buy['count'].values[0]
print("The repurchase percentage for New Buyers participants is: " + str(round(pct_repurch_new_buyers*100)))

The New Buyers participants that repurchased during the post 4.3 weeks is: 40579
The repurchase percentage for New Buyers participants is: 14


In [104]:
repurch_existing_buyers = pd.read_sql(f'''
    SELECT count(distinct ord_designated_cnsmr_id_key)
    FROM VMR_{brand_nm}_repurch_details
    WHERE brand_group != 'Never Buyer'
    ''',conn)

total_exist_buy = pd.read_sql(f'''
    SELECT count(distinct ord_designated_cnsmr_id_key)
    FROM VMR_{brand_nm}_brand_buyers_ids_{analyst} a
    INNER JOIN shopper_consistent_{analyst} s ON (a.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
    WHERE brand_grp != 'Never Buyer'
    ''',conn)

repurch_existing_buyers = repurch_existing_buyers['count'].values[0]
print(f'''The Existing participants that repurchased during the post {weeks_per_period_final} weeks is: ''' + str(repurch_existing_buyers))

pct_repurch_existing_buyers = repurch_existing_buyers/total_exist_buy['count'].values[0]
print("The repurchase percentage among the existing participants is: " + str(round(pct_repurch_existing_buyers*100)))

The Existing participants that repurchased during the post 4.3 weeks is: 738114
The repurchase percentage among the existing participants is: 41


## **13) GETTING DATA FOR CATEGORY SHARE SLIDE:**

Getting category shares metrics for campaign participants

In [105]:
curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_vmr_category_shade;
    CREATE temp table VMR_{brand_nm}_vmr_category_shade as
    
        SELECT  'VMR Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids f ON (a.ord_designated_cnsmr_id_key = f.ord_designated_cnsmr_id_key)

            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{reward_print_start}' and '{reward_print_end}'
              
        UNION
        
        SELECT  'Pre52wk Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
        
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids f ON (a.ord_designated_cnsmr_id_key = f.ord_designated_cnsmr_id_key)

            
        WHERE prior_period = 'Prior-period ({round(pre_weeks)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{prior_period_start}' and '{prior_period_end}'
              
              
        UNION
        
        SELECT  'Post4wk Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids f ON (a.ord_designated_cnsmr_id_key = f.ord_designated_cnsmr_id_key)

            
        WHERE post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{post_4wk_start}' and '{post_4wk_end}'
              
        UNION
        
        
        SELECT  'YAGO Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
                
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids f ON (a.ord_designated_cnsmr_id_key = f.ord_designated_cnsmr_id_key)

            
        WHERE analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' AND
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{yago_print_start}' and '{yago_print_end}'
        
    DISTRIBUTE REPLICATE;

''')


In [106]:
curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_vmr_category_shade;
    CREATE temp table VMR_{brand_nm}_vmr_category_shade as
    
        SELECT  'VMR Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(a.purch_amt) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(a.purch_qty) as units_category
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids f ON (a.ord_designated_cnsmr_id_key = f.ord_designated_cnsmr_id_key)

            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{reward_print_start}' and '{reward_print_end}'
              
        UNION
        
        SELECT  'Pre52wk Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(a.purch_amt) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(a.purch_qty) as units_category
        
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids f ON (a.ord_designated_cnsmr_id_key = f.ord_designated_cnsmr_id_key)

            
        WHERE prior_period = 'Prior-period ({round(pre_weeks)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{prior_period_start}' and '{prior_period_end}'
              
              
        UNION
        
        SELECT  'Post Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(a.purch_amt) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(a.purch_qty) as units_category
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids f ON (a.ord_designated_cnsmr_id_key = f.ord_designated_cnsmr_id_key)

            
        WHERE post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{post_4wk_start}' and '{post_4wk_end}'
              
        UNION
        
        
        SELECT  'YAGO Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(a.purch_amt) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(a.purch_qty) as units_category
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids f ON (a.ord_designated_cnsmr_id_key = f.ord_designated_cnsmr_id_key)

            
        WHERE analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' AND
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{yago_print_start}' and '{yago_print_end}'
        
    DISTRIBUTE REPLICATE;

''')


In [107]:
share_category_data = pd.read_sql(f'''
    SELECT *
    FROM VMR_{brand_nm}_vmr_category_shade
    ''',conn)
share_category_data 

,period,dollars_promoted,dollars_category,units_promoted,units_category
0,Pre52wk Period,4.306571e+08,8.294306e+08,45059617,91007618
1,YAGO Period,3.446352e+07,6.589144e+07,3558192,7094917
2,VMR Period,1.409887e+08,1.948251e+08,13526778,19904921
3,Post Period,1.945300e+07,3.706941e+07,2070235,4115938


In [108]:
dollars_promoted_yago = share_category_data.loc[share_category_data['period']=='YAGO Period', 'dollars_promoted'].values[0]
dollars_category_yago = share_category_data.loc[share_category_data['period']=='YAGO Period', 'dollars_category'].values[0]
units_promoted_yago = share_category_data.loc[share_category_data['period']=='YAGO Period', 'units_promoted'].values[0]
units_category_yago = share_category_data.loc[share_category_data['period']=='YAGO Period', 'units_category'].values[0]

dollars_promoted_vmr = share_category_data.loc[share_category_data['period']=='VMR Period', 'dollars_promoted'].values[0]
dollars_category_vmr = share_category_data.loc[share_category_data['period']=='VMR Period', 'dollars_category'].values[0]
units_promoted_vmr = share_category_data.loc[share_category_data['period']=='VMR Period', 'units_promoted'].values[0]
units_category_vmr = share_category_data.loc[share_category_data['period']=='VMR Period', 'units_category'].values[0]

dollars_promoted_pre52 = share_category_data.loc[share_category_data['period']=='Pre52wk Period', 'dollars_promoted'].values[0]
dollars_category_pre52 = share_category_data.loc[share_category_data['period']=='Pre52wk Period', 'dollars_category'].values[0]
units_promoted_pre52 = share_category_data.loc[share_category_data['period']=='Pre52wk Period', 'units_promoted'].values[0]
units_category_pre52 = share_category_data.loc[share_category_data['period']=='Pre52wk Period', 'units_category'].values[0]

dollars_promoted_post = share_category_data.loc[share_category_data['period']=='Post Period', 'dollars_promoted'].values[0]
dollars_category_post = share_category_data.loc[share_category_data['period']=='Post Period', 'dollars_category'].values[0]
units_promoted_post = share_category_data.loc[share_category_data['period']=='Post Period', 'units_promoted'].values[0]
units_category_post = share_category_data.loc[share_category_data['period']=='Post Period', 'units_category'].values[0]


In [109]:
share_dollar_vmr = dollars_promoted_vmr/dollars_category_vmr
share_dollar_vmr = 0 if np.isnan(share_dollar_vmr) else share_dollar_vmr
print("share_dollar_vmr: " + str(share_dollar_vmr))
share_dollar_pre52 = dollars_promoted_pre52/dollars_category_pre52
share_dollar_pre52 = 0 if np.isnan(share_dollar_pre52) else share_dollar_pre52
print("share_dollar_pre52: " + str(share_dollar_pre52))
share_dollar_post_4wk = dollars_promoted_post/dollars_category_post
share_dollar_post_4wk = 0 if np.isnan(share_dollar_post_4wk) else share_dollar_post_4wk
print("share_dollar_post_4wk: " + str(share_dollar_post_4wk))
share_dollar_yago = dollars_promoted_yago/dollars_category_yago
share_dollar_yago = 0 if np.isnan(share_dollar_yago) else share_dollar_yago
print("share_dollar_yago: " + str(share_dollar_yago))

share_units_vmr = units_promoted_vmr/units_category_vmr
share_units_vmr = 0 if np.isnan(share_units_vmr) else share_units_vmr
print("share_units_vmr: " + str(share_units_vmr))
share_units_pre52 = units_promoted_pre52/units_category_pre52
share_units_pre52 = 0 if np.isnan(share_units_pre52) else share_units_pre52
print("share_units_pre52: " + str(share_units_pre52))
share_units_post_4wk = units_promoted_post/units_category_post
share_units_post_4wk = 0 if np.isnan(share_units_post_4wk) else share_units_post_4wk
print("share_units_post_4wk: " + str(share_units_post_4wk))
share_units_yago = units_promoted_yago/units_category_yago
share_units_yago = 0 if np.isnan(share_units_yago) else share_units_yago
print("share_units_yago: " + str(share_units_yago))


share_dollar_vmr: 0.7236680930566717
share_dollar_pre52: 0.519220221609446
share_dollar_post_4wk: 0.5247723190475978
share_dollar_yago: 0.5230349252923381
share_units_vmr: 0.6795695396128425
share_units_pre52: 0.4951191778253113
share_units_post_4wk: 0.5029801226354722
share_units_yago: 0.5015128436315746


Getting category shares metrics for all IDs:

In [110]:
curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_vmr_category_shade_all;
    CREATE temp table VMR_{brand_nm}_vmr_category_shade_all as
    
        SELECT  'VMR Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)

            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{reward_print_start}' and '{reward_print_end}'
              
        UNION
        
        SELECT  'Pre52wk Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
        
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)

            
        WHERE prior_period = 'Prior-period ({round(pre_weeks)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{prior_period_start}' and '{prior_period_end}'
              
              
        UNION
        
        SELECT  'Post Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)

            
        WHERE post_period = 'Post Period ({round(weeks_per_period_final)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{post_4wk_start}' and '{post_4wk_end}'
              
        UNION
        
        
        SELECT  'YAGO Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
                
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)

            
        WHERE analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' AND
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{yago_print_start}' and '{yago_print_end}'
                    
              
        UNION
        
        
        SELECT  'Prior 26wk' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(a.purch_amt) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(a.purch_qty) as units_category
                
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)

            
        WHERE a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{vmr_pre26wk_start}' and '{vmr_pre26wk_end}'
              
              
              
        UNION
        
        
         
        SELECT  'Pre-Period' as Period,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_promoted,
                SUM(CASE WHEN analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN a.purch_amt ELSE 0 END) as dollars_category,
                SUM(CASE WHEN brand_nbr IN ({brand_nbr_str}) and analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_promoted,
                SUM(CASE WHEN analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN a.purch_qty ELSE 0 END) as units_category
                
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)

            
        WHERE analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' and
              a.purch_amt > 0
              and a.purch_qty > 0 and
              cal_dt between '{vmr_pre_period_start}' and '{vmr_pre_period_end}' 
        
        
    DISTRIBUTE REPLICATE;
''')


In [111]:
share_category_data_all = pd.read_sql(f'''
    SELECT *
    FROM VMR_{brand_nm}_vmr_category_shade_all
    ''',conn)
share_category_data_all

,period,dollars_promoted,dollars_category,units_promoted,units_category
0,VMR Period,2.507479e+08,6.501556e+08,26899195,67996790
1,YAGO Period,2.390810e+08,6.666203e+08,26092326,70677966
2,Pre-Period,2.354315e+08,6.333732e+08,25094079,67275061
3,Pre52wk Period,2.797224e+09,7.734050e+09,307586002,822540743
4,Post Period,1.161383e+08,3.062998e+08,12414449,31769455
5,Prior 26wk,1.355303e+09,3.769791e+09,148362140,400888883


In [112]:
dollars_promoted_vmr_all = share_category_data_all.loc[share_category_data_all['period']=='VMR Period', 'dollars_promoted'].values[0]
dollars_category_vmr_all = share_category_data_all.loc[share_category_data_all['period']=='VMR Period', 'dollars_category'].values[0]
units_promoted_vmr_all = share_category_data_all.loc[share_category_data_all['period']=='VMR Period', 'units_promoted'].values[0]
units_category_vmr_all = share_category_data_all.loc[share_category_data_all['period']=='VMR Period', 'units_category'].values[0]

dollars_promoted_yago_all = share_category_data_all.loc[share_category_data_all['period']=='YAGO Period', 'dollars_promoted'].values[0]
dollars_category_yago_all = share_category_data_all.loc[share_category_data_all['period']=='YAGO Period', 'dollars_category'].values[0]
units_promoted_yago_all = share_category_data_all.loc[share_category_data_all['period']=='YAGO Period', 'units_promoted'].values[0]
units_category_yago_all = share_category_data_all.loc[share_category_data_all['period']=='YAGO Period', 'units_category'].values[0]

dollars_promoted_26wk_all = share_category_data_all.loc[share_category_data_all['period']=='Prior 26wk', 'dollars_promoted'].values[0]
dollars_category_26wk_all = share_category_data_all.loc[share_category_data_all['period']=='Prior 26wk', 'dollars_category'].values[0]
units_promoted_26wk_all = share_category_data_all.loc[share_category_data_all['period']=='Prior 26wk', 'units_promoted'].values[0]
units_category_26wk_all = share_category_data_all.loc[share_category_data_all['period']=='Prior 26wk', 'units_category'].values[0]

dollars_promoted_prep_all = share_category_data_all.loc[share_category_data_all['period']=='Pre-Period', 'dollars_promoted'].values[0]
dollars_category_prep_all = share_category_data_all.loc[share_category_data_all['period']=='Pre-Period', 'dollars_category'].values[0]
units_promoted_prep_all = share_category_data_all.loc[share_category_data_all['period']=='Pre-Period', 'units_promoted'].values[0]
units_category_prep_all = share_category_data_all.loc[share_category_data_all['period']=='Pre-Period', 'units_category'].values[0]


share_vmr_dol = dollars_promoted_vmr_all/dollars_category_vmr_all
share_vmr_dol = 0 if np.isnan(share_vmr_dol) else share_vmr_dol
share_yago_dol = dollars_promoted_yago_all/dollars_category_yago_all
share_yago_dol = 0 if np.isnan(share_yago_dol) else share_yago_dol
share_26wk_dol = dollars_promoted_26wk_all/dollars_category_26wk_all
share_26wk_dol = 0 if np.isnan(share_26wk_dol) else share_26wk_dol
share_prep_dol = dollars_promoted_prep_all/dollars_category_prep_all
share_prep_dol = 0 if np.isnan(share_prep_dol) else share_prep_dol

share_vmr_unit = units_promoted_vmr_all/units_category_vmr_all
share_vmr_unit = 0 if np.isnan(share_vmr_unit) else share_vmr_unit
share_yago_unit = units_promoted_yago_all/units_category_yago_all
share_yago_unit = 0 if np.isnan(share_yago_unit) else share_yago_unit
share_26wk_unit = units_promoted_26wk_all/units_category_26wk_all
share_26wk_unit = 0 if np.isnan(share_26wk_unit) else share_26wk_unit
share_prep_unit = units_promoted_prep_all/units_category_prep_all
share_prep_unit = 0 if np.isnan(share_prep_unit) else share_prep_unit


In [113]:
share_vmr_dol

0.38567359601523526

In [114]:
share_yago_unit

0.36917199909233384

In [115]:
share_26wk_dol

0.3595167467982936

In [116]:
dollars_category_yago_all

666620324.43

In [117]:
share_prep_dol

0.37171051846569475

In [118]:
share_prep_unit

0.3730071534234655

## **14) GETTING DATA FOR PRIOR 52 WEEKS (PER # VMR WEEKS) AND VMR CAMPAIGN PARTICIPANTS SLIDE:**

In [119]:
vmr_length = (reward_print_end - reward_print_start).days
vmr_length

29

In [120]:
curs.execute(f'''

    DROP table if exists VMR_{brand_nm}_prior_table_shares;
    CREATE temp table VMR_{brand_nm}_prior_table_shares as
        
          SELECT CASE WHEN prior_period = 'Prior-period ({round(pre_weeks)} weeks)' THEN 'Prior 52 wks (per {vmr_length} days)'
                     ELSE 'NA' END as Time_Period,
          
                COUNT(distinct o.ord_designated_cnsmr_id_key) as campaign_buyers,
                COUNT(distinct o.ord_event_key) as campaign_trips,
                SUM(o.purch_amt) as campaign_dollars,
                SUM(o.purch_qty) as campaign_units,
                
                (campaign_dollars:: float/campaign_buyers:: float) :: float as dollars_per_buyer_prior,
                (campaign_units:: float/campaign_buyers:: float) :: float as units_per_buyer_prior,
                (campaign_trips:: float/campaign_buyers:: float) :: float as trips_per_buyer_prior,
                (campaign_dollars:: float/campaign_trips:: float) :: float as dollars_per_trip_prior,
                (campaign_units:: float/campaign_trips:: float) :: float as units_per_trip_prior,
                
                round(((dollars_per_buyer_prior)/364)*{vmr_length} :: float, 4) as dollars_per_buyer,
                round(((units_per_buyer_prior)/364)*{vmr_length} ::float, 4) as units_per_buyer,
                round(((trips_per_buyer_prior)/364)*{vmr_length} :: float, 4) as trips_per_buyer,
                round((dollars_per_trip_prior) :: float, 4) as dollars_per_trip,
                round((units_per_trip_prior) :: float, 4) as units_per_trip
                
                
          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)                 
                
          WHERE o.purch_amt > 0
                and o.purch_qty > 0
                and prior_period = 'Prior-period ({round(pre_weeks)} weeks)' and prior_period != 'NA'
                and c.brand_nbr in ({brand_nbr_str})
                and cal_dt between '{prior_period_start}' and '{prior_period_end}'
                
        GROUP BY 1 
        
        DISTRIBUTE REPLICATE;
        
''')


curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_vmr_metrics_participants;
    CREATE temp table VMR_{brand_nm}_vmr_metrics_participants as
    
        SELECT  
                CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'Campaign Period'
                     ELSE 'NA' END as Time_Period,
        
                COUNT(distinct a.ord_designated_cnsmr_id_key) as campaign_buyers,
                COUNT(distinct a.ord_event_key) as campaign_trips,
                SUM(a.purch_amt) as campaign_dollars,
                SUM(a.purch_qty) as campaign_units,
                
                round((campaign_dollars::float/campaign_buyers::float),4) as dollars_per_buyer,
                round((campaign_units::float/campaign_buyers::float),4) as units_per_buyer,
                round((campaign_trips::float/campaign_buyers::float),4) as trips_per_buyer,
                round((campaign_dollars::float/campaign_trips::float),4) as dollars_per_trip,
                round((campaign_units::float/campaign_trips::float),4) as units_per_trip
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            --INNER JOIN shopper_consistent_{analyst} v ON (a.ord_designated_cnsmr_id_key = v.cnsmr_id_key)
            
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str})
              and Time_Period != 'NA'
              
        GROUP BY 1
        
   
        UNION
        
        
        SELECT  
                Time_Period,
        
                campaign_buyers,
                campaign_trips,
                campaign_dollars,
                campaign_units,
                
                dollars_per_buyer,
                units_per_buyer,
                trips_per_buyer,
                dollars_per_trip,
                units_per_trip
                
        FROM VMR_{brand_nm}_prior_table_shares
        
        
    DISTRIBUTE REPLICATE;

''')


In [121]:
results_participants = pd.read_sql(f'''
    SELECT *
    FROM VMR_{brand_nm}_vmr_metrics_participants
    ORDER BY CASE WHEN Time_Period = 'Campaign Period' then 1
              WHEN Time_Period = 'Prior 52 wks (per {vmr_length} days)' then 2
              END

    ''',conn)

results_participants

,time_period,campaign_buyers,campaign_trips,campaign_dollars,campaign_units,dollars_per_buyer,units_per_buyer,trips_per_buyer,dollars_per_trip,units_per_trip
0,Campaign Period,2099785,3265956,1.104637e+08,10388473,52.6072,4.9474,1.5554,33.8228,3.1808
1,Prior 52 wks (per 29 days),1802002,14880268,3.377358e+08,34962146,14.9320,1.5458,0.6579,22.6969,2.3496


In [122]:
results_participants = results_participants[['time_period', 'campaign_buyers', 'campaign_dollars', 'campaign_units','dollars_per_buyer', 'units_per_buyer', 'trips_per_buyer', 'dollars_per_trip', 'units_per_trip']]
results_participants = results_participants.rename(columns = {'time_period':'Time Period','campaign_buyers': 'Buyers','campaign_dollars':'Brand Dollars', 'campaign_units': 'Brand Units','dollars_per_buyer':'Dollars per Buyer', 'units_per_buyer':'Units per Buyer', 'trips_per_buyer':'Trips per Buyer', 'dollars_per_trip':'Dollars per Trip', 'units_per_trip':'Units per Trip'})

change_by = (results_participants['Buyers'][0] - results_participants['Buyers'][1])/results_participants['Buyers'][1]
change_ds = (results_participants['Brand Dollars'][0] - results_participants['Brand Dollars'][1])/results_participants['Brand Dollars'][1]
change_us = (results_participants['Brand Units'][0] - results_participants['Brand Units'][1])/results_participants['Brand Units'][1]

change_dpb = (results_participants['Dollars per Buyer'][0] - results_participants['Dollars per Buyer'][1])/results_participants['Dollars per Buyer'][1]
change_upb = (results_participants['Units per Buyer'][0] - results_participants['Units per Buyer'][1])/results_participants['Units per Buyer'][1]
change_tpb = (results_participants['Trips per Buyer'][0] - results_participants['Trips per Buyer'][1])/results_participants['Trips per Buyer'][1]
change_dpt = (results_participants['Dollars per Trip'][0] - results_participants['Dollars per Trip'][1])/results_participants['Dollars per Trip'][1]
change_upt = (results_participants['Units per Trip'][0] - results_participants['Units per Trip'][1])/results_participants['Units per Trip'][1]

if str(change_by) == 'inf':
    change_by = ''
if str(change_ds) == 'inf':
    change_ds = ''
if str(change_us) == 'inf':
    change_us = ''

if str(change_dpb) == 'inf':
    change_dpb = ''
if str(change_upb) == 'inf':
    change_upb = ''
if str(change_tpb) == 'inf':
    change_tpb = ''
if str(change_dpt) == 'inf':
    change_dpt = ''
if str(change_upt) == 'inf':
    change_upt = ''

    
new_row = pd.DataFrame({'Time Period' : ['% Change'], 'Buyers': [change_by], 'Brand Dollars': [change_ds], 'Brand Units':[change_us],'Dollars per Buyer': [change_dpb], 'Units per Buyer':[change_upb], 'Trips per Buyer': [change_tpb], 'Dollars per Trip': [change_dpt], 'Units per Trip': [change_upt]})
results_participants = pd.concat([results_participants, new_row], ignore_index=True)
results_participants = results_participants.T.reset_index().T.reset_index(drop=True)
results_participants_1 = results_participants
results_participants_01 = results_participants_1 
results_participants_01

,0,1,2,3,4,5,6,7,8
0,Time Period,Buyers,Brand Dollars,Brand Units,Dollars per Buyer,Units per Buyer,Trips per Buyer,Dollars per Trip,Units per Trip
1,Campaign Period,2099785.0,110463736.52,10388473.0,52.6072,4.9474,1.5554,33.8228,3.1808
2,Prior 52 wks (per 29 days),1802002.0,337735802.35,34962146.0,14.932,1.5458,0.6579,22.6969,2.3496
3,% Change,0.165251,-0.672929,-0.702865,2.523118,2.200543,1.364189,0.490195,0.353762


In [123]:
curs.execute(f'''

    DROP table if exists VMR_{brand_nm}_prior_table_shares;
    CREATE temp table VMR_{brand_nm}_prior_table_shares as
        
          SELECT CASE WHEN analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)' THEN 'VMR Pre Period ({round(weeks_per_period_final)} weeks)'
                     ELSE 'NA' END as Time_Period,
                     
                     min(cal_dt) as min,
                     max(cal_dt) as max,
          
                COUNT(distinct o.ord_designated_cnsmr_id_key) as campaign_buyers,
                COUNT(distinct o.ord_event_key) as campaign_trips,
                SUM(o.purch_amt) as campaign_dollars,
                SUM(o.purch_qty) as campaign_units,
                
                round((campaign_dollars::float/campaign_buyers::float),4) as dollars_per_buyer,
                round((campaign_units::float/campaign_buyers::float),4) as units_per_buyer,
                round((campaign_trips::float/campaign_buyers::float),4) as trips_per_buyer,
                round((campaign_dollars::float/campaign_trips::float),4) as dollars_per_trip,
                round((campaign_units::float/campaign_trips::float),4) as units_per_trip
                
                
          FROM ord_trd_itm_cnsmr_fact_ne_v o
                 
                 INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                 INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)
                --INNER JOIN shopper_consistent_{analyst} v ON (o.ord_designated_cnsmr_id_key = v.cnsmr_id_key)
                
          WHERE o.purch_amt > 0
                and o.purch_qty > 0
                and analysis_period = 'VMR Pre-period ({round(weeks_per_period_final)} weeks)'
                and c.brand_nbr in ({brand_nbr_str})
                and Time_Period != 'NA'

                
        GROUP BY 1 
        
        DISTRIBUTE REPLICATE;
        
''')


curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_vmr_metrics_participants;
    CREATE temp table VMR_{brand_nm}_vmr_metrics_participants as
    
        SELECT  
                CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'Campaign Period'
                     ELSE 'NA' END as Time_Period,
                     
                     
                     min(cal_dt) as min,
                     max(cal_dt) as max,
        
                COUNT(distinct a.ord_designated_cnsmr_id_key) as campaign_buyers,
                COUNT(distinct a.ord_event_key) as campaign_trips,
                SUM(a.purch_amt) as campaign_dollars,
                SUM(a.purch_qty) as campaign_units,
                
                round((campaign_dollars::float/campaign_buyers::float),4) as dollars_per_buyer,
                round((campaign_units::float/campaign_buyers::float),4) as units_per_buyer,
                round((campaign_trips::float/campaign_buyers::float),4) as trips_per_buyer,
                round((campaign_dollars::float/campaign_trips::float),4) as dollars_per_trip,
                round((campaign_units::float/campaign_trips::float),4) as units_per_trip
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            --INNER JOIN shopper_consistent_{analyst} v ON (a.ord_designated_cnsmr_id_key = v.cnsmr_id_key)
            
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str})
              and Time_Period != 'NA'
              
        GROUP BY 1
        
   
        UNION
        
        
        SELECT  
                Time_Period,
                
                min,
                max,
        
                campaign_buyers,
                campaign_trips,
                campaign_dollars,
                campaign_units,
                
                dollars_per_buyer,
                units_per_buyer,
                trips_per_buyer,
                dollars_per_trip,
                units_per_trip
                
        FROM VMR_{brand_nm}_prior_table_shares
        
        
    DISTRIBUTE REPLICATE;

''')


In [124]:
results_participants = pd.read_sql(f'''
    SELECT *
    FROM VMR_{brand_nm}_vmr_metrics_participants
    ORDER BY CASE WHEN Time_Period = 'Campaign Period' then 1
              END

    ''',conn)

results_participants

,time_period,min,max,campaign_buyers,campaign_trips,campaign_dollars,campaign_units,dollars_per_buyer,units_per_buyer,trips_per_buyer,dollars_per_trip,units_per_trip
0,Campaign Period,2026-02-28,2026-03-29,2099785,3265956,1.104637e+08,10388473,52.6072,4.9474,1.5554,33.8228,3.1808
1,VMR Pre Period (4 weeks),2026-01-29,2026-02-27,793558,1327033,3.040137e+07,3116936,38.3102,3.9278,1.6723,22.9093,2.3488


ADD THIS IN THE NEXT

if len(results_participants[results_participants['time_period']==f'VMR Pre Period ({round(weeks_per_period_final)} weeks)']) == 0 :
    
    print('k')

In [125]:
if len(results_participants[results_participants['time_period']==f'VMR Pre Period ({round(weeks_per_period_final)} weeks)']) != 0 :


    results_participants = results_participants[['time_period','campaign_buyers', 'campaign_dollars', 'campaign_units','dollars_per_buyer', 'units_per_buyer', 'trips_per_buyer', 'dollars_per_trip', 'units_per_trip']]
    results_participants = results_participants.rename(columns = {'time_period':'Time Period', 'campaign_buyers': 'Buyers','campaign_dollars':'Brand Dollars', 'campaign_units': 'Brand Units','dollars_per_buyer':'Dollars per Buyer', 'units_per_buyer':'Units per Buyer', 'trips_per_buyer':'Trips per Buyer', 'dollars_per_trip':'Dollars per Trip', 'units_per_trip':'Units per Trip'})

    change_by = (results_participants['Buyers'][0] - results_participants['Buyers'][1])/results_participants['Buyers'][1]
    change_ds = (results_participants['Brand Dollars'][0] - results_participants['Brand Dollars'][1])/results_participants['Brand Dollars'][1]
    change_us = (results_participants['Brand Units'][0] - results_participants['Brand Units'][1])/results_participants['Brand Units'][1]

    change_dpb = (results_participants['Dollars per Buyer'][0] - results_participants['Dollars per Buyer'][1])/results_participants['Dollars per Buyer'][1]
    change_upb = (results_participants['Units per Buyer'][0] - results_participants['Units per Buyer'][1])/results_participants['Units per Buyer'][1]
    change_tpb = (results_participants['Trips per Buyer'][0] - results_participants['Trips per Buyer'][1])/results_participants['Trips per Buyer'][1]
    change_dpt = (results_participants['Dollars per Trip'][0] - results_participants['Dollars per Trip'][1])/results_participants['Dollars per Trip'][1]
    change_upt = (results_participants['Units per Trip'][0] - results_participants['Units per Trip'][1])/results_participants['Units per Trip'][1]

    if str(change_by) == 'inf':
        change_by = ''
    if str(change_ds) == 'inf':
        change_ds = ''
    if str(change_us) == 'inf':
        change_us = ''

    if str(change_dpb) == 'inf':
        change_dpb = ''
    if str(change_upb) == 'inf':
        change_upb = ''
    if str(change_tpb) == 'inf':
        change_tpb = ''
    if str(change_dpt) == 'inf':
        change_dpt = ''
    if str(change_upt) == 'inf':
        change_upt = ''


    new_row = pd.DataFrame({'Time Period' : ['% Change'], 'Buyers': [change_by], 'Brand Dollars': [change_ds], 'Brand Units':[change_us], 'Dollars per Buyer': [change_dpb], 'Units per Buyer':[change_upb], 'Trips per Buyer': [change_tpb], 'Dollars per Trip': [change_dpt], 'Units per Trip': [change_upt]})
    results_participants = pd.concat([results_participants, new_row], ignore_index=True)
    results_participants = results_participants.T.reset_index().T.reset_index(drop=True)
    results_participants_2 = results_participants
    results_participants_02 = results_participants_2
    results_participants_02

In [126]:
curs.execute(f'''

        DROP table if exists VMR_{brand_nm}_prior_table_shares;
        CREATE temp table VMR_{brand_nm}_prior_table_shares as

              SELECT CASE WHEN analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)' THEN 'VMR Period - YAGO ({round(weeks_per_period_final)} weeks)'

                         ELSE 'NA' END as Time_Period,

                         min(cal_dt) as min,
                         max(cal_dt) as max,


                    COUNT(distinct o.ord_designated_cnsmr_id_key) as campaign_buyers,
                    COUNT(distinct o.ord_event_key) as campaign_trips,
                    SUM(o.purch_amt) as campaign_dollars,
                    SUM(o.purch_qty) as campaign_units,

                    round((campaign_dollars::float/campaign_buyers::float),4) as dollars_per_buyer,
                    round((campaign_units::float/campaign_buyers::float),4) as units_per_buyer,
                    round((campaign_trips::float/campaign_buyers::float),4) as trips_per_buyer,
                    round((campaign_dollars::float/campaign_trips::float),4) as dollars_per_trip,
                    round((campaign_units::float/campaign_trips::float),4) as units_per_trip


              FROM ord_trd_itm_cnsmr_fact_ne_v o

                     INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                     INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                    INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)                 
                   -- INNER JOIN shopper_consistent_{analyst} v ON (o.ord_designated_cnsmr_id_key = v.cnsmr_id_key)

              WHERE o.purch_amt > 0
                    and o.purch_qty > 0
                    and analysis_period = 'Period Year Ago ({round(weeks_per_period_final)} weeks)'
                    and c.brand_nbr in ({brand_nbr_str})
                    and Time_Period != 'NA'


            GROUP BY 1 

            DISTRIBUTE REPLICATE;

''')


curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_vmr_metrics_participants;
    CREATE temp table VMR_{brand_nm}_vmr_metrics_participants as
    
        SELECT  
                CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'Campaign Period'
                     ELSE 'NA' END as Time_Period,
                     
                     min(cal_dt) as min,
                     max(cal_dt) as max,
        
        
                COUNT(distinct a.ord_designated_cnsmr_id_key) as campaign_buyers,
                COUNT(distinct a.ord_event_key) as campaign_trips,
                SUM(a.purch_amt) as campaign_dollars,
                SUM(a.purch_qty) as campaign_units,
                
                round((campaign_dollars::float/campaign_buyers::float),4) as dollars_per_buyer,
                round((campaign_units::float/campaign_buyers::float),4) as units_per_buyer,
                round((campaign_trips::float/campaign_buyers::float),4) as trips_per_buyer,
                round((campaign_dollars::float/campaign_trips::float),4) as dollars_per_trip,
                round((campaign_units::float/campaign_trips::float),4) as units_per_trip
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            --INNER JOIN shopper_consistent_{analyst} v ON (a.ord_designated_cnsmr_id_key = v.cnsmr_id_key)
            
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str})
              and Time_Period != 'NA'
              
        GROUP BY 1
        
   
        UNION
        
        
        SELECT  
                Time_Period,
                
                min,
                max,
        
                campaign_buyers,
                campaign_trips,
                campaign_dollars,
                campaign_units,
                
                dollars_per_buyer,
                units_per_buyer,
                trips_per_buyer,
                dollars_per_trip,
                units_per_trip
                
        FROM VMR_{brand_nm}_prior_table_shares
        
        
    DISTRIBUTE REPLICATE;

''')


In [127]:
results_participants = pd.read_sql(f'''
    SELECT *
    FROM VMR_{brand_nm}_vmr_metrics_participants
    ORDER BY CASE WHEN Time_Period = 'Campaign Period' then 1
              END

    ''',conn)

results_participants

,time_period,min,max,campaign_buyers,campaign_trips,campaign_dollars,campaign_units,dollars_per_buyer,units_per_buyer,trips_per_buyer,dollars_per_trip,units_per_trip
0,Campaign Period,2026-02-28,2026-03-29,2099785,3265956,1.104637e+08,10388473,52.6072,4.9474,1.5554,33.8228,3.1808
1,VMR Period - YAGO (4 weeks),2025-03-01,2025-03-30,733924,1192477,2.732577e+07,2802491,37.2324,3.8185,1.6248,22.9151,2.3501


In [128]:
vmr_trend_yago_summary

,Analysis Periods,Count Distinct Trips,Dollar Sales,Dollars per Trip,Units Sold,Units per Trip,% Change Dollars per Trip,% Change Units per Trip
0,YAGO Period,13574255,2.390810e+08,17.612829,26092326,1.9,-,-
1,VMR Period,13533312,2.507479e+08,18.528196,26899195,2.0,0.051972,0.052632


In [129]:
if vmr_trend_yago_summary.loc[0, 'Analysis Periods'] == 'YAGO Period':

    results_participants = results_participants[['time_period','campaign_buyers', 'campaign_dollars', 'campaign_units','dollars_per_buyer', 'units_per_buyer', 'trips_per_buyer', 'dollars_per_trip', 'units_per_trip']]
    results_participants = results_participants.rename(columns = {'time_period':'Time Period','campaign_buyers': 'Buyers','campaign_dollars':'Brand Dollars', 'campaign_units': 'Brand Units','dollars_per_buyer':'Dollars per Buyer', 'units_per_buyer':'Units per Buyer', 'trips_per_buyer':'Trips per Buyer', 'dollars_per_trip':'Dollars per Trip', 'units_per_trip':'Units per Trip'})

    change_by = (results_participants['Buyers'][0] - results_participants['Buyers'][1])/results_participants['Buyers'][1]
    change_ds = (results_participants['Brand Dollars'][0] - results_participants['Brand Dollars'][1])/results_participants['Brand Dollars'][1]
    change_us = (results_participants['Brand Units'][0] - results_participants['Brand Units'][1])/results_participants['Brand Units'][1]

    change_dpb = (results_participants['Dollars per Buyer'][0] - results_participants['Dollars per Buyer'][1])/results_participants['Dollars per Buyer'][1]
    change_upb = (results_participants['Units per Buyer'][0] - results_participants['Units per Buyer'][1])/results_participants['Units per Buyer'][1]
    change_tpb = (results_participants['Trips per Buyer'][0] - results_participants['Trips per Buyer'][1])/results_participants['Trips per Buyer'][1]
    change_dpt = (results_participants['Dollars per Trip'][0] - results_participants['Dollars per Trip'][1])/results_participants['Dollars per Trip'][1]
    change_upt = (results_participants['Units per Trip'][0] - results_participants['Units per Trip'][1])/results_participants['Units per Trip'][1]

    
    if str(change_by) == 'inf':
        change_by = ''
    if str(change_ds) == 'inf':
        change_ds = ''
    if str(change_us) == 'inf':
        change_us = ''
    
    if str(change_dpb) == 'inf':
        change_dpb = ''
    if str(change_upb) == 'inf':
        change_upb = ''
    if str(change_tpb) == 'inf':
        change_tpb = ''
    if str(change_dpt) == 'inf':
        change_dpt = ''
    if str(change_upt) == 'inf':
        change_upt = ''


    new_row = pd.DataFrame({'Time Period' : ['% Change'], 'Buyers': [change_by], 'Brand Dollars': [change_ds], 'Brand Units':[change_us], 'Dollars per Buyer': [change_dpb], 'Units per Buyer':[change_upb], 'Trips per Buyer': [change_tpb], 'Dollars per Trip': [change_dpt], 'Units per Trip': [change_upt]})
    results_participants = pd.concat([results_participants, new_row], ignore_index=True)
    results_participants = results_participants.T.reset_index().T.reset_index(drop=True)
    results_participants_3 = results_participants
    results_participants_03 = results_participants_3
    print(results_participants_03)
    
else:
    
    results_participants_3 = pd.DataFrame()
    change_dpt = 0
    print('halo')

                             0          1              2            3  \
0                  Time Period     Buyers  Brand Dollars  Brand Units   
1              Campaign Period  2099785.0   110463736.52   10388473.0   
2  VMR Period - YAGO (4 weeks)   733924.0    27325769.51    2802491.0   
3                     % Change   1.861039       3.042475     2.706871   

                   4                5                6                 7  \
0  Dollars per Buyer  Units per Buyer  Trips per Buyer  Dollars per Trip   
1            52.6072           4.9474           1.5554           33.8228   
2            37.2324           3.8185           1.6248           22.9151   
3           0.412941          0.29564        -0.042713          0.476005   

                8  
0  Units per Trip  
1          3.1808  
2          2.3501  
3        0.353474  


### Prior 26 weeks metrics:

In [130]:
curs.execute(f'''

        DROP table if exists VMR_{brand_nm}_prior_table_shares;
        CREATE temp table VMR_{brand_nm}_prior_table_shares as

              SELECT 'Prior 26 wks' as Time_Period,

                         min(cal_dt) as min,
                         max(cal_dt) as max,


                    COUNT(distinct o.ord_designated_cnsmr_id_key) as campaign_buyers,
                    COUNT(distinct o.ord_event_key) as campaign_trips,
                    SUM(o.purch_amt) as campaign_dollars,
                    SUM(o.purch_qty) as campaign_units,

                    round((campaign_dollars::float/campaign_buyers::float),4) as dollars_per_buyer,
                    round((campaign_units::float/campaign_buyers::float),4) as units_per_buyer,
                    round((campaign_trips::float/campaign_buyers::float),4) as trips_per_buyer,
                    round((campaign_dollars::float/campaign_trips::float),4) as dollars_per_trip,
                    round((campaign_units::float/campaign_trips::float),4) as units_per_trip


              FROM ord_trd_itm_cnsmr_fact_ne_v o

                     INNER JOIN VMR_{brand_nm}_date_filter b ON (o.ord_date_key = b.date_key)
                     INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                    INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = o.ord_designated_cnsmr_id_key)                 

              WHERE o.purch_amt > 0
                    and o.purch_qty > 0
                    and b.cal_dt between '{vmr_pre26wk_start}' and '{vmr_pre26wk_end}'
                    and c.brand_nbr in ({brand_nbr_str})


            GROUP BY 1 

            DISTRIBUTE REPLICATE;

''')


curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_vmr_metrics_participants;
    CREATE temp table VMR_{brand_nm}_vmr_metrics_participants as
    
        SELECT  
                CASE WHEN analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)' THEN 'Campaign Period'
                     ELSE 'NA' END as Time_Period,
                     
                     min(cal_dt) as min,
                     max(cal_dt) as max,
        
        
                COUNT(distinct a.ord_designated_cnsmr_id_key) as campaign_buyers,
                COUNT(distinct a.ord_event_key) as campaign_trips,
                SUM(a.purch_amt) as campaign_dollars,
                SUM(a.purch_qty) as campaign_units,
                
                round((campaign_dollars::float/campaign_buyers::float),4) as dollars_per_buyer,
                round((campaign_units::float/campaign_buyers::float),4) as units_per_buyer,
                round((campaign_trips::float/campaign_buyers::float),4) as trips_per_buyer,
                round((campaign_dollars::float/campaign_trips::float),4) as dollars_per_trip,
                round((campaign_units::float/campaign_trips::float),4) as units_per_trip
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            --INNER JOIN shopper_consistent_{analyst} v ON (a.ord_designated_cnsmr_id_key = v.cnsmr_id_key)
            
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str})
              and Time_Period != 'NA'
              
        GROUP BY 1
        
   
        UNION
        
        
        SELECT  
                Time_Period,
                
                min,
                max,
        
                campaign_buyers,
                campaign_trips,
                campaign_dollars,
                campaign_units,
                
                dollars_per_buyer,
                units_per_buyer,
                trips_per_buyer,
                dollars_per_trip,
                units_per_trip
                
        FROM VMR_{brand_nm}_prior_table_shares
        
        
    DISTRIBUTE REPLICATE;

''')


In [131]:
results_participants = pd.read_sql(f'''
    SELECT *
    FROM VMR_{brand_nm}_vmr_metrics_participants
    ORDER BY CASE WHEN Time_Period = 'Campaign Period' then 1
              END

    ''',conn)

results_participants

,time_period,min,max,campaign_buyers,campaign_trips,campaign_dollars,campaign_units,dollars_per_buyer,units_per_buyer,trips_per_buyer,dollars_per_trip,units_per_trip
0,Campaign Period,2026-02-28,2026-03-29,2099785,3265956,1.104637e+08,10388473,52.6072,4.9474,1.5554,33.8228,3.1808
1,Prior 26 wks,2025-08-30,2026-02-27,1603497,7576251,1.730937e+08,17813730,107.9476,11.1093,4.7248,22.8469,2.3513


ADD THIS TO THE NEXT

if len(results_participants.loc[results_participants['time_period']== 'Prior 26 wks']) != 0 :


In [132]:
if len(results_participants.loc[results_participants['time_period']== 'Prior 26 wks']) != 0 :

    results_participants = results_participants[['time_period','campaign_buyers', 'campaign_dollars', 'campaign_units','dollars_per_buyer', 'units_per_buyer', 'trips_per_buyer', 'dollars_per_trip', 'units_per_trip']]
    results_participants = results_participants.rename(columns = {'time_period':'Time Period','campaign_buyers': 'Buyers','campaign_dollars':'Brand Dollars', 'campaign_units': 'Brand Units','dollars_per_buyer':'Dollars per Buyer', 'units_per_buyer':'Units per Buyer', 'trips_per_buyer':'Trips per Buyer', 'dollars_per_trip':'Dollars per Trip', 'units_per_trip':'Units per Trip'})

    change_by_26 = (results_participants['Buyers'][0] - results_participants['Buyers'][1])/results_participants['Buyers'][1]
    change_ds_26 = (results_participants['Brand Dollars'][0] - results_participants['Brand Dollars'][1])/results_participants['Brand Dollars'][1]
    change_us_26 = (results_participants['Brand Units'][0] - results_participants['Brand Units'][1])/results_participants['Brand Units'][1]

    change_dpb_26 = (results_participants['Dollars per Buyer'][0] - results_participants['Dollars per Buyer'][1])/results_participants['Dollars per Buyer'][1]
    change_upb_26 = (results_participants['Units per Buyer'][0] - results_participants['Units per Buyer'][1])/results_participants['Units per Buyer'][1]
    change_tpb_26 = (results_participants['Trips per Buyer'][0] - results_participants['Trips per Buyer'][1])/results_participants['Trips per Buyer'][1]
    change_dpt_26 = (results_participants['Dollars per Trip'][0] - results_participants['Dollars per Trip'][1])/results_participants['Dollars per Trip'][1]
    change_upt_26 = (results_participants['Units per Trip'][0] - results_participants['Units per Trip'][1])/results_participants['Units per Trip'][1]


    if str(change_by_26) == 'inf':
        change_by_26 = ''
    if str(change_ds_26) == 'inf':
        change_ds_26 = ''
    if str(change_us_26) == 'inf':
        change_us_26 = ''

    if str(change_dpb_26) == 'inf':
        change_dpb_26 = ''
    if str(change_upb_26) == 'inf':
        change_upb_26 = ''
    if str(change_tpb_26) == 'inf':
        change_tpb_26 = ''
    if str(change_dpt_26) == 'inf':
        change_dpt_26 = ''
    if str(change_upt_26) == 'inf':
        change_upt_26 = ''


    new_row = pd.DataFrame({'Time Period' : ['% Change'], 'Buyers': [change_by_26], 'Brand Dollars': [change_ds_26], 'Brand Units':[change_us_26],'Dollars per Buyer': [change_dpb_26], 'Units per Buyer':[change_upb_26], 'Trips per Buyer': [change_tpb_26], 'Dollars per Trip': [change_dpt_26], 'Units per Trip': [change_upt_26]})
    results_participants = pd.concat([results_participants, new_row], ignore_index=True)
    results_participants = results_participants.T.reset_index().T.reset_index(drop=True)
    results_participants_4 = results_participants
    results_participants_04 = results_participants_4
    results_participants_04

## **15) GETTING THE TREND FOR ALL BRAND TRANSACTIONS (65 WEEKS PRIOR PERIOD + VMR PERIOD):**

In [133]:

vmr_preperiod_start = int(str(vmr_pre_period_start.strftime('%Y-%m-%d')).replace('-',''))
vmr_preperiod_end = int(str(vmr_pre_period_end.strftime('%Y-%m-%d')).replace('-',''))

yago_26wk_start_var = int(str(yago_26wk_start.strftime('%Y-%m-%d')).replace('-',''))
vmr_pre26wk_start_var = int(str(vmr_pre26wk_start.strftime('%Y-%m-%d')).replace('-',''))
yago_start = int(str(yago_print_start.strftime('%Y-%m-%d')).replace('-',''))
reward_start = int(str(reward_print_start.strftime('%Y-%m-%d')).replace('-',''))
reward_end = int(str(reward_print_end.strftime('%Y-%m-%d')).replace('-',''))
yago_start = int(str(yago_print_start.strftime('%Y-%m-%d')).replace('-',''))

#if we want to use redemption end as a max date limit for the trend:
#redempt_end = int(str(redemption_end.strftime('%Y-%m-%d')).replace('-','')) #redemption_end defined at the beggining of the code
#redemption_end_yago = redemption_end - dt.timedelta(weeks = 52)
#yago_end = int(str(redemption_end_yago.strftime('%Y-%m-%d')).replace('-',''))

#if we want to use current date as a max date limit for the trend
redempt_end = int(str(post_4wk_end.strftime('%Y-%m-%d')).replace('-','')) #redemption_end defined at the beggining of the code
redemption_end_yago = post_4wk_end - dt.timedelta(weeks = 52)
yago_end = int(str(redemption_end_yago.strftime('%Y-%m-%d')).replace('-',''))


curs.execute(f'''       
    DROP table if exists VMR_{brand_nm}_brand_trend;
    CREATE temp table VMR_{brand_nm}_brand_trend as
    
    
        SELECT  
               cal_sun_wk_ending_dt,
               CASE WHEN ord_date_key between {yago_start} and {yago_end} THEN 'YAGO Period'
                    WHEN ord_date_key between {yago_end}+1 and {reward_start} THEN 'Prior Period in between'
                    WHEN ord_date_key between {reward_start}+1 and {reward_end} THEN 'VMR Period'
                    ELSE 'NA'
                    END as analysis_periods,
               CASE WHEN ord_date_key between '{vmr_preperiod_start}' and '{vmr_preperiod_end}' THEN 'VMR Prior Period'
                    WHEN ord_date_key between {reward_start}+1 and {reward_end} THEN 'VMR Period'
                    WHEN ord_date_key between {yago_start} and {yago_end} THEN 'YAGO Period'
                    ELSE 'NA'
                    END as analysis_periods_second_chart,
                CASE WHEN ord_date_key between {vmr_pre26wk_start_var} and {redempt_end} THEN 'Recent Trend Period'
                     WHEN ord_date_key between {yago_26wk_start_var} and {yago_end} THEN 'YAGO Trend Period'
                    ELSE 'NA'
                    END as trend_chart,
               count(distinct a.ord_event_key) as count_distinct_trips,
               sum(a.purch_amt) as dollar_sales,
               dollar_sales/count_distinct_trips as dollars_per_trip,
               sum(a.purch_qty) as units_sales,
               units_sales/count_distinct_trips::float as units_per_trip,
               RANK() OVER(PARTITION BY analysis_periods ORDER BY cal_sun_wk_ending_dt ASC) as nbr_week
              
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            
        WHERE brand_nbr IN ({brand_nbr_str})
        
        GROUP BY 1,2,3,4
        
        ORDER BY 1 DESC
        
    DISTRIBUTE REPLICATE;

''')

In [134]:
vmr_trend_brand = pd.read_sql(f'''
    SELECT *
    FROM VMR_{brand_nm}_brand_trend
    ORDER BY 1 DESC
    ''',conn)

vmr_trend_brand

,cal_sun_wk_ending_dt,analysis_periods,analysis_periods_second_chart,trend_chart,count_distinct_trips,dollar_sales,dollars_per_trip,units_sales,units_per_trip,nbr_week
0,2026-04-19,NA,NA,Recent Trend Period,2008,32610.35,16.240214,3737,1.861056,30
1,2026-04-12,NA,NA,Recent Trend Period,3188079,57943269.78,18.174979,6174705,1.936811,29
2,2026-04-05,NA,NA,Recent Trend Period,3244930,58161821.55,17.923906,6249549,1.925943,28
3,2026-03-29,VMR Period,VMR Period,Recent Trend Period,3089991,56620188.29,18.323738,5947720,1.924834,5
4,2026-03-22,VMR Period,VMR Period,Recent Trend Period,3180929,58644543.88,18.436294,6365625,2.001184,4
...,...,...,...,...,...,...,...,...,...,...
87,2024-09-29,NA,NA,YAGO Trend Period,2897349,46974872.72,16.213052,5276585,1.821177,5
88,2024-09-22,NA,NA,YAGO Trend Period,2954573,48182895.40,16.307904,5337724,1.806597,4
89,2024-09-15,NA,NA,YAGO Trend Period,2956887,47808971.80,16.168684,5307489,1.794958,3
90,2024-09-08,NA,NA,YAGO Trend Period,2958720,47711832.62,16.125835,5291878,1.788570,2


In [135]:
vmr_trend_brand['analysis_periods'].unique()

array(['NA', 'VMR Period', 'Prior Period in between', 'YAGO Period'],
      dtype=object)

In [136]:
vmr_trend_brand_sl = pd.read_sql(f'''
    SELECT cal_sun_wk_ending_dt, 
           trend_chart,
           SUM(dollar_sales) as dollar_sales,
           SUM(units_sales) as units_sales
    FROM VMR_{brand_nm}_brand_trend
    --WHERE analysis_periods = 'Recent Trend Period' OR trend_chart = 'YAGO Trend Period'
    WHERE trend_chart = 'Recent Trend Period' OR trend_chart = 'YAGO Trend Period'
    GROUP BY 1,2
    ORDER BY 1 DESC
    
    ''',conn)

vmr_trend_brand_sl

,cal_sun_wk_ending_dt,trend_chart,dollar_sales,units_sales
0,2026-04-19,Recent Trend Period,32610.35,3737
1,2026-04-12,Recent Trend Period,57943269.78,6174705
2,2026-04-05,Recent Trend Period,58161821.55,6249549
3,2026-03-29,Recent Trend Period,56620188.29,5947720
4,2026-03-22,Recent Trend Period,58644543.88,6365625
...,...,...,...,...
65,2024-09-29,YAGO Trend Period,46974872.72,5276585
66,2024-09-22,YAGO Trend Period,48182895.40,5337724
67,2024-09-15,YAGO Trend Period,47808971.80,5307489
68,2024-09-08,YAGO Trend Period,47711832.62,5291878


In [137]:
vmr_trend_brand_sl_exc = pd.read_sql(f'''
    SELECT cal_sun_wk_ending_dt as week, 
           SUM(dollar_sales) as dollars,
           SUM(units_sales) as units,
           SUM(count_distinct_trips) as trips,
           analysis_periods_second_chart as period
    FROM VMR_{brand_nm}_brand_trend
    WHERE analysis_periods_second_chart != 'NA'
    GROUP BY 1,5
    ORDER BY 1 DESC
    
    ''',conn)

vmr_trend_brand_sl_exc

vmr_trend_brand_sl_exc = vmr_trend_brand_sl_exc.rename(columns = {'week':'Week', 'dollars':'Dollars', 'units':'Units', 'period':'Period'})
vmr_trend_brand_sl_exc

,Week,Dollars,Units,trips,Period
0,2026-03-29,56620188.29,5947720,3089991,VMR Period
1,2026-03-22,58644543.88,6365625,3180929,VMR Period
2,2026-03-15,58957523.53,6357537,3194893,VMR Period
3,2026-03-08,59471623.76,6444756,3216012,VMR Period
4,2026-03-01,38729881.89,4078334,2136066,VMR Prior Period
5,2026-03-01,7916432.54,855711,430079,VMR Period
6,2026-02-22,53282188.19,5615962,2949423,VMR Prior Period
7,2026-02-15,57511382.01,6152671,3243435,VMR Prior Period
8,2026-02-08,54699919.37,5843684,3060082,VMR Prior Period
9,2026-02-01,31207044.51,3424375,1726207,VMR Prior Period


## Nestle Metrics

In [138]:
if dominance != 'Kroger':

    curs.execute(f'''

        DROP table if exists nestle_VMR_{brand_nm}_prebuyers;
        CREATE temp table nestle_VMR_{brand_nm}_prebuyers as

            SELECT distinct o.ord_designated_cnsmr_id_key,
                   count(distinct CASE WHEN c.brand_nbr in ({brand_nbr_str}) THEN o.ord_event_key else null end) as b_trips,
                   count(distinct CASE WHEN c.brand_nbr in ({cat_nbr_str}) THEN o.ord_event_key else null end) as c_trips,
                   sum(CASE WHEN c.brand_nbr in ({brand_nbr_str}) THEN o.purch_amt else 0 end) as bdollars,
                   sum(CASE WHEN c.brand_nbr in ({cat_nbr_str}) THEN o.purch_amt else 0 end) as cdollars

             FROM ORD_TRD_ITM_CNSMR_FACT_NE_V o

                    INNER JOIN shopper_consistent_{analyst} s ON (o.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
                    INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                    INNER JOIN date_v b ON (o.ord_date_key = b.date_key)

             WHERE b.cal_dt BETWEEN '{prior_period_start}' and '{prior_period_end}' 
                   and purch_amt > 0
                   and purch_qty > 0

             GROUP BY 1


             DISTRIBUTE RANDOM; 

    ''')

    
if dominance == 'Kroger':
    
    
        curs.execute(f'''

        DROP table if exists nestle_VMR_{brand_nm}_prebuyers;
        CREATE temp table nestle_VMR_{brand_nm}_prebuyers as

            SELECT distinct o.ord_designated_cnsmr_id_key,
                   count(distinct CASE WHEN c.brand_nbr in ({brand_nbr_str}) THEN o.ord_event_key else null end) as b_trips,
                   count(distinct CASE WHEN c.brand_nbr in ({cat_nbr_str}) THEN o.ord_event_key else null end) as c_trips,
                   sum(CASE WHEN c.brand_nbr in ({brand_nbr_str}) THEN o.purch_amt else 0 end) as bdollars,
                   sum(CASE WHEN c.brand_nbr in ({cat_nbr_str}) THEN o.purch_amt else 0 end) as cdollars

             FROM ORD_TRD_ITM_CNSMR_FACT_NE_V o

                    INNER JOIN shopper_consistent_{analyst} s ON (o.ord_designated_cnsmr_id_key = s.cnsmr_id_key)
                    INNER JOIN VMR_{brand_nm}_upc_filter c ON (o.trade_item_key = c.trade_item_key)
                    INNER JOIN date_v b ON (o.ord_date_key = b.date_key)

             WHERE b.cal_dt BETWEEN '{vmr_pre26wk_start}' and '{vmr_pre26wk_end}' 
                   and purch_amt > 0
                   and purch_qty > 0

             GROUP BY 1


             DISTRIBUTE RANDOM; 

    ''')


In [139]:
curs.execute(f'''


 DROP table if exists nestle_buyers_{brand_nm}_vmr_metrics_participants;
    CREATE temp table nestle_buyers_{brand_nm}_vmr_metrics_participants as

SELECT a.ord_designated_cnsmr_id_key,
       CASE 
       WHEN b_trips=0 or  b.ord_designated_cnsmr_id_key is null THEN 'New'
       WHEN b_trips>0 THEN 'Existing'
       ELSE 'NA' END as type
       
FROM VMR_{brand_nm}_reward_ids_csmr a
    LEFT JOIN nestle_VMR_{brand_nm}_prebuyers b ON (b.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
WHERE type != 'NA'
GROUP BY 1,2
ORDER BY CASE WHEN type = 'New' then 1
              WHEN type = 'Existing' then 2
              END

''')

In [140]:
pd.read_sql(f'''

with prior_table AS (
select * from nestle_buyers_{brand_nm}_vmr_metrics_participants)


SELECT type, count(distinct ord_designated_cnsmr_id_key) as ids
from prior_table
group by 1


''', conn)

,type,ids
0,New,297895
1,Existing,1801890


In [141]:
presales_metrics_nest = pd.read_sql(f'''

        SELECT  d.type,
                (SELECT COUNT(distinct ord_designated_cnsmr_id_key) FROM nestle_buyers_{brand_nm}_vmr_metrics_participants WHERE type = 'Existing') as ID_count,
                CASE WHEN SUM(a.purch_amt) > 0 THEN  SUM(a.purch_amt) ELSE 0 END as Pre_Sales_total,
                CASE WHEN AVG(a.purch_amt) > 0 THEN AVG(a.purch_amt) ELSE 0 END as Pre_Sales_Avg,
                CASE WHEN SUM(a.purch_qty)>0 THEN SUM(a.purch_qty) ELSE 0 END as Pre_Units_total,
                CASE WHEN AVG(a.purch_qty)>0 THEN AVG(a.purch_qty) ELSE 0 END as Pre_Units_Avg,
                CASE WHEN COUNT(distinct a.ord_event_key)>0 THEN COUNT(distinct a.ord_event_key) ELSE 0 END as Pre_Trips_total,
                CASE WHEN round(COUNT(distinct a.ord_event_key)/COUNT(distinct a.ord_designated_cnsmr_id_key),2)>0 THEN round(COUNT(distinct a.ord_event_key)/COUNT(distinct a.ord_designated_cnsmr_id_key),2) ELSE 0 END as Pre_Trips_Avg
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN date_v b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            INNER JOIN nestle_buyers_{brand_nm}_vmr_metrics_participants d ON (d.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            
            
        WHERE b.cal_dt BETWEEN '{vmr_pre26wk_start}' and '{vmr_pre26wk_end}' 
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str}) and type = 'Existing'
              
        GROUP BY 1
        
        
        
        UNION
        
        
        
        SELECT  type,
                COUNT(distinct a.ord_designated_cnsmr_id_key) as ID_count,
                0 as VMR_Sales_total,
                0 as VMR_Sales_Avg,
                0 as VMR_Units_total,
                0 as VMR_Sales_Avg,
                0 as VMR_Trips_total,
                0 as VMR_Trips_Avg
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            INNER JOIN nestle_buyers_{brand_nm}_vmr_metrics_participants d ON (d.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str}) and
              type = 'New'

              
        GROUP BY 1
        
''',conn)

presales_metrics_nest


,type,id_count,pre_sales_total,pre_sales_avg,pre_units_total,pre_units_avg,pre_trips_total,pre_trips_avg
0,New,297895,0.000000e+00,0.00,0,0.000000,0,0.0
1,Existing,1801890,1.730830e+08,10.63,17812346,1.094076,7575654,4.0


In [142]:
vmr_metrics_nest = pd.read_sql(f'''

        
        SELECT  d.type,
                COUNT(distinct a.ord_designated_cnsmr_id_key) as ID_count,
                SUM(a.purch_amt) as VMR_Sales_total,
                AVG(a.purch_amt) as VMR_Sales_Avg,
                SUM(a.purch_qty) as VMR_Units_total,
                AVG(a.purch_qty) as VMR_Sales_Avg,
                COUNT(distinct a.ord_event_key) as VMR_Trips_total,
                round(COUNT(distinct a.ord_event_key)/COUNT(distinct a.ord_designated_cnsmr_id_key),2)::float as VMR_Trips_Avg
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            INNER JOIN nestle_buyers_{brand_nm}_vmr_metrics_participants d ON (d.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str})

              
        GROUP BY 1
        
        
        
''',conn)


vmr_metrics_nest

,type,id_count,vmr_sales_total,vmr_sales_avg,vmr_units_total,vmr_sales_avg,vmr_trips_total,vmr_trips_avg
0,New,297895,13320344.33,12.00,1216608,1.096327,358397,1.0
1,Existing,1801890,97143392.19,11.64,9171865,1.099913,2907559,1.0


In [143]:
nestle_merged_df = pd.merge(presales_metrics_nest, vmr_metrics_nest, on=['type', 'id_count'], how='inner')
nestle_merged_df = nestle_merged_df.rename(columns={'type':'Shopper', 'id_count':'IDS',	'pre_sales_total':'Pre-Sales', 	'pre_sales_avg': 'AVG Pre-Sales', 'pre_units_total': 'Pre-Units', 'pre_units_avg':'AVG Pre-Units', 'pre_trips_total':'Pre-trips', 'pre_trips_avg':'AVG Pre-Trips', 'vmr_sales_total':'VMR-Sales', 'vmr_sales_avg':'AVG VMR-Sales', 'vmr_units_total':'VMR-Units', 'vmr_sales_avg':'AVG VMR-Sales', 'vmr_trips_total':'VMR-Trips', 'vmr_trips_avg':'AVG VMR-Trips'})
nestle_merged_df

,Shopper,IDS,Pre-Sales,AVG Pre-Sales,Pre-Units,AVG Pre-Units,Pre-trips,AVG Pre-Trips,VMR-Sales,AVG VMR-Sales,VMR-Units,AVG VMR-Sales,VMR-Trips,AVG VMR-Trips
0,New,297895,0.000000e+00,0.00,0,0.000000,0,0.0,13320344.33,12.00,1216608,1.096327,358397,1.0
1,Existing,1801890,1.730830e+08,10.63,17812346,1.094076,7575654,4.0,97143392.19,11.64,9171865,1.099913,2907559,1.0


In [144]:
lift_metrics = pd.read_sql(f'''


        SELECT  'Pre-Period Existing' as type,
                round(SUM(a.purch_amt) :: FLOAT/COUNT(distinct a.ord_event_key) :: FLOAT, 4) as dollars_per_trip,
                round(SUM(a.purch_qty) :: FLOAT/COUNT(distinct a.ord_event_key) :: FLOAT, 4) as units_per_trip
                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN date_v b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            INNER JOIN nestle_buyers_{brand_nm}_vmr_metrics_participants d ON (d.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            
            
        WHERE b.cal_dt BETWEEN '{vmr_pre26wk_start}' and '{vmr_pre26wk_end}' 
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str})
              
        GROUP BY 1
        
        
        
        UNION
        
        
         
        SELECT  d.type,
                round(SUM(a.purch_amt) :: FLOAT/COUNT(distinct a.ord_event_key) :: FLOAT, 4) as dollars_per_trip,
                round(SUM(a.purch_qty) :: FLOAT/COUNT(distinct a.ord_event_key) :: FLOAT, 4) as  units_per_trip

                
        FROM ord_trd_itm_cnsmr_fact_ne_v a 
        
            INNER JOIN VMR_{brand_nm}_date_filter b ON (a.ord_date_key = b.date_key)
            INNER JOIN VMR_{brand_nm}_upc_filter c ON (a.trade_item_key = c.trade_item_key)
            INNER JOIN VMR_{brand_nm}_printing_stores s ON (a.ord_touchpoint_key = s.touchpoint_key)
            INNER JOIN VMR_{brand_nm}_reward_ids_csmr t ON (t.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            INNER JOIN nestle_buyers_{brand_nm}_vmr_metrics_participants d ON (d.ord_designated_cnsmr_id_key = a.ord_designated_cnsmr_id_key)
            
            
        WHERE analysis_period = 'VMR Print Period ({round(weeks_per_period_final)} weeks)'
              AND a.purch_amt > 0
              and a.purch_qty > 0
              and brand_nbr IN ({brand_nbr_str})

              
        GROUP BY 1
    
        
       
        
''',conn)

lift_metrics


,type,dollars_per_trip,units_per_trip
0,Pre-Period Existing,22.8469,2.3513
1,New,37.1665,3.3946
2,Existing,33.4106,3.1545


values_1 = ['NA', lift_metrics[] ,'NA']

lift_metrics.insert(2, 'Lift', , allow_duplicates=False)

nestle_merged_df_2

## **16) CREATING EXCEL OUTPUT:**

In [145]:
from pandas import ExcelWriter
from pandas import ExcelFile
import xlsxwriter
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import NamedStyle, PatternFill, Border, Side, Alignment, Protection, Font
from openpyxl.drawing.image import Image
from datetime import date

if segment_type == 1:
    segment_type_def = "UPC"
elif segment_type == 2:
    segment_type_def = "BRAND"
elif segment_type == 3:
    segment_type_def = "CATEGORY"
elif segment_type == 4:
    segment_type_def = "CUSTOM BRAND DESCR"
elif segment_type == 5:
    segment_type_def = "RETAILER DESCR"
elif segment_type == 6:
    segment_type_def = "KROGER DESCR"



export_file_name = report_name_for_export + "_" + segment_type_def +  "_VMR_Scorecard"  # Name of the final Excel file 
out_name = export_path + export_file_name + '_' + str(last_data_dt) + ".xlsx"

print('Name and location for output workbook is',out_name,'\n')

head_color = "Accent1"

wb = Workbook()
ws1 = wb.active
ws1.sheet_view.showGridLines = False

# create easier to read named styles to control number format in Excel
format_comma_no_decimal = NamedStyle(name='format_comma_no_decimal')
format_comma_no_decimal.number_format = '#,##0'
wb.add_named_style(format_comma_no_decimal)

format_comma_one_decimal = NamedStyle(name='format_comma_one_decimal')
format_comma_one_decimal.number_format = '#,##0.0'
wb.add_named_style(format_comma_one_decimal)

format_short_date = NamedStyle(name='format_short_date')
format_short_date.number_format = 'm/d/yyyy'
wb.add_named_style(format_short_date)

format_dollars_two_decimals = NamedStyle(name='format_dollars_two_decimals')
format_dollars_two_decimals.number_format = '$#,##0.00'
wb.add_named_style(format_dollars_two_decimals)

format_dollars_no_decimals = NamedStyle(name='format_dollars_no_decimals')
format_dollars_no_decimals.number_format = '$#,##0'
wb.add_named_style(format_dollars_no_decimals)

format_percent_one_decimal = NamedStyle(name='format_percent_one_decimal')
format_percent_one_decimal.number_format = '0.0%'
wb.add_named_style(format_percent_one_decimal)

format_percent_no_decimal = NamedStyle(name='format_percent_no_decimal')
format_percent_no_decimal.number_format = '0%'
wb.add_named_style(format_percent_no_decimal)

Name and location for output workbook is /analytical_services/Public/moricejuan/2026_WG_March_Beauty_Event_S25G10_UPC_VMR_Scorecard_2026-04-12.xlsx 



In [146]:
##### WORKSHEET 1 ####

ws1.title = 'PARAMETERS'

ws1['A1'] = 'Brand Name:'
ws1['B1'] = brand_nm

ws1['A3'] = 'UPC LMC List:'
ws1['B3'] = lmc_list_id

ws1['A5'] = 'Brands:'
ws1['B5'] = brand_nbr_str
ws1['A6'] = 'Categories:'
ws1['B6'] = cat_nbr_str

ws1['A8'] = 'Reward BL Codes:'
ws1['B8'] = BL_CODES

ws1['A10'] = 'Announcements:'
ws1['B10'] = Announcement

ws1['A12'] = 'Qualifying Reward Amount:'
ws1['B12'] = min_threshold_statement

ws1['A14'] = 'Redemption days:'
ws1['B14'] = redemption_days

ws1['B15'] = ''

ws1['A16'] = 'Static_1:'
ws1['B16'] = static_1
ws1['A17'] = 'Static_2:'
ws1['B17'] = static_2


ws1['B3'].alignment = Alignment(wrap_text=True, horizontal="left" )

for i in range(14,18):
    
    ws1['B'+str(i)].alignment = Alignment(wrap_text=True, horizontal="left" )
    
ws1['A18'] = ''

ws1['A19'] = 'Trackable ID:'
ws1['A20'] = 'Qualifying Trip:'
ws1['A21'] = 'Qualifying ID:'
ws1['A22'] = 'Reward ID:'
ws1['B19'] = 'ID that we can track purchases over time'
ws1['B20'] = 'Trip where the qualifying reward amount was purchased'
ws1['B21'] = 'Trackable ID that purchased the qualifying amount'
ws1['B22'] = 'Trackable ID that purchased the qualifying amount AND received a reward print'

ws1['A23'] = ''

ws1['A24'] = 'Reward BLs Analyzed'

for r in dataframe_to_rows(df_promo_summary, index=False, header=True):
    ws1.append(r)

for cell in ws1[25]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")

for i in range(1,len(df_promo_summary)+1):
    for cell in ws1[25+i]:
        cell.alignment=Alignment(wrap_text=True, horizontal="center")

        
ws1['A'+str(25 + len(df_promo_summary)+1)] = ''

n = 25 + len(df_promo_summary)+1 


#------------------------------------------------------------

ws1['A'+str(n+2)] = "Analysis Period Dates"
    
for r in dataframe_to_rows(df_date_check, index=False, header=True):
    ws1.append(r)
    
for cell in ws1[n+3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center" ) 

    
#------------------------------------------------------------
ws1['A'+str(n+10)] = 'Note:  VMR Period = Campaign Period'

ws1['A'+str(n+12)] = 'UPC Hierarchy'
    
for r in dataframe_to_rows(df_upcs_quick[['brand_nbr', 'brand_desc', 'nbr_of_upcs']], index=False, header=True):
    ws1.append(r) 
    
for cell in ws1[n+13]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    


#------------------------------------------------------------

next_cell = n+12+len(df_upcs_quick[['brand_nbr', 'brand_desc', 'nbr_of_upcs']]) + 4


if segment_type_def != 'UPC':
    
    ws1['A'+str(next_cell)] = "Segment List based on " + segment_type_def

    for r in dataframe_to_rows(df_segments_2, index=False, header=True):
        ws1.append(r)

    for cell in ws1[next_cell+1]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center" ) 

    
    
##############################################################################

for cell in ws1[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

ws1.column_dimensions['A'].width = 35        

for col_name in ['B','C']:
    for cols in ws1[col_name]:
        ws1.column_dimensions[col_name].width = 23       

for col_name in ['D','E','F','G']:
    for cols in ws1[col_name]:
        ws1.column_dimensions[col_name].width = 18 

#look
for i in range(n+13,n+16):
    for j in ['B', 'C']:
        ws1[j+str(i)].alignment=Alignment(wrap_text=True, horizontal="center")
        
for i in range(n+14,n+16):
    for j in ['C']:       
        ws1[j+str(i)].style = format_comma_no_decimal
        
for i in range(n+2,n+10):
    for j in ['B', 'C']:
        ws1[j+str(i)].alignment=Alignment(wrap_text=True, horizontal="center")

        
#update n+11
for i in range(n+13,next_cell-1):
    for j in ['A', 'B', 'C']:
        ws1[j+str(i)].alignment=Alignment(wrap_text=True, horizontal="center")
        
        
for i in range(next_cell+1,next_cell+1+len(df_segments_2)+10):
    for j in ['A', 'B', 'C']:
        ws1[j+str(i)].alignment=Alignment(wrap_text=True, horizontal="center")
        
        
for i in [19,20,21,22]:
    ws1['A'+str(i)].font = Font(bold=True, color = '000000', size = 11, name = 'Calibri' )


In [147]:
##### WORKSHEET 2 ####

ws8 = wb.create_sheet(title='TTL trips-Weekly Metrics')
ws8.sheet_view.showGridLines = False

ws8['A1'] = 'Total Trips - Weekly Metrics'

ws8['A2'] = 'Metrics based on all trips where the promoted product was purchased (reward level and non-reward level trips)'

vmr_trend_brand_sl_2 = vmr_trend_brand_sl[['cal_sun_wk_ending_dt', 'dollar_sales', 'units_sales']]

for r in dataframe_to_rows(vmr_trend_brand_sl_2, index=False, header=True):
    ws8.append(r)

for cell in ws8[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    

for cell in ws8[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )



for col_name in ['A','B','C','D','E']:
    for cols in ws8[col_name]:
        ws8.column_dimensions[col_name].width = 25 
    

for i in (range(4,len(vmr_trend_brand_sl)+4)):
    for j in ('B'):
        ws8[j+str(i)].style = format_dollars_two_decimals
        
        
for i in (range(4,len(vmr_trend_brand_sl)+4)):
    for j in ('C'):
        ws8[j+str(i)].style = format_comma_no_decimal


        #Center all rows

In [148]:
vmr_trend_brand_sl[['cal_sun_wk_ending_dt', 'dollar_sales', 'units_sales']]

,cal_sun_wk_ending_dt,dollar_sales,units_sales
0,2026-04-19,32610.35,3737
1,2026-04-12,57943269.78,6174705
2,2026-04-05,58161821.55,6249549
3,2026-03-29,56620188.29,5947720
4,2026-03-22,58644543.88,6365625
...,...,...,...
65,2024-09-29,46974872.72,5276585
66,2024-09-22,48182895.40,5337724
67,2024-09-15,47808971.80,5307489
68,2024-09-08,47711832.62,5291878


In [149]:
##### WORKSHEET 2 ####

ws9 = wb.create_sheet(title='TTL trips-Metrics')
ws9.sheet_view.showGridLines = False

ws9['A1'] = 'Total Trips Metrics'

ws9['A2'] = 'Metrics based on all trips where the promoted product was purchased (reward level and non-reward level trips)'


df_transposed = vmr_trend_yago_summary[['Dollars per Trip', 'Units per Trip']].T
df_transposed = df_transposed.reset_index()
df_transposed = df_transposed.rename(columns = {0:'YAGO Period', 1:'VMR Period', 'index': ''})
#update
new_row_1 = {'':'Dollars Share','YAGO Period': "{:.1%}".format(share_yago_dol), 'VMR Period': "{:.1%}".format(share_vmr_dol)}
new_row_2 = {'':'Units Share','YAGO Period': "{:.1%}".format(share_yago_unit), 'VMR Period': "{:.1%}".format(share_vmr_unit)}
df_transposed.loc[len(df_transposed)] = new_row_1
df_transposed.loc[len(df_transposed)] = new_row_2


for r in dataframe_to_rows(df_transposed, index=False, header=True):
    ws9.append(r)

for cell in ws9[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    

for cell in ws9[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

    
ws9.column_dimensions['A'].width = 40     


ws9['B4'].style = format_dollars_two_decimals
ws9['C4'].style = format_dollars_two_decimals

#update
ws9['B6'].style = format_comma_one_decimal
ws9['B7'].style = format_comma_one_decimal
ws9['C6'].style = format_comma_one_decimal
ws9['C7'].style = format_comma_one_decimal

#update

for i in range(4,9):
    for j in ['B', 'C']:
        ws9[j+str(i)].alignment=Alignment(wrap_text=True, horizontal="center")


In [150]:
##### WORKSHEET 2 ####

ws14 = wb.create_sheet(title='Reward Trips')
ws14.sheet_view.showGridLines = False

ws14['A1'] = 'Campaign Reward Trips'

ws14['A2'] = 'Metrics based on qualifying trips'


df_rwd_trans  = {
    
    '': ['All Other Transactions', 'Reward Level Transactions'],
    '% trips': ["{:.0%}".format(1-pct_reward_trips), "{:.0%}".format(pct_reward_trips)]
    
}

df_rwd_trans = pd.DataFrame(df_rwd_trans)


for r in dataframe_to_rows(df_rwd_trans, index=False, header=True):
    ws14.append(r)

for cell in ws14[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    

for cell in ws14[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

    
#update
ws14['A6'] = ''
ws14['A7'] = 'VMR Campaign Period:'


for r in dataframe_to_rows(level_ct, index=False, header=True):
    ws14.append(r)

for cell in ws14[8]: #update
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    

row_end = 8 + len(level_ct)    


ws14['A'+str(row_end+1)] = ''


if len(level_ct_yago)>1:

    ws14['A'+str(row_end+2)] = 'YAGO Period:'

    for r in dataframe_to_rows(level_ct_yago, index=False, header=True):
        ws14.append(r)

    for cell in ws14[row_end+3]: #update
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")
        
else:
    ws14['A'+str(row_end+2)] = 'YAGO Period:'
    ws14['A'+str(row_end+3)] = '[No data available]'


row_end_2 = row_end+ 3 + len(level_ct_yago)    


ws14['A'+str(row_end_2+1)] = ''


if len(level_ct_prep)>1:

    ws14['A'+str(row_end_2+2)] = 'Pre-Period:'

    for r in dataframe_to_rows(level_ct_prep, index=False, header=True):
        ws14.append(r)

    for cell in ws14[row_end_2+3]: #update
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")  
           
else:
    ws14['A'+str(row_end_2+2)] = 'Pre-Period:'
    ws14['A'+str(row_end_2+3)] = '[No data available]'



row_end_3 = row_end_2 + 3 + len(level_ct_prep)

ws14['A'+str(row_end_3+1)] = ''


if len(level_ct_prior)>1:

    ws14['A'+str(row_end_3+2)] = 'Prior 52 wk Period:'

    for r in dataframe_to_rows(level_ct_prior, index=False, header=True):
        ws14.append(r)

    for cell in ws14[row_end_3+3]: #update
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    
else:
    ws14['A'+str(row_end_3+2)] = 'Prior 52 wk Period:'
    ws14['A'+str(row_end_3+3)] = '[No data available]'


    
final_row = row_end_3+3+len(level_ct_prior)


ws14.column_dimensions['A'].width = 40     
ws14.column_dimensions['B'].width = 20
ws14.column_dimensions['C'].width = 20
ws14.column_dimensions['D'].width = 20


#update
for i in range(8, final_row+2):
    for j in ('B'):
        ws14[j+str(i)].style = format_dollars_two_decimals
        
for i in range(8, final_row+2):
    for j in ('C','D'):
        ws14[j+str(i)].style = format_comma_no_decimal
        

for cell in ws14[8]: #update
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    

for cell in ws14[row_end+3]: #update
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")

for cell in ws14[row_end_2+3]: #update
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")  

for cell in ws14[row_end_3+3]: #update
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    

    

ADD FOR PARTICIPANTS RESULTS 2 NEXT

In [151]:
##### WORKSHEET 2 ####

ws10 = wb.create_sheet(title='Campaign Trends')
ws10.sheet_view.showGridLines = False

ws10['A1'] = 'Campaign Trends'

ws10['A2'] = 'Metrics based on qualifying IDs'


results_participants_1 = results_participants_01
results_participants_1.columns = results_participants_1.iloc[0]
results_participants_1 = results_participants_1[1:]


for r in dataframe_to_rows(results_participants_1, index=False, header=True):
    ws10.append(r)

for cell in ws10[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    

for cell in ws10[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

    
ws10.column_dimensions['A'].width = 40       
       

for i in range(4,6):
    for j in ('B'):
        ws10[j+str(i)].style = format_comma_no_decimal

for i in range(4,6):
    for j in ('C'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
for i in range(4,6):
    for j in ('D'):
        ws10[j+str(i)].style = format_comma_no_decimal
    
for i in range(4,6):
    for j in ('E'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
        
for i in range(4,6):
    for j in ('F'):
        ws10[j+str(i)].style = format_comma_one_decimal
        
        
for i in range(4,6):
    for j in ('G'):
        ws10[j+str(i)].style = format_comma_one_decimal
        
        
for i in range(4,6):
    for j in ('H'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
                
for i in range(4,6):
    for j in ('I'):
        ws10[j+str(i)].style = format_comma_one_decimal

        
ws10['B6'].style = format_percent_no_decimal        
ws10['C6'].style = format_percent_no_decimal        
ws10['D6'].style = format_percent_no_decimal        
ws10['E6'].style = format_percent_no_decimal        
ws10['F6'].style = format_percent_no_decimal        
ws10['G6'].style = format_percent_no_decimal  
ws10['H6'].style = format_percent_no_decimal  
ws10['I6'].style = format_percent_no_decimal  

#update

ws10['A7'] = '** Per X days is meant to equivailize metrics for the same length as campaign period over prior 52 weeks'

ws10['A8'] = '** Brand dollars and brand units are aggregated for the entire pre 52 weeks'


names = list(unique_trips_by_period['analysis_periods'])

for i in range(len(names)):
    
    if names[i] == 'VMR Pre-Period':


        results_participants_2 = results_participants_02 
        results_participants_2.columns = results_participants_2.iloc[0]
        results_participants_2 = results_participants_2[1:]


        for r in dataframe_to_rows(results_participants_2, index=False, header=True):
            ws10.append(r)

        for cell in ws10[9]:
            cell.style = 'Accent1'
            cell.alignment=Alignment(wrap_text=True, horizontal="center")    


        for i in range(10,12):
            for j in ('B'):
                ws10[j+str(i)].style = format_comma_no_decimal

        for i in range(10,12):
            for j in ('C'):
                ws10[j+str(i)].style = format_dollars_two_decimals

        for i in range(10,12):
            for j in ('D'):
                ws10[j+str(i)].style = format_comma_no_decimal

        for i in range(10,12):
            for j in ('E'):
                ws10[j+str(i)].style = format_dollars_two_decimals


        for i in range(10,12):
            for j in ('F'):
                ws10[j+str(i)].style = format_comma_one_decimal


        for i in range(10,12):
            for j in ('G'):
                ws10[j+str(i)].style = format_comma_one_decimal


        for i in range(10,12):
            for j in ('H'):
                ws10[j+str(i)].style = format_dollars_two_decimals


        for i in range(10,12):
            for j in ('I'):
                ws10[j+str(i)].style = format_comma_one_decimal



        ws10['B12'].style = format_percent_no_decimal        
        ws10['C12'].style = format_percent_no_decimal        
        ws10['D12'].style = format_percent_no_decimal        
        ws10['E12'].style = format_percent_no_decimal        
        ws10['F12'].style = format_percent_no_decimal
        ws10['G12'].style = format_percent_no_decimal
        ws10['H12'].style = format_percent_no_decimal
        ws10['I12'].style = format_percent_no_decimal

    else:
        continue

        
#-------

ws10['A14'] = ''

names = list(unique_trips_by_period['analysis_periods'])

for i in range(len(names)):
    
    if names[i] ==  'Prior 26wk':
        results_participants_4 = results_participants_04 
        results_participants_4.columns = results_participants_4.iloc[0]
        results_participants_4 = results_participants_4[1:]


        for r in dataframe_to_rows(results_participants_4, index=False, header=True):
            ws10.append(r)

        for cell in ws10[15]:
            cell.style = 'Accent1'
            cell.alignment=Alignment(wrap_text=True, horizontal="center")
            
    else:
        continue



for i in range(16,18):
    for j in ('B'):
        ws10[j+str(i)].style = format_comma_no_decimal

for i in range(16,18):
    for j in ('C'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
for i in range(16,18):
    for j in ('D'):
        ws10[j+str(i)].style = format_comma_no_decimal

            
for i in range(16,18):
    for j in ('E'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
        
for i in range(16,18):
    for j in ('F'):
        ws10[j+str(i)].style = format_comma_one_decimal
        
        
for i in range(16,18):
    for j in ('G'):
        ws10[j+str(i)].style = format_comma_one_decimal
        
        
for i in range(16,18):
    for j in ('H'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
                
for i in range(16,18):
    for j in ('I'):
        ws10[j+str(i)].style = format_comma_one_decimal

        
ws10['B18'].style = format_percent_no_decimal        
ws10['C18'].style = format_percent_no_decimal        
ws10['D18'].style = format_percent_no_decimal        
ws10['E18'].style = format_percent_no_decimal        
ws10['F18'].style = format_percent_no_decimal  
ws10['G18'].style = format_percent_no_decimal
ws10['H18'].style = format_percent_no_decimal
ws10['I18'].style = format_percent_no_decimal

ws10['A19'] = ''
ws10['A20'] = ''
    

if unique_trips_by_period.loc[0, 'analysis_periods'] == 'YAGO Period': 

    results_participants_3 = results_participants_03 
    results_participants_3.columns = results_participants_3.iloc[0]
    results_participants_3 = results_participants_3[1:]


    for r in dataframe_to_rows(results_participants_3, index=False, header=True):
        ws10.append(r)

    for cell in ws10[21]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    


for i in range(22,25):
    for j in ('B'):
        ws10[j+str(i)].style = format_comma_no_decimal

for i in range(22,25):
    for j in ('C'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
for i in range(22,25):
    for j in ('D'):
        ws10[j+str(i)].style = format_comma_no_decimal

    
for i in range(22,25):
    for j in ('E'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
        
for i in range(22,25):
    for j in ('F'):
        ws10[j+str(i)].style = format_comma_one_decimal
        
        
for i in range(22,25):
    for j in ('G'):
        ws10[j+str(i)].style = format_comma_one_decimal
        
        
for i in range(22,25):
    for j in ('H'):
        ws10[j+str(i)].style = format_dollars_two_decimals
        
                
for i in range(22,25):
    for j in ('I'):
        ws10[j+str(i)].style = format_comma_one_decimal


ws10['B24'].style = format_percent_no_decimal        
ws10['C24'].style = format_percent_no_decimal        
ws10['D24'].style = format_percent_no_decimal        
ws10['E24'].style = format_percent_no_decimal        
ws10['F24'].style = format_percent_no_decimal         
ws10['G24'].style = format_percent_no_decimal  
ws10['H24'].style = format_percent_no_decimal  
ws10['I24'].style = format_percent_no_decimal  

In [152]:
##### WORKSHEET 2 ####

if category_show_parameter == True:

    ws11 = wb.create_sheet(title='Category Share')
    ws11.sheet_view.showGridLines = False

    ws11['A1'] = 'Category Share'

    ws11['A2'] = 'Metrics based on qualifying IDs'


    df_dollar_share = {

        '': ['YAGO Period', 'VMR Period', 'Post Period'],
        'Dollar Share': [share_dollar_yago, share_dollar_vmr, share_dollar_post_4wk]

    }

    df_dollar_share = pd.DataFrame(df_dollar_share)


    df_unit_share = {

        '': ['YAGO Period', 'VMR Period', 'Post Period'],
        'Unit Share': [share_units_yago, share_units_vmr, share_units_post_4wk]

    }

    df_unit_share = pd.DataFrame(df_unit_share)



    for r in dataframe_to_rows(df_dollar_share, index=False, header=True):
        ws11.append(r)

    for cell in ws11[3]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    


    for cell in ws11[1]:
        cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )


    ws11.column_dimensions['A'].width = 40        


    ws11['A7'] = ''
    ws11['A8'] = ''


    for r in dataframe_to_rows(df_unit_share, index=False, header=True):
        ws11.append(r)

    for cell in ws11[9]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    



    #update

    for i in range(4,7):
        for j in ('B'):
            ws11[j+str(i)].style = format_percent_no_decimal


    for i in range(10,15):
        for j in ('B'):
            ws11[j+str(i)].style = format_percent_no_decimal #check this



##### WORKSHEET 3 ####

ws3 = wb.create_sheet(title='Per Trip Metrics')
ws3.sheet_view.showGridLines = False

ws3['A1'] = 'Dollars per Trip and Units per Trip for each period by segment among qualified trackable IDs:'

ws3['A2'] = ''

if vmr_trend_yago_summary.loc[0, 'Analysis Periods'] == 'YAGO Period':

    ws3['A3'] = 'Comparison VMR period vs YAGO period:'

    for r in dataframe_to_rows(comparison_yago, index=False, header=True):
        ws3.append(r)

    for cell in ws3[4]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    

    for i in (range(5,5+len(comparison_yago))):
        for j in ('B', 'C'):
            ws3[j+str(i)].style = format_dollars_two_decimals

    for i in (range(5,5+len(comparison_yago))):
        for j in ('D', 'G'):
            ws3[j+str(i)].style = format_percent_no_decimal

    for i in (range(5,5+len(comparison_yago))):
        for j in ('E', 'F'):
            ws3[j+str(i)].style = format_comma_one_decimal


    ################################################################################################    

    next_row = 4+len(comparison_yago)+1

    ws3['A'+str(next_row)] = 'Comparison VMR period vs Pre-period period:'

    for r in dataframe_to_rows(comparison_preperiod, index=False, header=True):
        ws3.append(r)

    for cell in ws3[next_row+1]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")   

    for i in (range(next_row+1,len(comparison_preperiod)+next_row+3)):
        for j in ('B', 'C'):
            ws3[j+str(i)].style = format_dollars_two_decimals

    for i in (range(next_row+1,len(comparison_preperiod)+next_row+3)):
        for j in ('D', 'G'):
            ws3[j+str(i)].style = format_percent_no_decimal

    for i in (range(next_row+1,len(comparison_preperiod)+next_row+3)):
        for j in ('E', 'F'):
            ws3[j+str(i)].style = format_comma_one_decimal
            
    for cell in ws3[next_row+1]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")   
    

            
else:
    
    ws3['A3'] = 'Comparison VMR period vs Pre-period period:'

    for r in dataframe_to_rows(comparison_preperiod, index=False, header=True):
        ws3.append(r)

    for cell in ws3[4]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")   

    for i in (range(5,5+len(comparison_preperiod))):
        for j in ('B', 'C'):
            ws3[j+str(i)].style = format_dollars_two_decimals

    for i in (range(5,5+len(comparison_preperiod))):
        for j in ('D', 'G'):
            ws3[j+str(i)].style = format_percent_no_decimal

    for i in (range(5,5+len(comparison_preperiod))):
        for j in ('E', 'F'):
            ws3[j+str(i)].style = format_comma_one_decimal

    
    
    ################################################################################################


for cell in ws3[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

    
ws3.column_dimensions['A'].width = 40        

for col_name in ['B','C']:
    for cols in ws3[col_name]:
        ws3.column_dimensions[col_name].width = 23       

for col_name in ['D','E','F','G']:
    for cols in ws3[col_name]:
        ws3.column_dimensions[col_name].width = 18 
    

In [153]:
##### WORKSHEET 4 ####

ws4 = wb.create_sheet(title='Pre-Period Profile')
ws4.sheet_view.showGridLines = False

ws4['A1'] = 'Count of New Brand Buyers:'

ws4['B1'] = brand_consump.loc[0, '% Brand IDs'] * total_reward_ids
ws4['B1'].style = format_comma_no_decimal
ws4['B1'].alignment = Alignment(wrap_text=True, horizontal="left" )
ws4['B1'].font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

ws4['A2'] = '**New Brand Buyer Count is projected off total reward IDs'
ws4['A3'] = ''

if dominance != 'Kroger' :
    ws4['A4'] = 'Pre-52 week brand and category profile among qualifying IDs'
else:
    ws4['A4'] = 'Pre-26 week brand and category profile among qualifying IDs'
    
ws4['A5'] = '**Profile groups based on DOLLAR spend'
ws4['A6'] = ''


for r in dataframe_to_rows(brand_consump[['Brand Group', '% Brand IDs']], index=False, header=True):
    ws4.append(r)

for cell in ws4[7]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    

for i in (range(8,14)):
    for j in ('B'):
        ws4[j+str(i)].style = format_percent_no_decimal

        
###################################################
        
if category_show_parameter == True:


    ws4['A14'] = ''

    for r in dataframe_to_rows(cat_consump[['Category Group', '% Category IDs']], index=False, header=True):
        ws4.append(r)

    for cell in ws4[15]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    


    for i in (range(16,22)):
        for j in ('B'):
            ws4[j+str(i)].style = format_percent_no_decimal


###################################################################
        
ws4['A1'].font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )
ws4['A4'].font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )
    
ws4.column_dimensions['A'].width = 40        

for col_name in ['B','C']:
    for cols in ws4[col_name]:
        ws4.column_dimensions[col_name].width = 23       

for col_name in ['D','E','F','G']:
    for cols in ws4[col_name]:
        ws4.column_dimensions[col_name].width = 18 
    

In [154]:
##### WORKSHEET 2 ####

ws12 = wb.create_sheet(title='Repeat Rate')
ws12.sheet_view.showGridLines = False

ws12['A1'] = 'Repeat Rate'

ws12['A2'] = 'Metrics based on qualifying IDs'


df_repeat_rate = {
    
    '': ['Total Participants', 'New Buyers Participants', 'Existing Buyers Participants'],
    'Repeat Rate': [pct_repurch, pct_repurch_new_buyers, pct_repurch_existing_buyers]
    
}

df_repeat_rate = pd.DataFrame(df_repeat_rate)


for r in dataframe_to_rows(df_repeat_rate, index=False, header=True):
    ws12.append(r)

for cell in ws12[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    

for cell in ws12[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

    
ws12.column_dimensions['A'].width = 40        


for i in (range(4,len(df_repeat_rate)+4)):
    for j in ('B'):
        ws12[j+str(i)].style = format_percent_no_decimal
        

ADD NEXT FOR TABLE SLIDE 2

In [155]:
##### WORKSHEET 2 ####

ws13 = wb.create_sheet(title='Trends by Segment')
ws13.sheet_view.showGridLines = False

ws13['A1'] = 'Trends by Segment'

ws13['A2'] = 'Metrics based on qualifying IDs'

ws13['A3'] = ''

ws13['A4'] = 'Campaign vs Prior 52 wks'


#first table: Prior 52 wk

df_trend_by_segment_1 = table_slide_1
df_trend_by_segment_1.columns = df_trend_by_segment_1.iloc[0]
df_trend_by_segment_1 = df_trend_by_segment_1[1:].reset_index(drop=True)


for r in dataframe_to_rows(df_trend_by_segment_1, index=False, header=True):
    ws13.append(r)

for cell in ws13[5]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    
    
for cell in ws13[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

ws13.column_dimensions['A'].width = 40        


#formatting first table

for i in (range(6,len(df_trend_by_segment_1)+7)):
    for j in ('B'):
        ws13[j+str(i)].style = format_dollars_two_decimals
                
for i in (range(6,len(df_trend_by_segment_1)+7)):
    for j in ('C'):
        ws13[j+str(i)].style = format_dollars_two_decimals
                
for i in (range(6,len(df_trend_by_segment_1)+7)):
    for j in ('D'):
        ws13[j+str(i)].style = format_percent_no_decimal

for i in (range(6,len(df_trend_by_segment_1)+7)):
    for j in ('E'):
        ws13[j+str(i)].style = format_comma_one_decimal
        
for i in (range(6,len(df_trend_by_segment_1)+7)):
    for j in ('F'):
        ws13[j+str(i)].style = format_comma_one_decimal
        
for i in (range(6,len(df_trend_by_segment_1)+7)):
    for j in ('G'):
        ws13[j+str(i)].style = format_percent_no_decimal
        
        
#------------------------------------------------------------------------

next_cell = len(df_trend_by_segment_1)+7+1


names = list(unique_trips_by_period['analysis_periods'])

for i in range(len(names)):
    
    if names[i] == 'VMR Pre-Period':


        ws13['A'+str(next_cell)] = 'Campaign vs Pre Period'


        #Second table: Pre Period

        df_trend_by_segment_2 = table_slide_2
        df_trend_by_segment_2.columns = df_trend_by_segment_2.iloc[0]
        df_trend_by_segment_2 = df_trend_by_segment_2[1:].reset_index(drop=True)


        for r in dataframe_to_rows(df_trend_by_segment_2, index=False, header=True):
            ws13.append(r)

        for cell in ws13[next_cell+1]:
            cell.style = 'Accent1'
            cell.alignment=Alignment(wrap_text=True, horizontal="center")    



        #formatting second table

        for i in (range(next_cell+2,len(df_trend_by_segment_2)+next_cell+3)):
            for j in ('B'):
                ws13[j+str(i)].style = format_dollars_two_decimals

        for i in (range(next_cell+2,len(df_trend_by_segment_2)+next_cell+3)):
            for j in ('C'):
                ws13[j+str(i)].style = format_dollars_two_decimals

        for i in (range(next_cell+2,len(df_trend_by_segment_2)+next_cell+3)):
            for j in ('D'):
                ws13[j+str(i)].style = format_percent_no_decimal

        for i in (range(next_cell+2,len(df_trend_by_segment_2)+next_cell+3)):
            for j in ('E'):
                ws13[j+str(i)].style = format_comma_one_decimal

        for i in (range(next_cell+2,len(df_trend_by_segment_2)+next_cell+3)):
            for j in ('F'):
                ws13[j+str(i)].style = format_comma_one_decimal

        for i in (range(next_cell+2,len(df_trend_by_segment_2)+next_cell+3)):
            for j in ('G'):
                ws13[j+str(i)].style = format_percent_no_decimal
                
    else:
        
        df_trend_by_segment_2 = df_trend_by_segment_1


        
#------------------------------------------------------------------------

#Third table: YAGO Period

if unique_trips_by_period.loc[0, 'analysis_periods'] == 'YAGO Period':
    
    next_cell_b = len(df_trend_by_segment_2)+next_cell+3+1

    ws13['A'+str(next_cell_b)] = 'Campaign vs YAGO'


    df_trend_by_segment = table_slide
    df_trend_by_segment.columns = df_trend_by_segment.iloc[0]
    df_trend_by_segment = df_trend_by_segment[1:].reset_index(drop=True)


    for r in dataframe_to_rows(df_trend_by_segment, index=False, header=True):
        ws13.append(r)

    for cell in ws13[next_cell_b+1]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    


    #formatting third table

    for i in (range(next_cell_b+2,len(df_trend_by_segment)+next_cell_b+4)):
        for j in ('B'):
            ws13[j+str(i)].style = format_dollars_two_decimals

    for i in (range(next_cell_b+2,len(df_trend_by_segment)+next_cell_b+4)):
        for j in ('C'):
            ws13[j+str(i)].style = format_dollars_two_decimals

    for i in (range(next_cell_b+2,len(df_trend_by_segment)+next_cell_b+4)):
        for j in ('D'):
            ws13[j+str(i)].style = format_percent_no_decimal

    for i in (range(next_cell_b+2,len(df_trend_by_segment)+next_cell_b+4)):
        for j in ('E'):
            ws13[j+str(i)].style = format_comma_one_decimal

    for i in (range(next_cell_b+2,len(df_trend_by_segment)+next_cell_b+4)):
        for j in ('F'):
            ws13[j+str(i)].style = format_comma_one_decimal

    for i in (range(next_cell_b+2,len(df_trend_by_segment)+next_cell_b+4)):
        for j in ('G'):
            ws13[j+str(i)].style = format_percent_no_decimal


    for cell in ws13[next_cell_b+1]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    

In [156]:
##### WORKSHEET 5 ####

if nbr_segments > 1 :

    ws5 = wb.create_sheet(title='Trip Segment Combo')
    ws5.sheet_view.showGridLines = False

    ws5['A1'] = 'Brand combinations purchased on the VMR qualified Trip among qualifying IDs:'

    ws5['A2'] = ''

    ws5['A3'] = '% of Trips including 2 or more segments:'
    ws5['A3'].font = Font(bold=True, color = '000000', size = 11, name = 'Calibri' )
    

    nbr_trips_two_segments = (df_vmr_period_combos_all.loc[(df_vmr_period_combos_all['No. of Segments']>=2), ['Trips']].sum())
    nbr_total_trips = (df_vmr_period_combos_all['Trips'].sum())

    ws5['B3'] = float(nbr_trips_two_segments/nbr_total_trips)
    ws5['B3'].style = format_percent_no_decimal
    ws5['B3'].alignment = Alignment(wrap_text=True, horizontal="left" )

    ws5['A4'] = ''

    for r in dataframe_to_rows(df_vmr_period_combos_all_out[["Segment", "Pct Of Trips", "No. of Segments"]], index=False, header=True):
        ws5.append(r)

    for cell in ws5[5]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    


    for i in (range(6,len(df_vmr_period_combos_all_out)+6)):
        for j in ('B'):
            ws5[j+str(i)].style = format_percent_no_decimal


    for cell in ws5[1]:
        cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )


    ws5.column_dimensions['A'].width = 40        

    for col_name in ['B','C']:
        for cols in ws5[col_name]:
            ws5.column_dimensions[col_name].width = 23       

    for col_name in ['D','E','F','G']:
        for cols in ws5[col_name]:
            ws5.column_dimensions[col_name].width = 18 


In [157]:
##### WORKSHEET 6 ####

if nbr_segments > 1 :

    ws6 = wb.create_sheet(title='New-Existing Segments')
    ws6.sheet_view.showGridLines = False

    ws6['A1'] = 'Segment analysis of purchases on VMR qualifying trip among qualifying IDs'
    ws6['A2'] = ''


    for r in dataframe_to_rows(df_plus_one[['','Pct Of Shoppers']], index=False, header=True):
        ws6.append(r)

    for cell in ws6[3]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    


    for i in (range(4,6)):
        for j in ('B'):
            ws6[j+str(i)].style = format_percent_no_decimal



    #################################################################
    ws6['A6'] = ''

    for r in dataframe_to_rows(df_new_existing_combos[['','Pct Of Shoppers']], index=False, header=True):
        ws6.append(r)

    for cell in ws6[7]:
        cell.style = 'Accent1'
        cell.alignment=Alignment(wrap_text=True, horizontal="center")    


    for i in (range(8,11)):
        for j in ('B'):
            ws6[j+str(i)].style = format_percent_no_decimal


    ###################################################################

    for cell in ws6[1]:
        cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )


    ws6.column_dimensions['A'].width = 40        

    for col_name in ['B','C']:
        for cols in ws6[col_name]:
            ws6.column_dimensions[col_name].width = 23       

    for col_name in ['D','E','F','G']:
        for cols in ws6[col_name]:
            ws6.column_dimensions[col_name].width = 18 


In [158]:
##### WORKSHEET 7 ####

ws7 = wb.create_sheet(title='Baskets Sizes')
ws7.sheet_view.showGridLines = False

ws7['A1'] = 'Baskets sizes during VMR and redemption period (All trips - No ID Filter):'

ws7['A2'] = ''

for r in dataframe_to_rows(df_vmr_ttls_basket[['Analysis period','Average Basket Size']], index=False, header=True):
    ws7.append(r)

for cell in ws7[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    

for i in (range(4,6)):
    for j in ('B'):
        ws7[j+str(i)].style = format_dollars_two_decimals

for i in (range(7,9)):
    for j in ('B'):
        ws7[j+str(i)].style = format_dollars_two_decimals


ws7['B6'].style = format_percent_no_decimal
ws7['B9'].style = format_percent_no_decimal

####################################################

ws7['A10'] = ''

if nbr_redemp_promoted['trips'].values[0] != 0:

    ws7['A11'] = '% redemption trips including promoted product'
    ws7['A11'].font = Font(bold=True, color = '000000', size = 11, name = 'Calibri' )
    ws7['B11'] = str(round((nbr_redemp_promoted['trips'].values[0]/nbr_redemp_trips['trips'].values[0])*100,0)) + '%'
    ws7['B11'].alignment=Alignment(wrap_text=True, horizontal="right")    



for cell in ws7[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )


ws7.column_dimensions['A'].width = 41.09        

for col_name in ['B','C']:
    for cols in ws7[col_name]:
        ws7.column_dimensions[col_name].width = 23       

for col_name in ['D','E','F','G']:
    for cols in ws7[col_name]:
        ws7.column_dimensions[col_name].width = 18 


In [159]:
##### WORKSHEET 2 ####

ws2 = wb.create_sheet(title='Sales by Segment')
ws2.sheet_view.showGridLines = False

ws2['A1'] = 'Total Dollars and Units purchased on qualifying trips  (All trips - No ID filter):'

ws2['A2'] = ''

for r in dataframe_to_rows(total_vmr_details_by_brand, index=False, header=True):
    ws2.append(r)

for cell in ws2[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")    
    

for cell in ws2[1]:
    cell.font = Font(bold=True, color = '4F81BD', size = 15, name = 'Calibri' )

    
ws2.column_dimensions['A'].width = 40        

for col_name in ['B','C']:
    for cols in ws2[col_name]:
        ws2.column_dimensions[col_name].width = 23       

for col_name in ['D','E','F','G']:
    for cols in ws2[col_name]:
        ws2.column_dimensions[col_name].width = 18 
    

    
for i in (range(4,len(total_vmr_details_by_brand)+4)):
    for j in ('B'):
        ws2[j+str(i)].style = format_comma_no_decimal
        
        
for i in (range(4,len(total_vmr_details_by_brand)+4)):
    for j in ('C'):
        ws2[j+str(i)].style = format_dollars_no_decimals

        
for i in (range(4,len(total_vmr_details_by_brand)+4)):
    for j in ('D','E'):
        ws2[j+str(i)].style = format_percent_no_decimal


In [160]:
##### WORKSHEET 14 ####

ws15 = wb.create_sheet(title='Custom Metrics')
ws15.sheet_view.showGridLines = False

if dominance == 'Kroger':

    ws15['A1'] = 'ID Summaries: Based on 26 week Pre-Period prior to the VMR Event'
else:
    
    ws15['A1'] = 'ID Summaries: Based on 52 week Pre-Period prior to the VMR Event'
    
    
ws15['A2'] = ''

for r in dataframe_to_rows(nestle_merged_df
,index=False, header=True):
    ws15.append(r)

for cell in ws15[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")   
    
ws15['A8'] = ''

ws15['A9'] = 'Lift Calculations'

for r in dataframe_to_rows(lift_metrics
,index=False, header=True):
    ws15.append(r)

for cell in ws15[10]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")   
    


In [161]:

ws16 = wb.create_sheet(title='Dominance Report')
ws16.sheet_view.showGridLines = False


ws16['A1'] = 'Dominance Report'

ws16['A2'] = ''

for r in dataframe_to_rows(df_parent_pivot_2
,index=False, header=True):
    ws16.append(r)

for cell in ws16[3]:
    cell.style = 'Accent1'
    cell.alignment=Alignment(wrap_text=True, horizontal="center")   

In [162]:
wb.save(filename = out_name)    

In [163]:
print('If this is printed, The Excel output has been saved sucessfully!')

If this is printed, The Excel output has been saved sucessfully!


## **17) CREATING POWER POINT OUTPUT:**

In [164]:
from pptx import Presentation
from pptx.chart.data import ChartData
from pptx.enum.chart import XL_CHART_TYPE
from pptx.enum.chart import XL_LEGEND_POSITION
from pptx.util import Pt
from pptx.dml.color import RGBColor
from pptx.enum.chart import XL_LABEL_POSITION
from pptx.util import Inches
from pptx.enum.text import PP_ALIGN
from pptx.enum.chart import XL_TICK_MARK
from pptx.chart.data import CategoryChartData
from pptx.util import Inches
from pptx.enum.dml import MSO_LINE

#len(prs.slide_layouts)

sld = prs.slides.add_slide(prs.slide_layouts[1])

for shape in sld.shapes:
    if shape.is_placeholder:  # Check if the shape is a placeholder
        placeholder_idx = shape.placeholder_format.idx
        placeholder_type = shape.placeholder_format.type
        print(f"  - Placeholder Index: {placeholder_idx}, Type: {placeholder_type}")


In [165]:
#Loading Power Point template to build the final deck

#prs = Presentation(template_path)
prs = Presentation("/analytical_services/Public/moricejuan/VMR_pptx_tests/VMR_Scorecard_Template - new.pptx")

In [166]:
#Adding Slide 0: Catalina Front Page

sld_0 = prs.slides.add_slide(prs.slide_layouts[0])

In [167]:
#Adding Slide 1: General Metrics

if dominance != 'Kroger':

    sld_1 = prs.slides.add_slide(prs.slide_layouts[1])

    sld_1.placeholders[10].text = f'{str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")}'


    if nbr_retailers < 3:
        sld_1.placeholders[11].text = f'{retailers}'
    else:
        sld_1.placeholders[11].text = f'National'

    new_brand_buyers_proj = brand_consump.loc[0, '% Brand IDs'] * total_reward_ids


    sld_1.placeholders[13].text = f'${"{:,}".format(round(dollars_moved_vmr))}'

    sld_1.placeholders[14].text = f'{"{:.0%}".format(change_dpt)}'

    sld_1.placeholders[15].text = f'{"{:,}".format(round(new_brand_buyers_proj))}'

    sld_1.placeholders[16].text = f'{"{:.0%}".format(pct_repurch)}'

    sld_1.placeholders[12].text = f'Drive sales of Promoted Brand with a spend {min_threshold_statement} threshold next shopping trip offer'

    if dominance == 'Kroger':
        sld_1.placeholders[17].text = f'BL Dominance: Kroger over 45%'
    elif dominance == 'Retailer':
        sld_1.placeholders[17].text = f'BL Dominance: {dominant_retailer} over 70%'
    else:
        sld_1.placeholders[17].text = f'BL Dominance: No dominance.'



In [168]:
if dominance == 'Kroger':

    sld_1 = prs.slides.add_slide(prs.slide_layouts[18])

    sld_1.placeholders[10].text = f'{str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")}'


    if nbr_retailers < 3:
        sld_1.placeholders[11].text = f'{retailers}'
    else:
        sld_1.placeholders[11].text = f'National'

    new_brand_buyers_proj = brand_consump.loc[0, '% Brand IDs'] * total_reward_ids


    sld_1.placeholders[13].text = f'${"{:,}".format(round(dollars_moved_vmr))}'

    sld_1.placeholders[14].text = f'{"{:.0%}".format(change_dpt_26)}'

    sld_1.placeholders[15].text = f'{"{:,}".format(round(new_brand_buyers_proj))}'

    sld_1.placeholders[16].text = f'{"{:.0%}".format(pct_repurch)}'

    sld_1.placeholders[12].text = f'Drive sales of Promoted Brand with a spend {min_threshold_statement} threshold next shopping trip offer'

    if dominance == 'Kroger':
        sld_1.placeholders[17].text = f'BL Dominance: Kroger over 45%'
    elif dominance == 'Retailer':
        sld_1.placeholders[17].text = f'BL Dominance: {dominant_retailer} over 70%'
    else:
        sld_1.placeholders[17].text = f'BL Dominance: No dominance.'



In [169]:
if dominance != 'Kroger':

    if vmr_trend_yago_summary.loc[0, 'Analysis Periods'] == 'YAGO Period':


        #Adding Slide 2: VMR Campaign Trends DOLLARS WITH VERTICAL LINES 

        sld_2 = prs.slides.add_slide(prs.slide_layouts[2])

        #Chart 2.1: Trend

        # X and Y values for Current Dollars:

        reward_dates = [df_promo_summary['Actual Start'].min(), df_promo_summary['Actual Start'].min(), df_promo_summary['Actual End'].max(), df_promo_summary['Actual End'].max()]
        dates_trend  = list(vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).cal_sun_wk_ending_dt)
        curr_dol_val = list(vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).dollar_sales.values)
        yago_dol_val = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'YAGO Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).dollar_sales
        
        print(len(dates_trend))
        print(len(curr_dol_val))
        print(len(yago_dol_val))

        curr_dol_val_cp = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).dollar_sales
        dates_trend_cp  = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).cal_sun_wk_ending_dt

        if len(yago_dol_val) != len(curr_dol_val):
            if len(yago_dol_val) > len(curr_dol_val):
                yago_dol_val = list(yago_dol_val.head(len(curr_dol_val)).values)
            else:
                curr_dol_val = list(curr_dol_val_cp.head(len(yago_dol_val)).values)
                dates_trend = list(dates_trend_cp.head(len(yago_dol_val)).values)

        print(len(dates_trend))
        print(len(curr_dol_val))
        print(len(yago_dol_val))

        #Creating Df with all the dates
        df_trend = pd.DataFrame({'cal_sun_wk_ending_dt': dates_trend, 'dollar_sales': curr_dol_val, 'dollar_sales_yago': yago_dol_val})
        #print(df_trend)

        #Adding reward period dates
        reward_dates = [df_promo_summary['Actual Start'].min(), df_promo_summary['Actual Start'].min(), df_promo_summary['Actual End'].max(), df_promo_summary['Actual End'].max()] 
        new_data = pd.DataFrame({'cal_sun_wk_ending_dt': reward_dates, 'dollar_sales': ['#N/A','#N/A','#N/A','#N/A'], 'dollar_sales_yago': ['#N/A','#N/A','#N/A','#N/A'] })

        # Insert the new line at the end of the DataFrame
        df_trend = df_trend.append(new_data, ignore_index=True)

        df_trend = df_trend.sort_values(by=['cal_sun_wk_ending_dt'], ascending = True)
        df_trend = df_trend.reset_index(drop=True)
        #print(df_trend)



        #VERTICAL ONE LINE

        vertical_one = []
        counter = 0
        for i in range(len(df_trend['cal_sun_wk_ending_dt'])):
            if df_trend['cal_sun_wk_ending_dt'][i] == df_promo_summary['Actual Start'].min():
                if counter == 0:
                    vertical_one.append(0)
                    counter += 1
                else:
                    vertical_one.append(max(curr_dol_val) + max(curr_dol_val)/4)
                    counter = 0
            else:
                vertical_one.append('')

        #print(vertical_one)


        #VERTICAL SECOND LINE

        vertical_second = []
        counter = 0
        for i in range(len(df_trend['cal_sun_wk_ending_dt'])):
            if df_trend['cal_sun_wk_ending_dt'][i] == df_promo_summary['Actual End'].max():
                if counter == 0:
                    vertical_second.append(0)
                    counter += 1
                else:
                    vertical_second.append(max(curr_dol_val) + max(curr_dol_val)/4)
                    counter = 0
            else:
                vertical_second.append('')

        #print(vertical_second)



        chart_data = CategoryChartData()
        chart_data.categories = df_trend['cal_sun_wk_ending_dt']
        chart_data.add_series('YAGO Period', df_trend['dollar_sales_yago'], '$#,##0')
        chart_data.add_series('VMR Period', df_trend['dollar_sales'], '$#,##0')
        chart_data.add_series('Campaign Start', vertical_one, '$#,##0')
        chart_data.add_series('Campaign End', vertical_second, '$#,##0')


        chart2_1 = sld_2.placeholders[10].insert_chart(XL_CHART_TYPE.LINE, chart_data).chart


        chart2_1.font.size  = Pt(12)
        chart2_1.value_axis.has_major_gridlines = False
        chart2_1.has_legend = True
        chart2_1.legend.position = XL_LEGEND_POSITION.BOTTOM
        chart2_1.legend.include_in_layout = False
        chart2_1.value_axis.major_tick_mark = XL_TICK_MARK.OUTSIDE

        widths_sizes = [0.04,0.04,0.02,0.02]

        series = chart2_1.series
        for i, serie in enumerate(chart2_1.series):
            line_format = serie.format.line
            line_format.width = Inches(widths_sizes[i])  

            if i!=0 and i!=1:
                line_format.dash_style = MSO_LINE.SQUARE_DOT


        #Chart 2.2: Bar Chart 1

        chart_data = ChartData()
        chart_data.categories = vmr_trend_yago_summary['Analysis Periods']
        chart_data.add_series('', vmr_trend_yago_summary['Dollars per Trip'], '$#,##0.00')


        chart2_2 = sld_2.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart2_2.font.size  = Pt(12)
        chart2_2.category_axis.tick_labels.font.bold = True
        chart2_2.value_axis.has_major_gridlines = False


        chart2_2.plots[0].has_data_labels = True
        data_labels_2 = chart2_2.plots[0].data_labels
        data_labels_2.font.size = Pt(14)
        data_labels_2.font.bold = True
        data_labels_2.number_format = '$#,##0.00'



        #Chart 2.3: Bar Chart 2

        pct_chg_dol_share = (share_vmr_dol - share_yago_dol)*100

        chart_data = ChartData()
        chart_data.categories = vmr_trend_yago_summary['Analysis Periods']
        chart_data.add_series('', [share_yago_dol, share_vmr_dol], '0.0%')


        chart2_3 = sld_2.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart2_3.font.size  = Pt(12)
        chart2_3.category_axis.tick_labels.font.bold = True
        chart2_3.value_axis.has_major_gridlines = False


        chart2_3.plots[0].has_data_labels = True
        data_labels_3 = chart2_3.plots[0].data_labels
        data_labels_3.font.size = Pt(14)
        data_labels_3.font.bold = True
        data_labels_3.number_format = '0.0%'


        sld_2.placeholders[13].text = f'VMR Period {dollars_chg_yago}% chg vs YAGO'

        if dollars_per_trip_chg_yago > 0:    
            sld_2.placeholders[14].text = f'+{dollars_per_trip_chg_yago}%                       vs YAGO'
        else:
            sld_2.placeholders[14].text = f'{dollars_per_trip_chg_yago}%                       vs YAGO'

        if pct_chg_dol_share > 0:
            sld_2.placeholders[15].text = f'+{"{:.1f}".format(pct_chg_dol_share)} Pts vs YAGO' 
        else:
            sld_2.placeholders[15].text = f'{"{:.1f}".format(pct_chg_dol_share)} Pts vs YAGO' 


        sld_2.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nYAGO Period: {str(yago_print_start).replace("-","/")} - {str(yago_print_end).replace("-","/")}' 

        phr = sld_2.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


34
34
36
34
34
34


In [170]:
#Double checking the content of the dollars trend chart

x = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).cal_sun_wk_ending_dt
y = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'YAGO Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).dollar_sales.values
z = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).dollar_sales.values

print("The amount of dates that will be plot is: " + str(len(x)))
print("The number of dollar points that will be plot is: " + str(len(y)))
print("The number of dollar points from YAGO that will be plot is: " + str(len(z)))

The amount of dates that will be plot is: 34
The number of dollar points that will be plot is: 36
The number of dollar points from YAGO that will be plot is: 34


In [171]:
if dominance != 'Kroger':

    if vmr_trend_yago_summary.loc[0, 'Analysis Periods'] != 'YAGO Period':

        #Adding Slide 2: VMR Campaign Trends DOLLARS WITH VERTICAL LINES 

        sld_2 = prs.slides.add_slide(prs.slide_layouts[2])

        #Chart 2.1: Trend

        # X and Y values for Current Dollars:

        reward_dates = [df_promo_summary['Actual Start'].min(), df_promo_summary['Actual Start'].min(), df_promo_summary['Actual End'].max(), df_promo_summary['Actual End'].max()]
        dates_trend  = list(vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).cal_sun_wk_ending_dt)
        curr_dol_val = list(vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).dollar_sales.values)
        yago_dol_val = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'YAGO Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).dollar_sales

        print(len(dates_trend))
        print(len(curr_dol_val))
        print(len(yago_dol_val))
                


        #Creating Df with all the dates
        #df_trend = pd.DataFrame({'cal_sun_wk_ending_dt': dates_trend, 'dollar_sales': curr_dol_val, 'dollar_sales_yago': yago_dol_val})
        df_trend = pd.DataFrame({'cal_sun_wk_ending_dt': dates_trend, 'dollar_sales': curr_dol_val})

        #print(df_trend)

        #Adding reward period dates
        reward_dates = [df_promo_summary['Actual Start'].min(), df_promo_summary['Actual Start'].min(), df_promo_summary['Actual End'].max(), df_promo_summary['Actual End'].max()] 
        new_data = pd.DataFrame({'cal_sun_wk_ending_dt': reward_dates, 'dollar_sales': ['#N/A','#N/A','#N/A','#N/A'] })

        # Insert the new line at the end of the DataFrame
        df_trend = df_trend.append(new_data, ignore_index=True)

        df_trend = df_trend.sort_values(by=['cal_sun_wk_ending_dt'], ascending = True)
        df_trend = df_trend.reset_index(drop=True)
        #print(df_trend)



        #VERTICAL ONE LINE

        vertical_one = []
        counter = 0
        for i in range(len(df_trend['cal_sun_wk_ending_dt'])):
            if df_trend['cal_sun_wk_ending_dt'][i] == df_promo_summary['Actual Start'].min():
                if counter == 0:
                    vertical_one.append(0)
                    counter += 1
                else:
                    vertical_one.append(max(curr_dol_val) + max(curr_dol_val)/4)
                    counter = 0
            else:
                vertical_one.append('')

        #print(vertical_one)


        #VERTICAL SECOND LINE

        vertical_second = []
        counter = 0
        for i in range(len(df_trend['cal_sun_wk_ending_dt'])):
            if df_trend['cal_sun_wk_ending_dt'][i] == df_promo_summary['Actual End'].max():
                if counter == 0:
                    vertical_second.append(0)
                    counter += 1
                else:
                    vertical_second.append(max(curr_dol_val) + max(curr_dol_val)/4)
                    counter = 0
            else:
                vertical_second.append('')

        #print(vertical_second)



        chart_data = CategoryChartData()
        chart_data.categories = df_trend['cal_sun_wk_ending_dt']
        #chart_data.add_series('YAGO Period', df_trend['dollar_sales_yago'], '$#,##0')
        chart_data.add_series('VMR Period', df_trend['dollar_sales'], '$#,##0')
        chart_data.add_series('Campaign Start', vertical_one, '$#,##0')
        chart_data.add_series('Campaign End', vertical_second, '$#,##0')


        chart2_1 = sld_2.placeholders[10].insert_chart(XL_CHART_TYPE.LINE, chart_data).chart


        chart2_1.font.size  = Pt(12)
        chart2_1.value_axis.has_major_gridlines = False
        chart2_1.has_legend = True
        chart2_1.legend.position = XL_LEGEND_POSITION.BOTTOM
        chart2_1.legend.include_in_layout = False
        chart2_1.value_axis.major_tick_mark = XL_TICK_MARK.OUTSIDE

        widths_sizes = [0.04,0.04,0.02,0.02]

        series = chart2_1.series
        for i, serie in enumerate(chart2_1.series):
            line_format = serie.format.line
            line_format.width = Inches(widths_sizes[i])  

            if i!=0 and i!=1:
                line_format.dash_style = MSO_LINE.SQUARE_DOT


        #Chart 2.2: Bar Chart 1

        chart_data = ChartData()
        chart_data.categories = vmr_trend_yago_summary['Analysis Periods']
        chart_data.add_series('', vmr_trend_yago_summary['Dollars per Trip'], '$#,##0.00')


        chart2_2 = sld_2.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart2_2.font.size  = Pt(12)
        chart2_2.category_axis.tick_labels.font.bold = True
        chart2_2.value_axis.has_major_gridlines = False


        chart2_2.plots[0].has_data_labels = True
        data_labels_2 = chart2_2.plots[0].data_labels
        data_labels_2.font.size = Pt(14)
        data_labels_2.font.bold = True
        data_labels_2.number_format = '$#,##0.00'



        #Chart 2.3: Bar Chart 2

        pct_chg_dol_share = (share_vmr_dol - share_yago_dol)*100

        chart_data = ChartData()
        chart_data.categories = vmr_trend_yago_summary['Analysis Periods']
        chart_data.add_series('', [share_vmr_dol], '0.0%')


        chart2_3 = sld_2.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart2_3.font.size  = Pt(12)
        chart2_3.category_axis.tick_labels.font.bold = True
        chart2_3.value_axis.has_major_gridlines = False


        chart2_3.plots[0].has_data_labels = True
        data_labels_3 = chart2_3.plots[0].data_labels
        data_labels_3.font.size = Pt(14)
        data_labels_3.font.bold = True
        data_labels_3.number_format = '0.0%'


        sld_2.placeholders[13].text = f'VMR Period +{dollars_chg_yago}% chg vs YAGO'


        if dollars_per_trip_chg_yago > 0:    
            sld_2.placeholders[14].text = f'+{dollars_per_trip_chg_yago}%                       vs YAGO'
        else:
            sld_2.placeholders[14].text = f'{dollars_per_trip_chg_yago}%                       vs YAGO'

        if pct_chg_dol_share > 0:
            sld_2.placeholders[15].text = f'+{"{:.1f}".format(pct_chg_dol_share)} Pts vs YAGO' 
        else:
            sld_2.placeholders[15].text = f'{"{:.1f}".format(pct_chg_dol_share)} Pts vs YAGO' 



        sld_2.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nYAGO Period: {str(yago_print_start).replace("-","/")} - {str(yago_print_end).replace("-","/")}' 

        phr = sld_2.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


In [172]:
if dominance != 'Kroger':

    if vmr_trend_yago_summary.loc[0, 'Analysis Periods'] == 'YAGO Period':

        #Adding Slide 3: VMR Campaign UNITS

        sld_3 = prs.slides.add_slide(prs.slide_layouts[11])

        #Chart 2.1: Trend

        # X and Y values for Current Dollars:

        reward_dates = [df_promo_summary['Actual Start'].min(), df_promo_summary['Actual Start'].min(), df_promo_summary['Actual End'].max(), df_promo_summary['Actual End'].max()]
        dates_trend  = list(vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).cal_sun_wk_ending_dt)
        curr_dol_val = list(vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).units_sales.values)
        yago_dol_val = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'YAGO Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).units_sales

        len(dates_trend)
        len(curr_dol_val)
        len(yago_dol_val)


        curr_dol_val_cp = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).units_sales
        dates_trend_cp  = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).cal_sun_wk_ending_dt

        if len(yago_dol_val) != len(curr_dol_val):
            if len(yago_dol_val) > len(curr_dol_val):
                yago_dol_val = list(yago_dol_val.head(len(curr_dol_val)).values)
            else:
                curr_dol_val = list(curr_dol_val_cp.head(len(yago_dol_val)).values)
                dates_trend = list(dates_trend_cp.head(len(yago_dol_val)).values)

        print(len(dates_trend))
        print(len(curr_dol_val))
        print(len(yago_dol_val))

            #Creating Df with all the dates
        df_trend = pd.DataFrame({'cal_sun_wk_ending_dt': dates_trend, 'units_sales': curr_dol_val, 'units_sales_yago': yago_dol_val})
        #print(df_trend)

        #Adding reward period dates
        reward_dates = [df_promo_summary['Actual Start'].min(), df_promo_summary['Actual Start'].min(), df_promo_summary['Actual End'].max(), df_promo_summary['Actual End'].max()] 
        new_data = pd.DataFrame({'cal_sun_wk_ending_dt': reward_dates, 'units_sales': ['#N/A','#N/A','#N/A','#N/A'], 'units_sales_yago': ['#N/A','#N/A','#N/A','#N/A'] })

        # Insert the new line at the end of the DataFrame
        df_trend = df_trend.append(new_data, ignore_index=True)

        df_trend = df_trend.sort_values(by=['cal_sun_wk_ending_dt'], ascending = True)
        df_trend = df_trend.reset_index(drop=True)
        #print(df_trend)



        #VERTICAL ONE LINE

        vertical_one = []
        counter = 0
        for i in range(len(df_trend['cal_sun_wk_ending_dt'])):
            if df_trend['cal_sun_wk_ending_dt'][i] == df_promo_summary['Actual Start'].min():
                if counter == 0:
                    vertical_one.append(0)
                    counter += 1
                else:
                    vertical_one.append(max(curr_dol_val) + max(curr_dol_val)/4)
                    counter = 0
            else:
                vertical_one.append('')

        #print(vertical_one)


        #VERTICAL SECOND LINE

        vertical_second = []
        counter = 0
        for i in range(len(df_trend['cal_sun_wk_ending_dt'])):
            if df_trend['cal_sun_wk_ending_dt'][i] == df_promo_summary['Actual End'].max():
                if counter == 0:
                    vertical_second.append(0)
                    counter += 1
                else:
                    vertical_second.append(max(curr_dol_val) + max(curr_dol_val)/4)
                    counter = 0
            else:
                vertical_second.append('')

        #print(vertical_second)



        chart_data = CategoryChartData()
        chart_data.categories = df_trend['cal_sun_wk_ending_dt']
        chart_data.add_series('YAGO Period', df_trend['units_sales_yago'], '#,##0')
        chart_data.add_series('VMR Period', df_trend['units_sales'], '#,##0')
        chart_data.add_series('Campaign Start', vertical_one, '#,##0')
        chart_data.add_series('Campaign End', vertical_second, '#,##0')


        chart3_1 = sld_3.placeholders[10].insert_chart(XL_CHART_TYPE.LINE, chart_data).chart


        chart3_1.font.size  = Pt(12)
        chart3_1.value_axis.has_major_gridlines = False
        chart3_1.has_legend = True
        chart3_1.legend.position = XL_LEGEND_POSITION.BOTTOM
        chart3_1.legend.include_in_layout = False
        chart3_1.value_axis.major_tick_mark = XL_TICK_MARK.OUTSIDE

        widths_sizes = [0.04,0.04,0.02,0.02]

        series = chart3_1.series
        for i, serie in enumerate(chart3_1.series):
            line_format = serie.format.line
            line_format.width = Inches(widths_sizes[i])  

            if i!=0 and i!=1:
                line_format.dash_style = MSO_LINE.SQUARE_DOT


        #Chart 3.2: Bar Chart 1

        chart_data = ChartData()
        chart_data.categories = vmr_trend_yago_summary['Analysis Periods']
        chart_data.add_series('', vmr_trend_yago_summary['Units per Trip'], '#,##0.0')


        chart3_2 = sld_3.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart3_2.font.size  = Pt(12)
        chart3_2.category_axis.tick_labels.font.bold = True
        chart3_2.value_axis.has_major_gridlines = False


        chart3_2.plots[0].has_data_labels = True
        data_labels_2 = chart3_2.plots[0].data_labels
        data_labels_2.font.size = Pt(14)
        data_labels_2.font.bold = True
        data_labels_2.number_format = '#,##0.0'



        #Chart 3.3: Bar Chart 2


        pct_chg_unit_share = (share_vmr_unit - share_yago_unit)*100

        chart_data = ChartData()
        chart_data.categories = vmr_trend_yago_summary['Analysis Periods']
        chart_data.add_series('', [share_yago_unit, share_vmr_unit], '0.0%')


        chart3_3 = sld_3.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart3_3.font.size  = Pt(12)
        chart3_3.category_axis.tick_labels.font.bold = True
        chart3_3.value_axis.has_major_gridlines = False


        chart3_3.plots[0].has_data_labels = True
        data_labels_3 = chart3_3.plots[0].data_labels
        data_labels_3.font.size = Pt(14)
        data_labels_3.font.bold = True
        data_labels_3.number_format = '0.0%'


        sld_3.placeholders[13].text = f'VMR Period {units_chg_yago}% chg vs YAGO'


        if units_per_trip_chg_yago > 0:
            sld_3.placeholders[14].text = f'+{units_per_trip_chg_yago}%                     vs YAGO'
        else:
            sld_3.placeholders[14].text = f'{units_per_trip_chg_yago}%                     vs YAGO'

        if pct_chg_unit_share > 0:
            sld_3.placeholders[15].text = f'+{"{:.1f}".format(pct_chg_unit_share)} Pts vs YAGO' 
        else:
            sld_3.placeholders[15].text = f'{"{:.1f}".format(pct_chg_unit_share)} Pts vs YAGO' 



        sld_3.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nYAGO Period: {str(yago_print_start).replace("-","/")} - {str(yago_print_end).replace("-","/")}' 

        phr = sld_3.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')



34
34
34


In [173]:
vmr_trend_brand_sl

,cal_sun_wk_ending_dt,trend_chart,dollar_sales,units_sales
0,2026-04-19,Recent Trend Period,32610.35,3737
1,2026-04-12,Recent Trend Period,57943269.78,6174705
2,2026-04-05,Recent Trend Period,58161821.55,6249549
3,2026-03-29,Recent Trend Period,56620188.29,5947720
4,2026-03-22,Recent Trend Period,58644543.88,6365625
...,...,...,...,...
65,2024-09-29,YAGO Trend Period,46974872.72,5276585
66,2024-09-22,YAGO Trend Period,48182895.40,5337724
67,2024-09-15,YAGO Trend Period,47808971.80,5307489
68,2024-09-08,YAGO Trend Period,47711832.62,5291878


In [174]:
if dominance != 'Kroger':

    if vmr_trend_yago_summary.loc[0, 'Analysis Periods'] != 'YAGO Period':

        #Adding Slide 3: VMR Campaign UNITS

        sld_3 = prs.slides.add_slide(prs.slide_layouts[11])

        #Chart 2.1: Trend

        # X and Y values for Current Dollars:

        reward_dates = [df_promo_summary['Actual Start'].min(), df_promo_summary['Actual Start'].min(), df_promo_summary['Actual End'].max(), df_promo_summary['Actual End'].max()]
        dates_trend  = list(vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).cal_sun_wk_ending_dt)
        curr_dol_val = list(vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).units_sales.values)
        yago_dol_val = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'YAGO Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).units_sales

        len(dates_trend)
        len(curr_dol_val)
        len(yago_dol_val)

        curr_dol_val_cp = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).units_sales
        dates_trend_cp  = vmr_trend_brand_sl[vmr_trend_brand_sl['trend_chart'] == 'Recent Trend Period'].sort_values(by=['cal_sun_wk_ending_dt'], ascending = True).cal_sun_wk_ending_dt


            #Creating Df with all the dates
        df_trend = pd.DataFrame({'cal_sun_wk_ending_dt': dates_trend, 'units_sales': curr_dol_val})
        #print(df_trend)

        #Adding reward period dates
        reward_dates = [df_promo_summary['Actual Start'].min(), df_promo_summary['Actual Start'].min(), df_promo_summary['Actual End'].max(), df_promo_summary['Actual End'].max()] 
        new_data = pd.DataFrame({'cal_sun_wk_ending_dt': reward_dates, 'units_sales': ['#N/A','#N/A','#N/A','#N/A']})

        # Insert the new line at the end of the DataFrame
        df_trend = df_trend.append(new_data, ignore_index=True)

        df_trend = df_trend.sort_values(by=['cal_sun_wk_ending_dt'], ascending = True)
        df_trend = df_trend.reset_index(drop=True)
        #print(df_trend)



        #VERTICAL ONE LINE

        vertical_one = []
        counter = 0
        for i in range(len(df_trend['cal_sun_wk_ending_dt'])):
            if df_trend['cal_sun_wk_ending_dt'][i] == df_promo_summary['Actual Start'].min():
                if counter == 0:
                    vertical_one.append(0)
                    counter += 1
                else:
                    vertical_one.append(max(curr_dol_val) + max(curr_dol_val)/4)
                    counter = 0
            else:
                vertical_one.append('')

        #print(vertical_one)


        #VERTICAL SECOND LINE

        vertical_second = []
        counter = 0
        for i in range(len(df_trend['cal_sun_wk_ending_dt'])):
            if df_trend['cal_sun_wk_ending_dt'][i] == df_promo_summary['Actual End'].max():
                if counter == 0:
                    vertical_second.append(0)
                    counter += 1
                else:
                    vertical_second.append(max(curr_dol_val) + max(curr_dol_val)/4)
                    counter = 0
            else:
                vertical_second.append('')

        #print(vertical_second)



        chart_data = CategoryChartData()
        chart_data.categories = df_trend['cal_sun_wk_ending_dt']
        #chart_data.add_series('YAGO Period', df_trend['units_sales_yago'], '#,##0')
        chart_data.add_series('VMR Period', df_trend['units_sales'], '#,##0')
        chart_data.add_series('Campaign Start', vertical_one, '#,##0')
        chart_data.add_series('Campaign End', vertical_second, '#,##0')


        chart3_1 = sld_3.placeholders[10].insert_chart(XL_CHART_TYPE.LINE, chart_data).chart


        chart3_1.font.size  = Pt(12)
        chart3_1.value_axis.has_major_gridlines = False
        chart3_1.has_legend = True
        chart3_1.legend.position = XL_LEGEND_POSITION.BOTTOM
        chart3_1.legend.include_in_layout = False
        chart3_1.value_axis.major_tick_mark = XL_TICK_MARK.OUTSIDE

        widths_sizes = [0.04,0.04,0.02,0.02]

        series = chart3_1.series
        for i, serie in enumerate(chart3_1.series):
            line_format = serie.format.line
            line_format.width = Inches(widths_sizes[i])  

            if i!=0 and i!=1:
                line_format.dash_style = MSO_LINE.SQUARE_DOT


        #Chart 3.2: Bar Chart 1

        chart_data = ChartData()
        chart_data.categories = vmr_trend_yago_summary['Analysis Periods']
        chart_data.add_series('', vmr_trend_yago_summary['Units per Trip'], '#,##0.0')


        chart3_2 = sld_3.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart3_2.font.size  = Pt(12)
        chart3_2.category_axis.tick_labels.font.bold = True
        chart3_2.value_axis.has_major_gridlines = False


        chart3_2.plots[0].has_data_labels = True
        data_labels_2 = chart3_2.plots[0].data_labels
        data_labels_2.font.size = Pt(14)
        data_labels_2.font.bold = True
        data_labels_2.number_format = '#,##0.0'



        #Chart 3.3: Bar Chart 2


        pct_chg_unit_share = (share_vmr_unit - share_yago_unit)*100

        chart_data = ChartData()
        chart_data.categories = vmr_trend_yago_summary['Analysis Periods']
        chart_data.add_series('', [share_vmr_unit], '0.0%')


        chart3_3 = sld_3.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart3_3.font.size  = Pt(12)
        chart3_3.category_axis.tick_labels.font.bold = True
        chart3_3.value_axis.has_major_gridlines = False


        chart3_3.plots[0].has_data_labels = True
        data_labels_3 = chart3_3.plots[0].data_labels
        data_labels_3.font.size = Pt(14)
        data_labels_3.font.bold = True
        data_labels_3.number_format = '0.0%'


        sld_3.placeholders[13].text = f'VMR Period +{units_chg_yago}% chg vs YAGO'


        if units_per_trip_chg_yago > 0:
            sld_3.placeholders[14].text = f'+{units_per_trip_chg_yago}%                     vs YAGO'
        else:
            sld_3.placeholders[14].text = f'{units_per_trip_chg_yago}%                     vs YAGO'

        if pct_chg_unit_share > 0:
            sld_3.placeholders[15].text = f'+{"{:.1f}".format(pct_chg_unit_share)} Pts vs YAGO' 
        else:
            sld_3.placeholders[15].text = f'{"{:.1f}".format(pct_chg_unit_share)} Pts vs YAGO' 


        sld_3.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nYAGO Period: {str(yago_print_start).replace("-","/")} - {str(yago_print_end).replace("-","/")}' 

        phr = sld_3.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')



In [175]:
if dominance == 'Kroger':
    
    if vmr_trend_26wk_summary.loc[0, 'Analysis Periods'] == 'Prior 26 wks':
        
        sld_2 = prs.slides.add_slide(prs.slide_layouts[15])
        
        #Chart 2.2: Bar Chart 1

        chart_data = ChartData()
        chart_data.categories = vmr_trend_26wk_summary['Analysis Periods']
        chart_data.add_series('', vmr_trend_26wk_summary['Dollars per Trip'], '$#,##0.00')


        chart2_2 = sld_2.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart2_2.font.size  = Pt(12)
        chart2_2.category_axis.tick_labels.font.bold = True
        chart2_2.value_axis.has_major_gridlines = False


        chart2_2.plots[0].has_data_labels = True
        data_labels_2 = chart2_2.plots[0].data_labels
        data_labels_2.font.size = Pt(14)
        data_labels_2.font.bold = True
        data_labels_2.number_format = '$#,##0.00'



        #Chart 2.3: Bar Chart 2

        pct_chg_dol_share = (share_vmr_dol - share_26wk_dol)*100

        chart_data = ChartData()
        chart_data.categories = vmr_trend_26wk_summary['Analysis Periods']
        chart_data.add_series('', [share_26wk_dol, share_vmr_dol], '0.0%')


        chart2_3 = sld_2.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart2_3.font.size  = Pt(12)
        chart2_3.category_axis.tick_labels.font.bold = True
        chart2_3.value_axis.has_major_gridlines = False


        chart2_3.plots[0].has_data_labels = True
        data_labels_3 = chart2_3.plots[0].data_labels
        data_labels_3.font.size = Pt(14)
        data_labels_3.font.bold = True
        data_labels_3.number_format = '0.0%'

        
        if dollars_per_trip_chg_yago > 0:    
            sld_2.placeholders[14].text = f'+{dollars_per_trip_chg_26wk}%                       vs Prior 26wk'
        else:
            sld_2.placeholders[14].text = f'{dollars_per_trip_chg_26wk}%                       vs Prior 26wk'

        if pct_chg_dol_share > 0:
            sld_2.placeholders[15].text = f'+{"{:.1f}".format(pct_chg_dol_share)} Pts vs Prior 26wk' 
        else:
            sld_2.placeholders[15].text = f'{"{:.1f}".format(pct_chg_dol_share)} Pts vs Prior 26wk' 


        sld_2.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nPrior 26 wk Period: {str(vmr_pre26wk_start).replace("-","/")} - {str(vmr_pre26wk_end).replace("-","/")}' 

        phr = sld_2.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


In [176]:
#Chart 2.2: Bar Chart 1

if dominance == 'Kroger':
    
    if vmr_trend_26wk_summary.loc[0, 'Analysis Periods'] != 'Prior 26 wks':
        
        sld_2 = prs.slides.add_slide(prs.slide_layouts[15])


        chart_data = ChartData()
        chart_data.categories = vmr_trend_26wk_summary['Analysis Periods']
        chart_data.add_series('', vmr_trend_26wk_summary['Dollars per Trip'], '$#,##0.00')


        chart2_2 = sld_2.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart2_2.font.size  = Pt(12)
        chart2_2.category_axis.tick_labels.font.bold = True
        chart2_2.value_axis.has_major_gridlines = False


        chart2_2.plots[0].has_data_labels = True
        data_labels_2 = chart2_2.plots[0].data_labels
        data_labels_2.font.size = Pt(14)
        data_labels_2.font.bold = True
        data_labels_2.number_format = '$#,##0.00'



        #Chart 2.3: Bar Chart 2

        pct_chg_dol_share = (share_vmr_dol - share_26wk_dol)*100

        chart_data = ChartData()
        chart_data.categories = vmr_trend_26wk_summary['Analysis Periods']
        chart_data.add_series('', [share_vmr_dol], '0.0%')


        chart2_3 = sld_2.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart2_3.font.size  = Pt(12)
        chart2_3.category_axis.tick_labels.font.bold = True
        chart2_3.value_axis.has_major_gridlines = False


        chart2_3.plots[0].has_data_labels = True
        data_labels_3 = chart2_3.plots[0].data_labels
        data_labels_3.font.size = Pt(14)
        data_labels_3.font.bold = True
        data_labels_3.number_format = '0.0%'


        if dollars_per_trip_chg_yago > 0:    
            sld_2.placeholders[14].text = f'+{dollars_per_trip_chg_26wk}%                       vs Prior 26wk'
        else:
            sld_2.placeholders[14].text = f'{dollars_per_trip_chg_26wk}%                       vs Prior 26wk'

        if pct_chg_dol_share > 0:
            sld_2.placeholders[15].text = f'+{"{:.1f}".format(pct_chg_dol_share)} Pts vs Prior 26wk' 
        else:
            sld_2.placeholders[15].text = f'{"{:.1f}".format(pct_chg_dol_share)} Pts vs Prior 26wk' 



        sld_2.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nPrior 26 wk Period: {str(vmr_pre26wk_start).replace("-","/")} - {str(vmr_pre26wk_end).replace("-","/")}' 

        phr = sld_2.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


In [177]:
#Chart 3.2: Bar Chart 1

if dominance == 'Kroger':
    
    if vmr_trend_26wk_summary.loc[0, 'Analysis Periods'] == 'Prior 26 wks':
        
        sld_3 = prs.slides.add_slide(prs.slide_layouts[16])

        #Chart 3.2: Bar Chart 1

        chart_data = ChartData()
        chart_data.categories = vmr_trend_26wk_summary['Analysis Periods']
        chart_data.add_series('', vmr_trend_26wk_summary['Units per Trip'], '#,##0.0')


        chart3_2 = sld_3.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart3_2.font.size  = Pt(12)
        chart3_2.category_axis.tick_labels.font.bold = True
        chart3_2.value_axis.has_major_gridlines = False


        chart3_2.plots[0].has_data_labels = True
        data_labels_2 = chart3_2.plots[0].data_labels
        data_labels_2.font.size = Pt(14)
        data_labels_2.font.bold = True
        data_labels_2.number_format = '#,##0.0'



        #Chart 3.3: Bar Chart 2


        pct_chg_unit_share = (share_vmr_unit - share_26wk_unit)*100

        chart_data = ChartData()
        chart_data.categories = vmr_trend_26wk_summary['Analysis Periods']
        chart_data.add_series('', [share_26wk_unit, share_vmr_unit], '0.0%')


        chart3_3 = sld_3.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart3_3.font.size  = Pt(12)
        chart3_3.category_axis.tick_labels.font.bold = True
        chart3_3.value_axis.has_major_gridlines = False


        chart3_3.plots[0].has_data_labels = True
        data_labels_3 = chart3_3.plots[0].data_labels
        data_labels_3.font.size = Pt(14)
        data_labels_3.font.bold = True
        data_labels_3.number_format = '0.0%'


        if units_per_trip_chg_yago > 0:
            sld_3.placeholders[14].text = f'+{units_per_trip_chg_26wk}%                     vs Prior 26wk'
        else:
            sld_3.placeholders[14].text = f'{units_per_trip_chg_26wk}%                     vs Prior 26wk'

        if pct_chg_unit_share > 0:
            sld_3.placeholders[15].text = f'+{"{:.1f}".format(pct_chg_unit_share)} Pts vs Prior 26wk' 
        else:
            sld_3.placeholders[15].text = f'{"{:.1f}".format(pct_chg_unit_share)} Pts vs Prior 26wk' 



        sld_3.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nPrior 26 wk Period: {str(vmr_pre26wk_start).replace("-","/")} - {str(vmr_pre26wk_end).replace("-","/")}' 

        phr = sld_3.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')



In [178]:
#Chart 3.2: Bar Chart 1

if dominance == 'Kroger':
    
    if vmr_trend_26wk_summary.loc[0, 'Analysis Periods'] != 'Prior 26 wks':
        
        sld_3 = prs.slides.add_slide(prs.slide_layouts[16])
        
        #Chart 3.2: Bar Chart 1

        chart_data = ChartData()
        chart_data.categories = vmr_trend_26wk_summary['Analysis Periods']
        chart_data.add_series('', vmr_trend_26wk_summary['Units per Trip'], '#,##0.0')


        chart3_2 = sld_3.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart3_2.font.size  = Pt(12)
        chart3_2.category_axis.tick_labels.font.bold = True
        chart3_2.value_axis.has_major_gridlines = False


        chart3_2.plots[0].has_data_labels = True
        data_labels_2 = chart3_2.plots[0].data_labels
        data_labels_2.font.size = Pt(14)
        data_labels_2.font.bold = True
        data_labels_2.number_format = '#,##0.0'



        #Chart 3.3: Bar Chart 2


        pct_chg_unit_share = (share_vmr_unit - share_26wk_unit)*100

        chart_data = ChartData()
        chart_data.categories = vmr_trend_26wk_summary['Analysis Periods']
        chart_data.add_series('', [share_vmr_unit], '0.0%')


        chart3_3 = sld_3.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart3_3.font.size  = Pt(12)
        chart3_3.category_axis.tick_labels.font.bold = True
        chart3_3.value_axis.has_major_gridlines = False


        chart3_3.plots[0].has_data_labels = True
        data_labels_3 = chart3_3.plots[0].data_labels
        data_labels_3.font.size = Pt(14)
        data_labels_3.font.bold = True
        data_labels_3.number_format = '0.0%'


        if units_per_trip_chg_yago > 0:
            sld_3.placeholders[14].text = f'+{units_per_trip_chg_26wk}%                     vs Prior 26wk'
        else:
            sld_3.placeholders[14].text = f'{units_per_trip_chg_26wk}%                     vs Prior 26wk'

        if pct_chg_unit_share > 0:
            sld_3.placeholders[15].text = f'+{"{:.1f}".format(pct_chg_unit_share)} Pts vs Prior 26wk' 
        else:
            sld_3.placeholders[15].text = f'{"{:.1f}".format(pct_chg_unit_share)} Pts vs Prior 26wk' 


        sld_3.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nPrior 26 wk Period: {str(vmr_pre26wk_start).replace("-","/")} - {str(vmr_pre26wk_end).replace("-","/")}' 

        phr = sld_3.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')




In [179]:
#Adding Slide 4: VMR Dollars Moved

sld_4 = prs.slides.add_slide(prs.slide_layouts[3])

sld_4.placeholders[10].text = f'VMR reward trips accounted for {round(pct_reward_trips*100)}% of total brand transactions during the VMR Campaign \n${"{:,}".format(round(dollars_moved_vmr))} total dollars moved among all campaign reward levels'
sld_4.placeholders[13].text = f'${"{:,}".format(round(dollars_moved_vmr))} TOTAL DOLLARS MOVED AT REWARD TRIP LEVELS'


phr = sld_4.placeholders[13]
# Access the text frame and text properties
text_frame = phr.text_frame
for paragraph in text_frame.paragraphs:
    paragraph.space_before = Pt(0)  # If you want to remove space before paragraph

#Chart 4.1: Pie Chart

chart_data = ChartData()
chart_data.categories = ['All Other Trips', 'Reward Level Trips']
chart_data.add_series('', (1-pct_reward_trips, pct_reward_trips), '0%')


chart4_1 = sld_4.placeholders[11].insert_chart(XL_CHART_TYPE.PIE, chart_data).chart
chart4_1.font.size  = Pt(12)
chart4_1.has_legend = True
chart4_1.legend.position = XL_LEGEND_POSITION.BOTTOM
chart4_1.legend.include_in_layout = False
chart4_1.has_title = False

leg = chart4_1.legend
if leg:
    leg.font.bold = True
    

chart4_1.plots[0].has_data_labels = True
data_labels_1 = chart4_1.plots[0].data_labels
data_labels_1.font.size = Pt(16)
data_labels_1.font.bold = True
data_labels_1.font.color.rgb = RGBColor(255, 255, 255)
data_labels_1.number_format = '0%'
data_labels_1.position = XL_LABEL_POSITION.CENTER


#Chart 4.2: Bar Chart

chart_data = ChartData()
chart_data.categories = level_ct.loc[(level_ct['level']!= 'All Other Transactions') & (level_ct['level']!= 'Grand Total')].level
chart_data.add_series('', level_ct.loc[(level_ct['level']!= 'All Other Transactions') & (level_ct['level']!= 'Grand Total')].dollars, '$#,##0')


chart4_2 = sld_4.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
chart4_2.font.size  = Pt(11)
chart4_2.category_axis.tick_labels.font.bold = True
chart4_2.value_axis.has_major_gridlines = False


chart4_2.plots[0].has_data_labels = True
data_labels_1 = chart4_2.plots[0].data_labels
data_labels_1.font.size = Pt(14)
data_labels_1.font.bold = True
data_labels_1.number_format = '$#,##0'



bar_colors = [RGBColor(152, 162, 167), RGBColor(230, 103, 73), RGBColor(240, 192, 86), RGBColor(152, 162, 167), RGBColor(230, 103, 73), RGBColor(240, 192, 86), RGBColor(152, 162, 167), RGBColor(230, 103, 73), RGBColor(240, 192, 86) ]

# Iterate through the bars and set colors
for i, series in enumerate(chart4_2.series):
    for j, point in enumerate(series.points):
        point.format.fill.solid()
        point.format.fill.fore_color.rgb = bar_colors[j]
        
sld_4.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")}' 

phr = sld_4.placeholders[16]
# Modify the font name and size
font_name = 'Arial Black'
font_size = Pt(10)
# Access the text frame and text properties
text_frame = phr.text_frame
for paragraph in text_frame.paragraphs:
    paragraph.font.name = font_name
    paragraph.font.size = font_size
    for run in paragraph.runs:
        run.text = run.text.replace('•', '')



In [180]:
level_ct

,level,dollars,trips,units
0,Buy $0+,1.228082e+08,2967989,11040784
1,Grand Total,1.228082e+08,2967989,11040784


In [181]:
level_ct

,level,dollars,trips,units
0,Buy $0+,1.228082e+08,2967989,11040784
1,Grand Total,1.228082e+08,2967989,11040784


In [182]:
chart_data

In [183]:
vmr_trend_yago_summary

,Analysis Periods,Count Distinct Trips,Dollar Sales,Dollars per Trip,Units Sold,Units per Trip,% Change Dollars per Trip,% Change Units per Trip
0,YAGO Period,13574255,2.390810e+08,17.612829,26092326,1.9,-,-
1,VMR Period,13533312,2.507479e+08,18.528196,26899195,2.0,0.051972,0.052632


In [184]:
#Adding Slide 5: CAMPAIGN TRENDS

if vmr_trend_yago_summary.loc[0, 'Analysis Periods'] == 'YAGO Period':
    
    if dominance != 'Kroger':


        sld_5 = prs.slides.add_slide(prs.slide_layouts[12])


        #Table 5.1:  #11

        results_participants_03 = results_participants_03[['Time Period', 'Dollars per Buyer', 'Units per Buyer', 'Trips per Buyer', 'Dollars per Trip', 'Units per Trip']]
        
        tph1_1 = sld_5.placeholders[11].insert_table(rows = results_participants_03.shape[0], cols = results_participants_03.shape[1])
        tph1_1._element.graphic.graphicData.tbl[0][-1].text = '{69012ECD-51FC-41F1-AA8D-1B2483CD663E}' 
        table5_1 = tph1_1.table



        #populate slide 1 table 1
        for i in range(results_participants_03.shape[0]):
            for j in range(results_participants_03.shape[1]):
                cell = table5_1.cell(i, j)

                if (i!= 0 and i!=3) and (j == 1 or j== 4) :
                    cell.text = "${:,.2f}".format(results_participants_03.iloc[i,j])
                elif (i!= 0 and i!=3) and (j == 2 or j==3 or j==5):
                    cell.text = "{:,.1f}".format(results_participants_03.iloc[i,j])
                elif i==3 and (j==1 or j==2 or j==3 or j==4 or j==5):

                    if results_participants_03.iloc[i,j] != '':
                        cell.text = "{:.0%}".format(results_participants_03.iloc[i,j])
                    else:
                        cell.text = results_participants_03.iloc[i,j]
                else:
                    cell.text = str(results_participants_03.iloc[i,j])

                cell.text_frame.paragraphs[0].font.size = Pt(14.5)
                cell.text_frame.paragraphs[0].alignment = PP_ALIGN.CENTER



        #Chart 5.1:   #12

        chart_data = ChartData()
        chart_data.categories = ['Dollars per Buyer', 'Units per Buyer', 'Trips per Buyer', 'Dollars per Trip', 'Units per Trip']

        chart_data.add_series('', [change_dpb, change_upb, change_tpb, change_dpt, change_upt], '0%')


        chart5_1 = sld_5.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart5_1.font.size  = Pt(12)
        chart5_1.category_axis.tick_labels.font.bold = True
        chart5_1.has_legend = False

        chart5_1.has_title = True
        chart5_1.chart_title.text_frame.text = f"% CHANGE - Campaign Period vs YAGO Period"
        chart_title = chart5_1.chart_title.text_frame
        font = chart_title.paragraphs[0].runs[0].font
        font.size = Pt(14) 
        font.bold = True

        chart5_1.value_axis.has_major_gridlines = False

        chart5_1.plots[0].has_data_labels = True
        data_labels_1 = chart5_1.plots[0].data_labels
        data_labels_1.font.size = Pt(14)
        data_labels_1.font.bold = True
        data_labels_1.font.color.rgb = RGBColor(0, 32, 96)
        data_labels_1.number_format = '0%'
        data_labels_1.position = XL_LABEL_POSITION.OUTSIDE_END



        #Applying conditional formatting to color the bars depending on the label dollars, units or trips

        colors = [RGBColor(0, 56, 104),RGBColor(127, 169, 184),RGBColor(240, 192, 86),RGBColor(0, 56, 104),RGBColor(127, 169, 184)]

        for index, series in enumerate(chart5_1.series):
            for point, color in zip(series.points, colors):
                point.format.fill.solid()
                point.format.fill.fore_color.rgb = color



        #Other option to turn off the revert negative values option:

        from pptx.oxml.xmlchemy import OxmlElement

        # Iterate over each series and point in the chart
        for serie in chart5_1.series:
            for point in serie.points:
                # Create the new XML element
                element = OxmlElement('c:invertIfNegative')
                # Set the attribute 'val' to '0'
                element.set('val', '0')
                # Append the new element to the point's format element
                point.format.element.append(element)



        #Text placeholder 10

        if change_dpb > 0:
            sld_5.placeholders[10].text = f'Campaign Participants avg dollars per buyer CHANGE +{"{:.0%}".format(change_dpb)} during the campaign period vs YAGO period'
        else:
            sld_5.placeholders[10].text = f'Campaign Participants avg dollars per buyer CHANGE {"{:.0%}".format(change_dpb)} during the campaign period vs YAGO period'


        sld_5.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nYAGO Period: {str(yago_print_start).replace("-","/")} - {str(yago_print_end).replace("-","/")}' 

        phr = sld_5.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


In [185]:
#Adding Slide 5: CAMPAIGN TRENDS

if vmr_trend_26wk_summary.loc[0, 'Analysis Periods'] == 'Prior 26 wks':
    
    if dominance == 'Kroger':

        sld_5 = prs.slides.add_slide(prs.slide_layouts[12])


        #Table 5.1:  #11

        results_participants_04 = results_participants_04[['Time Period', 'Dollars per Buyer', 'Units per Buyer', 'Trips per Buyer', 'Dollars per Trip', 'Units per Trip']]
        
        tph1_1 = sld_5.placeholders[11].insert_table(rows = results_participants_04.shape[0], cols = results_participants_04.shape[1])
        tph1_1._element.graphic.graphicData.tbl[0][-1].text = '{69012ECD-51FC-41F1-AA8D-1B2483CD663E}' 
        table5_1 = tph1_1.table



        #populate slide 1 table 1
        for i in range(results_participants_04.shape[0]):
            for j in range(results_participants_04.shape[1]):
                cell = table5_1.cell(i, j)

                if (i!= 0 and i!=3) and (j == 1 or j== 4) :
                    cell.text = "${:,.2f}".format(results_participants_04.iloc[i,j])
                elif (i!= 0 and i!=3) and (j == 2 or j==3 or j==5):
                    cell.text = "{:,.1f}".format(results_participants_04.iloc[i,j])
                elif i==3 and (j==1 or j==2 or j==3 or j==4 or j==5):

                    if results_participants.iloc[i,j] != '':
                        cell.text = "{:.0%}".format(results_participants_04.iloc[i,j])
                    else:
                        cell.text = results_participants_04.iloc[i,j]
                else:
                    cell.text = str(results_participants_04.iloc[i,j])

                cell.text_frame.paragraphs[0].font.size = Pt(14.5)
                cell.text_frame.paragraphs[0].alignment = PP_ALIGN.CENTER



        #Chart 5.1:   #12

        chart_data = ChartData()
        chart_data.categories = ['Dollars per Buyer', 'Units per Buyer', 'Trips per Buyer', 'Dollars per Trip', 'Units per Trip']

        chart_data.add_series('', [change_dpb_26, change_upb_26, change_tpb_26, change_dpt_26, change_upt_26], '0%')


        chart5_1 = sld_5.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart5_1.font.size  = Pt(12)
        chart5_1.category_axis.tick_labels.font.bold = True
        chart5_1.has_legend = False

        chart5_1.has_title = True
        chart5_1.chart_title.text_frame.text = f"% CHANGE - Campaign Period vs Prior 26wk Period"
        chart_title = chart5_1.chart_title.text_frame
        font = chart_title.paragraphs[0].runs[0].font
        font.size = Pt(14) 
        font.bold = True

        chart5_1.value_axis.has_major_gridlines = False

        chart5_1.plots[0].has_data_labels = True
        data_labels_1 = chart5_1.plots[0].data_labels
        data_labels_1.font.size = Pt(14)
        data_labels_1.font.bold = True
        data_labels_1.font.color.rgb = RGBColor(0, 32, 96)
        data_labels_1.number_format = '0%'
        data_labels_1.position = XL_LABEL_POSITION.OUTSIDE_END



        #Applying conditional formatting to color the bars depending on the label dollars, units or trips

        colors = [RGBColor(0, 56, 104),RGBColor(127, 169, 184),RGBColor(240, 192, 86),RGBColor(0, 56, 104),RGBColor(127, 169, 184)]

        for index, series in enumerate(chart5_1.series):
            for point, color in zip(series.points, colors):
                point.format.fill.solid()
                point.format.fill.fore_color.rgb = color



        #Other option to turn off the revert negative values option:

        from pptx.oxml.xmlchemy import OxmlElement

        # Iterate over each series and point in the chart
        for serie in chart5_1.series:
            for point in serie.points:
                # Create the new XML element
                element = OxmlElement('c:invertIfNegative')
                # Set the attribute 'val' to '0'
                element.set('val', '0')
                # Append the new element to the point's format element
                point.format.element.append(element)



        #Text placeholder 10

        if change_dpb_26 > 0:
            sld_5.placeholders[10].text = f'Campaign Participants avg dollars per buyer CHANGE +{"{:.0%}".format(change_dpb_26)} during the campaign period vs Prior 26wk period'
        else:
            sld_5.placeholders[10].text = f'Campaign Participants avg dollars per buyer CHANGE {"{:.0%}".format(change_dpb_26)} during the campaign period vs Prior 26wk period'


        sld_5.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nPrior 26 wk Period: {str(vmr_pre26wk_start).replace("-","/")} - {str(vmr_pre26wk_end).replace("-","/")}' 

        phr = sld_5.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


In [186]:
#Adding Slide 6: CATEGORY SHARE

if dominance != 'Kroger':

    sld_6 = prs.slides.add_slide(prs.slide_layouts[13])

    #Chart 6.1: Bar Chart

    chart_data = ChartData()
    chart_data.categories = ['YAGO Period', 'VMR Period', 'Post Period']
    chart_data.add_series('', (share_dollar_yago, share_dollar_vmr, share_dollar_post_4wk), '0%')


    chart6_1 = sld_6.placeholders[17].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
    chart6_1.font.size  = Pt(12)
    chart6_1.category_axis.tick_labels.font.bold = True
    chart6_1.value_axis.has_major_gridlines = False
    chart6_1.value_axis.visible = False


    chart6_1.plots[0].has_data_labels = True
    data_labels_1 = chart6_1.plots[0].data_labels
    data_labels_1.font.size = Pt(14)
    data_labels_1.font.bold = True
    data_labels_1.number_format = '0%'


    #Chart 6.2: Bar Chart

    chart_data = ChartData()
    chart_data.categories = ['YAGO Period', 'VMR Period', 'Post Period']
    chart_data.add_series('', (share_units_yago, share_units_vmr, share_units_post_4wk), '0%')


    chart6_2 = sld_6.placeholders[18].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
    chart6_2.font.size  = Pt(12)
    chart6_2.category_axis.tick_labels.font.bold = True
    chart6_2.value_axis.has_major_gridlines = False
    chart6_2.value_axis.visible = False


    chart6_2.plots[0].has_data_labels = True
    data_labels_1 = chart6_2.plots[0].data_labels
    data_labels_1.font.size = Pt(14)
    data_labels_1.font.bold = True
    data_labels_1.number_format = '0%'


    #Text

    chg_cat_share_dol = ( share_dollar_vmr - share_dollar_yago ) *100
    chg_cat_share_units = ( share_units_vmr - share_units_yago ) *100

    if chg_cat_share_dol >0:
        sld_6.placeholders[16].text = f'Campaign Participants dollar share changed +{"{:.0f}".format(chg_cat_share_dol)} pts during the campaign period vs YAGO'
    else:
        sld_6.placeholders[16].text = f'Campaign Participants dollar share changed {"{:.0f}".format(chg_cat_share_dol)} pts during the campaign period vs YAGO'


    if chg_cat_share_dol > 0:
        sld_6.placeholders[14].text = f'VMR Period +{"{:.0f}".format(chg_cat_share_dol)} pts vs YAGO'
    else:
        sld_6.placeholders[14].text = f'VMR Period {"{:.0f}".format(chg_cat_share_dol)} pts vs YAGO'


    if chg_cat_share_units > 0:
        sld_6.placeholders[15].text = f'VMR Period +{"{:.0f}".format(chg_cat_share_units)} pts vs YAGO'
    else:
        sld_6.placeholders[15].text = f'VMR Period {"{:.0f}".format(chg_cat_share_units)} pts vs YAGO'



    sld_6.placeholders[19].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nYAGO Period: {str(yago_print_start).replace("-","/")} - {str(yago_print_end).replace("-","/")} \nPost Period: {str(post_4wk_start).replace("-","/")} - {str(post_4wk_end).replace("-","/")}' 


    phr = sld_6.placeholders[19]
    # Modify the font name and size
    font_name = 'Arial Black'
    font_size = Pt(8)
    # Access the text frame and text properties
    text_frame = phr.text_frame
    for paragraph in text_frame.paragraphs:
        paragraph.font.name = font_name
        paragraph.font.size = font_size
        for run in paragraph.runs:
            run.text = run.text.replace('•', '')


In [187]:
#Adding Slide 7: PRE PERIOD PROFILE AND REPEAT

if dominance != 'Kroger':

    sld_7 = prs.slides.add_slide(prs.slide_layouts[6])

    #Chart 7.1: Pie Chart

    chart_data = ChartData()
    chart_data.categories = ['New Buyers', 'Existing Buyers']


    pct_new_buy = round(brand_consump.loc[0, '% Brand IDs'],2)

    chart_data.add_series('', (pct_new_buy, 1- pct_new_buy), '0%')


    chart7_1 = sld_7.placeholders[11].insert_chart(XL_CHART_TYPE.PIE, chart_data).chart
    chart7_1.font.size  = Pt(12)
    chart7_1.has_legend = True
    chart7_1.legend.position = XL_LEGEND_POSITION.BOTTOM
    chart7_1.legend.include_in_layout = False
    chart7_1.has_title = False

    leg = chart7_1.legend
    if leg:
        leg.font.bold = True

    chart7_1.plots[0].has_data_labels = True
    data_labels_1 = chart7_1.plots[0].data_labels
    data_labels_1.font.size = Pt(16)
    data_labels_1.font.bold = True
    data_labels_1.font.color.rgb = RGBColor(255, 255, 255)
    data_labels_1.number_format = '0%'
    data_labels_1.position = XL_LABEL_POSITION.CENTER



    #Chart 7.2: Bar Chart

    chart_data = ChartData()
    chart_data.categories = ['Total Participants', 'New Buyers Participants', 'Existing Buyers Participants']

    chart_data.add_series('', (pct_repurch, pct_repurch_new_buyers, pct_repurch_existing_buyers), '0%')


    chart7_2 = sld_7.placeholders[12].insert_chart(XL_CHART_TYPE.BAR_CLUSTERED, chart_data).chart
    chart7_2.font.size  = Pt(12)
    chart7_2.category_axis.tick_labels.font.bold = True
    chart7_2.value_axis.has_major_gridlines = False


    chart7_2.plots[0].has_data_labels = True
    data_labels_1 = chart7_2.plots[0].data_labels
    data_labels_1.font.size = Pt(14)
    data_labels_1.font.bold = True
    data_labels_1.number_format = '0%'


    bar_colors = [RGBColor(240, 192, 86), RGBColor(0, 56, 104), RGBColor(127, 169, 184)]

    # Iterate through the bars and set colors
    for i, series in enumerate(chart7_2.series):
        for j, point in enumerate(series.points):
            point.format.fill.solid()
            point.format.fill.fore_color.rgb = bar_colors[j]




    #Text placeholder

    sld_7.placeholders[10].text = f'{"{:.0%}".format(pct_new_buy)} of Campaign Participants were NEW to the brand with average repeat of {"{:.0%}".format(pct_repurch_new_buyers)} \n{"{:.0%}".format(pct_repurch_existing_buyers)} of existing Campaign Participants repeated on promoted products post VMR reward trip'
    
    sld_7.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nPost Period: {str(post_4wk_start).replace("-","/")} - {str(post_4wk_end).replace("-","/")} \nRepeat Period = Campaign Period + Post Period' 

    phr = sld_7.placeholders[16]
    # Modify the font name and size
    font_name = 'Arial Black'
    font_size = Pt(8)
    # Access the text frame and text properties
    text_frame = phr.text_frame
    for paragraph in text_frame.paragraphs:
        paragraph.font.name = font_name
        paragraph.font.size = font_size
        for run in paragraph.runs:
            run.text = run.text.replace('•', '')


In [188]:

if dominance == 'Kroger':

    sld_7 = prs.slides.add_slide(prs.slide_layouts[19])

    #Chart 7.1: Pie Chart

    chart_data = ChartData()
    chart_data.categories = ['New Buyers', 'Existing Buyers']


    pct_new_buy = round(brand_consump.loc[0, '% Brand IDs'],2)

    chart_data.add_series('', (pct_new_buy, 1- pct_new_buy), '0%')


    chart7_1 = sld_7.placeholders[11].insert_chart(XL_CHART_TYPE.PIE, chart_data).chart
    chart7_1.font.size  = Pt(12)
    chart7_1.has_legend = True
    chart7_1.legend.position = XL_LEGEND_POSITION.BOTTOM
    chart7_1.legend.include_in_layout = False
    chart7_1.has_title = False

    leg = chart7_1.legend
    if leg:
        leg.font.bold = True

    chart7_1.plots[0].has_data_labels = True
    data_labels_1 = chart7_1.plots[0].data_labels
    data_labels_1.font.size = Pt(16)
    data_labels_1.font.bold = True
    data_labels_1.font.color.rgb = RGBColor(255, 255, 255)
    data_labels_1.number_format = '0%'
    data_labels_1.position = XL_LABEL_POSITION.CENTER



    #Chart 7.2: Bar Chart

    chart_data = ChartData()
    chart_data.categories = ['Total Participants', 'New Buyers Participants', 'Existing Buyers Participants']

    chart_data.add_series('', (pct_repurch, pct_repurch_new_buyers, pct_repurch_existing_buyers), '0%')


    chart7_2 = sld_7.placeholders[12].insert_chart(XL_CHART_TYPE.BAR_CLUSTERED, chart_data).chart
    chart7_2.font.size  = Pt(12)
    chart7_2.category_axis.tick_labels.font.bold = True
    chart7_2.value_axis.has_major_gridlines = False


    chart7_2.plots[0].has_data_labels = True
    data_labels_1 = chart7_2.plots[0].data_labels
    data_labels_1.font.size = Pt(14)
    data_labels_1.font.bold = True
    data_labels_1.number_format = '0%'


    bar_colors = [RGBColor(240, 192, 86), RGBColor(0, 56, 104), RGBColor(127, 169, 184)]

    # Iterate through the bars and set colors
    for i, series in enumerate(chart7_2.series):
        for j, point in enumerate(series.points):
            point.format.fill.solid()
            point.format.fill.fore_color.rgb = bar_colors[j]




    #Text placeholder

    sld_7.placeholders[10].text = f'{"{:.0%}".format(pct_new_buy)} of Campaign Participants were NEW to the brand with average repeat of {"{:.0%}".format(pct_repurch_new_buyers)} \n{"{:.0%}".format(pct_repurch_existing_buyers)} of existing Campaign Participants repeated on promoted products post VMR reward trip'
    sld_7.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nPost Period: {str(post_4wk_start).replace("-","/")} - {str(post_4wk_end).replace("-","/")} \nRepeat Period = Campaign Period + Post Period' 

    phr = sld_7.placeholders[16]
    # Modify the font name and size
    font_name = 'Arial Black'
    font_size = Pt(8)
    # Access the text frame and text properties
    text_frame = phr.text_frame
    for paragraph in text_frame.paragraphs:
        paragraph.font.name = font_name
        paragraph.font.size = font_size
        for run in paragraph.runs:
            run.text = run.text.replace('•', '')


In [189]:

#Adding Slide 8: VMR Campaigns Trends by Segment

if nbr_segments > 1:

    if dominance != 'Kroger':

        sld_8 = prs.slides.add_slide(prs.slide_layouts[7])

        #Chart 8.1: CLustered bar

        chart_data = ChartData()
        chart_data.categories = table_slide.loc[table_slide['Segment']!='Total'].Segment.iloc[1:len(table_slide)]

        chart_data.add_series('Units per Trip', table_slide.loc[table_slide['Segment']!='Total']['% Change Units'].iloc[1:len(table_slide)], '0%')
        chart_data.add_series('Dollars per Trip', table_slide.loc[table_slide['Segment']!='Total']['% Change Dollars'].iloc[1:len(table_slide)], '0%')


        chart8_1 = sld_8.placeholders[11].insert_chart(XL_CHART_TYPE.BAR_CLUSTERED, chart_data).chart
        chart8_1.font.size  = Pt(12)
        chart8_1.category_axis.tick_labels.font.bold = True
        chart8_1.has_legend = True
        chart8_1.legend.position = XL_LEGEND_POSITION.BOTTOM
        chart8_1.legend.include_in_layout = False
        chart8_1.has_title = False
        chart8_1.value_axis.has_major_gridlines = False
        chart8_1.category_axis.tick_labels.font.bold = True

        leg = chart8_1.legend
        if leg:
            leg.font.bold = True


        chart8_1.plots[0].has_data_labels = True
        data_labels_1 = chart8_1.plots[0].data_labels
        data_labels_1.font.size = Pt(14)
        data_labels_1.font.bold = True
        data_labels_1.font.color.rgb = RGBColor(0, 32, 96)
        data_labels_1.number_format = '0%'
        data_labels_1.position = XL_LABEL_POSITION.OUTSIDE_END


        #Replacing color for each independent serie:

        bar_colors = [RGBColor(0, 56, 104), RGBColor(240, 192, 86)]

        for i,serie in enumerate(chart8_1.series):
            chart8_1.series[i].format.fill.solid()
            chart8_1.series[i].format.fill.fore_color.rgb = bar_colors[i]



        #Other option to turn off the revert negative values option:


        from pptx.oxml.xmlchemy import OxmlElement

        def SubElement(parent, tagname, **kwargs):
            element = OxmlElement(tagname)
            element.attrib.update(kwargs)
            parent.append(element)
            return element


        for i,serie in enumerate(chart8_1.series):
            point = chart8_1.series[i].points[0]
            SubElement(point.format.element, 'c:invertIfNegative', val=str(0))




        #Table 8.2:


        tph1_1 = sld_8.placeholders[12].insert_table(rows = table_slide_s.shape[0]-1, cols = table_slide_s.shape[1])
        tph1_1._element.graphic.graphicData.tbl[0][-1].text = '{69012ECD-51FC-41F1-AA8D-1B2483CD663E}' 
        table1_1 = tph1_1.table



        #populate slide 1 table 1
        for i in range(table_slide_s.shape[0]-1):
            for j in range(table_slide_s.shape[1]):
                cell = table1_1.cell(i, j)

                if i!= 0 and j == 1:
                    cell.text = "${:,}".format(table_slide_s.iloc[i,j])
                elif i!= 0 and (j == 2 or j==4):
                    cell.text = "{:.0%}".format(table_slide_s.iloc[i,j])
                else:
                    cell.text = str(table_slide_s.iloc[i,j])

                cell.text_frame.paragraphs[0].font.size = Pt(14)
                cell.text_frame.paragraphs[0].alignment = PP_ALIGN.CENTER


        #Text placeholder

        highest = x_high[x_high['Segment']!='Total'].sort_values(by='Campaign Dollars per Trip', ascending=False)
        highest = highest['Segment'].values[0]

        sld_8.placeholders[10].text = f'{highest} segment had the highest dollars per trip during campaign period vs YAGO Period'
        sld_8.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nYAGO Period: {str(yago_print_start).replace("-","/")} - {str(yago_print_end).replace("-","/")}' 


        phr = sld_8.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


In [190]:
if nbr_segments > 1:

    if dominance == 'Kroger':

        sld_8 = prs.slides.add_slide(prs.slide_layouts[20])

        #Chart 8.1: CLustered bar

        chart_data = ChartData()
        chart_data.categories = table_slide.loc[table_slide['Segment']!='Total'].Segment.iloc[1:len(table_slide)]

        chart_data.add_series('Units per Trip', table_slide.loc[table_slide['Segment']!='Total']['% Change Units'].iloc[1:len(table_slide)], '0%')
        chart_data.add_series('Dollars per Trip', table_slide.loc[table_slide['Segment']!='Total']['% Change Dollars'].iloc[1:len(table_slide)], '0%')


        chart8_1 = sld_8.placeholders[11].insert_chart(XL_CHART_TYPE.BAR_CLUSTERED, chart_data).chart
        chart8_1.font.size  = Pt(12)
        chart8_1.category_axis.tick_labels.font.bold = True
        chart8_1.has_legend = True
        chart8_1.legend.position = XL_LEGEND_POSITION.BOTTOM
        chart8_1.legend.include_in_layout = False
        chart8_1.has_title = False
        chart8_1.value_axis.has_major_gridlines = False
        chart8_1.category_axis.tick_labels.font.bold = True

        leg = chart8_1.legend
        if leg:
            leg.font.bold = True


        chart8_1.plots[0].has_data_labels = True
        data_labels_1 = chart8_1.plots[0].data_labels
        data_labels_1.font.size = Pt(14)
        data_labels_1.font.bold = True
        data_labels_1.font.color.rgb = RGBColor(0, 32, 96)
        data_labels_1.number_format = '0%'
        data_labels_1.position = XL_LABEL_POSITION.OUTSIDE_END


        #Replacing color for each independent serie:

        bar_colors = [RGBColor(0, 56, 104), RGBColor(240, 192, 86)]

        for i,serie in enumerate(chart8_1.series):
            chart8_1.series[i].format.fill.solid()
            chart8_1.series[i].format.fill.fore_color.rgb = bar_colors[i]



        #Other option to turn off the revert negative values option:


        from pptx.oxml.xmlchemy import OxmlElement

        def SubElement(parent, tagname, **kwargs):
            element = OxmlElement(tagname)
            element.attrib.update(kwargs)
            parent.append(element)
            return element


        for i,serie in enumerate(chart8_1.series):
            point = chart8_1.series[i].points[0]
            SubElement(point.format.element, 'c:invertIfNegative', val=str(0))




        #Table 8.2:


        tph1_1 = sld_8.placeholders[12].insert_table(rows = table_slide_s.shape[0]-1, cols = table_slide_s.shape[1])
        tph1_1._element.graphic.graphicData.tbl[0][-1].text = '{69012ECD-51FC-41F1-AA8D-1B2483CD663E}' 
        table1_1 = tph1_1.table



        #populate slide 1 table 1
        for i in range(table_slide_s.shape[0]-1):
            for j in range(table_slide_s.shape[1]):
                cell = table1_1.cell(i, j)

                if i!= 0 and j == 1:
                    cell.text = "${:,}".format(table_slide_s.iloc[i,j])
                elif i!= 0 and (j == 2 or j==4):
                    cell.text = "{:.0%}".format(table_slide_s.iloc[i,j])
                else:
                    cell.text = str(table_slide_s.iloc[i,j])

                cell.text_frame.paragraphs[0].font.size = Pt(14)
                cell.text_frame.paragraphs[0].alignment = PP_ALIGN.CENTER


        #Text placeholder

        highest = x_high[x_high['Segment']!='Total'].sort_values(by='Campaign Dollars per Trip', ascending=False)
        highest = highest['Segment'].values[0]

        sld_8.placeholders[10].text = f'{highest} segment had the highest dollars per trip during campaign period vs Prior 26 weeks'
        sld_8.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")} \nPrior 26 wk Period: {str(vmr_pre26wk_start).replace("-","/")} - {str(vmr_pre26wk_end).replace("-","/")}' 


        phr = sld_8.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(8)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


In [191]:

#Adding Slide 9: Combinations

if nbr_segments > 1:
    
    sld_9 = prs.slides.add_slide(prs.slide_layouts[8])


    #Chart 9.1: 
    
    combos_sum = []
    combos_single = []
    conditional_colors = []

    
    for i in range(len(combos_slide)):

        if '+' in combos_slide['Segment'][i]:
            combos_sum.append(combos_slide['Pct Of Trips'][i])
        else:
            combos_single.append(combos_slide['Pct Of Trips'][i])

    
    chart_data = ChartData()
    chart_data.categories = combos_slide['Segment']
    chart_data.add_series('', combos_slide['Pct Of Trips'], '0%')

    chart9_2 = sld_9.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
    chart9_2.font.size  = Pt(12)
    chart9_2.category_axis.tick_labels.font.bold = True
    chart9_2.value_axis.has_major_gridlines = False


    chart9_2.plots[0].has_data_labels = True
    data_labels_1 = chart9_2.plots[0].data_labels
    data_labels_1.font.size = Pt(14)
    data_labels_1.font.bold = True
    data_labels_1.number_format = '0%'
    
    
    for combination in combos_slide['Segment']:   
        
        if '+' in combination:
            conditional_colors.append(RGBColor(240, 192, 86))
        else:
            conditional_colors.append(RGBColor(0, 51, 104))

        
    colors = conditional_colors


    for index, series in enumerate(chart9_2.series):
        for point, color in zip(series.points, colors):
            point.format.fill.solid()
            point.format.fill.fore_color.rgb = color
    

    #Text placeholder
    
    pct_new_slide = float(nbr_trips_two_segments/nbr_total_trips)

    sld_9.placeholders[10].text = f'{"{:.0%}".format(pct_new_slide)} of reward baskets included 2 or more segments'

    sld_9.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")}' 

    
    phr = sld_9.placeholders[16]
    # Modify the font name and size
    font_name = 'Arial Black'
    font_size = Pt(10)
    # Access the text frame and text properties
    text_frame = phr.text_frame
    for paragraph in text_frame.paragraphs:
        paragraph.font.name = font_name
        paragraph.font.size = font_size
        for run in paragraph.runs:
            run.text = run.text.replace('•', '')


In [192]:

#Adding Slide 10: New Segment Purchase

if nbr_segments > 1:
    
    if len(all_ids_new_seg) != 0:

        sld_10 = prs.slides.add_slide(prs.slide_layouts[9])

        #Chart 10.1: NEW SEGMENT

        chart_data = ChartData()
        chart_data.categories = ['Did Not Buy a New Segment', 'Tried a New Segment']

        chart_data.add_series('', (1-df_plus_one.loc[1, 'Pct Of Shoppers'], df_plus_one.loc[1, 'Pct Of Shoppers']), '0%')


        chart10_1 = sld_10.placeholders[11].insert_chart(XL_CHART_TYPE.PIE, chart_data).chart
        chart10_1.font.size  = Pt(12)
        chart10_1.has_legend = True
        chart10_1.legend.position = XL_LEGEND_POSITION.BOTTOM
        chart10_1.legend.include_in_layout = False
        chart10_1.has_title = False

        leg = chart10_1.legend
        if leg:
            leg.font.bold = True


        chart10_1.plots[0].has_data_labels = True
        data_labels_1 = chart10_1.plots[0].data_labels
        data_labels_1.font.size = Pt(16)
        data_labels_1.font.bold = True
        data_labels_1.font.color.rgb = RGBColor(255, 255, 255)
        data_labels_1.number_format = '0%'
        data_labels_1.position = XL_LABEL_POSITION.CENTER


        #Chart 10.2: NEW SEGMENT 2

        chart_data = ChartData()
        chart_data.categories = segments_new_part

        chart_data.add_series('', pct_data_new_part/100, '0%')


        chart10_2 = sld_10.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
        chart10_2.font.size  = Pt(12)
        chart10_2.category_axis.tick_labels.font.bold = True
        chart10_2.value_axis.has_major_gridlines = False
        chart10_2.value_axis.visible = False


        chart10_2.plots[0].has_data_labels = True
        data_labels_1 = chart10_2.plots[0].data_labels
        data_labels_1.font.size = Pt(14)
        data_labels_1.font.bold = True
        data_labels_1.number_format = '0%'

        for series in chart10_2.series:
            for point in series.points:
                point.format.fill.solid()
                point.format.fill.fore_color.rgb = RGBColor(240, 192, 86)


        #Text placeholder

        pct_new_slide = df_plus_one.loc[1, 'Pct Of Shoppers']

        sld_10.placeholders[10].text = f'{"{:.0%}".format(pct_new_slide)} of participants who were previous buyers of campaign promoted products tried a NEW segment'
        sld_10.placeholders[16].text = f'Campaign Period: {str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")}' 

        phr = sld_10.placeholders[16]
        # Modify the font name and size
        font_name = 'Arial Black'
        font_size = Pt(10)
        # Access the text frame and text properties
        text_frame = phr.text_frame
        for paragraph in text_frame.paragraphs:
            paragraph.font.name = font_name
            paragraph.font.size = font_size
            for run in paragraph.runs:
                run.text = run.text.replace('•', '')


In [193]:

#Adding Slide 11: Basket Sizes

sld_11 = prs.slides.add_slide(prs.slide_layouts[5])

#Chart 11.1: Bar Chart

chart_data = ChartData()
chart_data.categories = ['All Other Baskets during VMR Reward Period', 'VMR Reward Baskets']

chart_data.add_series('', (round(df_vmr_ttls_basket.loc[1, 'Average Basket Size'],2), round(df_vmr_ttls_basket.loc[0, 'Average Basket Size'],2)), '$#,##0')


chart11_1 = sld_11.placeholders[11].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
chart11_1.font.size  = Pt(12)
chart11_1.category_axis.tick_labels.font.bold = True
chart11_1.value_axis.has_major_gridlines = False
#chart11_1.series[0].overlap = -30


chart11_1.plots[0].has_data_labels = True
data_labels_1 = chart11_1.plots[0].data_labels
data_labels_1.font.size = Pt(14)
data_labels_1.font.bold = True
data_labels_1.number_format = '$#,##0.00'


bar_colors = [RGBColor(127, 169, 184), RGBColor(0, 56, 104)]

# Iterate through the bars and set colors
for i, series in enumerate(chart11_1.series):
    for j, point in enumerate(series.points):
        point.format.fill.solid()
        point.format.fill.fore_color.rgb = bar_colors[j]


#Text placeholder

pct_1 = "${:,.2f}".format(nbr_reward_promoted_trips['avg_basket_size'].values[0])
sld_11.placeholders[13].text = f'{pct_1}'




#Chart 11.2: Bar Chart

if nbr_redemp_promoted['trips'].values[0] != 0:
    
    chart_data = ChartData()
    chart_data.categories = ['All Other Baskets during VMR Redemption Period', 'VMR Redemption Baskets']

    chart_data.add_series('', (round(df_vmr_ttls_basket.loc[4, 'Average Basket Size'],2), round(df_vmr_ttls_basket.loc[3, 'Average Basket Size'],2)), '$#,##0')


    chart11_2 = sld_11.placeholders[12].insert_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, chart_data).chart
    chart11_2.font.size  = Pt(12)
    chart11_2.category_axis.tick_labels.font.bold = True
    chart11_2.value_axis.has_major_gridlines = False


    chart11_2.plots[0].has_data_labels = True
    data_labels_1 = chart11_2.plots[0].data_labels
    data_labels_1.font.size = Pt(14)
    data_labels_1.font.bold = True
    data_labels_1.number_format = '$#,##0.00'
    
    
    bar_colors = [RGBColor(127, 169, 184), RGBColor(0, 56, 104)]

    # Iterate through the bars and set colors
    for i, series in enumerate(chart11_2.series):
        for j, point in enumerate(series.points):
            point.format.fill.solid()
            point.format.fill.fore_color.rgb = bar_colors[j]


    #Text placeholder

    pct_2 = round(nbr_redemp_promoted['trips'].values[0]/nbr_redemp_trips['trips'].values[0]*100)
    sld_11.placeholders[14].text = f'{pct_2}%'

    

pct_rew_bask = (round(df_vmr_ttls_basket.loc[0, 'Average Basket Size'],2) - round(df_vmr_ttls_basket.loc[1, 'Average Basket Size'],2))/ round(df_vmr_ttls_basket.loc[1, 'Average Basket Size'],2)

if nbr_redemp_promoted['trips'].values[0] != 0:
    pct_red_bask = (round(df_vmr_ttls_basket.loc[3, 'Average Basket Size'],2) - round(df_vmr_ttls_basket.loc[4, 'Average Basket Size'],2))/ round(df_vmr_ttls_basket.loc[4, 'Average Basket Size'],2)

    
#Text placeholder at bottom

if nbr_redemp_promoted['trips'].values[0] != 0:
    sld_11.placeholders[10].text = f'Reward Baskets were {"{:.0%}".format(pct_rew_bask)} larger than All Other baskets during same time period \nRedemption Baskets were {"{:.0%}".format(pct_red_bask)} larger than All Other baskets during same time period'
else:
    sld_11.placeholders[10].text = f'Reward Baskets were {"{:.0%}".format(pct_rew_bask)} larger than All Other baskets during same time period'
    
    
campaign_dts = f'''{str(reward_print_start).replace("-","/")} - {str(reward_print_end).replace("-","/")}'''

sld_11.placeholders[16].text = f'Campaign Period: {campaign_dts}' 

phr = sld_11.placeholders[16]
# Modify the font name and size
font_name = 'Arial Black'
font_size = Pt(10)
# Access the text frame and text properties
text_frame = phr.text_frame
for paragraph in text_frame.paragraphs:
    paragraph.font.name = font_name
    paragraph.font.size = font_size
    for run in paragraph.runs:
        run.text = run.text.replace('•', '')



In [194]:
sld_12 = prs.slides.add_slide(prs.slide_layouts[10])

In [195]:
sld_13 = prs.slides.add_slide(prs.slide_layouts[14])

In [196]:
prs.save(export_path + report_name_for_export + "_" + segment_type_def + "_" + "VMR" + "_" + str(last_data_dt) + ".pptx")

In [197]:
print('The report ran successfully!')

The report ran successfully!


# 